1. 共用參數設定


In [ ]:
'''
Measure the elapsed time of every code cell, including cells that raise errors.
記錄每個 Code Cell 的執行時間，即使發生錯誤也顯示已耗費的時間。
'''
from time import perf_counter as _cell_perf_counter
from IPython import get_ipython as _cell_get_ipython

_CELL_TIMER_SHELL = _cell_get_ipython()
if _CELL_TIMER_SHELL is None:
    raise RuntimeError('請在 Jupyter Notebook 中執行本教材。')
_CELL_TIMER_STATE = {'started': _cell_perf_counter()}

def _cell_timer_begin(_info):
    _CELL_TIMER_STATE['started'] = _cell_perf_counter()

def _cell_timer_end(result):
    started = _CELL_TIMER_STATE.pop('started', None)
    if started is None:
        return
    minutes, seconds = divmod(_cell_perf_counter() - started, 60)
    elapsed = f'{int(minutes)} 分 {seconds:05.2f} 秒' if minutes else f'{seconds:.2f} 秒'
    status = '執行失敗' if result.error_before_exec or result.error_in_exec else '執行完成'
    print(f'⏱ 本格{status}，耗時：{elapsed}')

'''
Replace existing timers when the first cell is executed again.
重新執行第一格時，先取消舊計時器，避免重複顯示耗時。
'''
for _event_name, _callback in globals().get('_CELL_TIMER_CALLBACKS', ()):
    try:
        _CELL_TIMER_SHELL.events.unregister(_event_name, _callback)
    except ValueError:
        pass
_CELL_TIMER_CALLBACKS = (
    ('pre_run_cell', _cell_timer_begin),
    ('post_run_cell', _cell_timer_end),
)
for _event_name, _callback in _CELL_TIMER_CALLBACKS:
    _CELL_TIMER_SHELL.events.register(_event_name, _callback)

'''
Import standard-library tools for editable paths and local service settings.
匯入設定路徑與本機服務參數所需的 Python 內建工具。
'''
import os
from pathlib import Path
ROOT = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.home() / 'ai_course'
ROOT = ROOT.expanduser().resolve()  # Ubuntu 可改成 Path('/home/your_user/ai_course')
SCRATCH_ROOT = Path('/tmp') if Path('/kaggle/working').is_dir() else ROOT / 'cache'
SCRATCH_ROOT = SCRATCH_ROOT.expanduser().resolve()  # 可改成其他儲存裝置或模型磁碟
ROOT.mkdir(parents=True, exist_ok=True)
SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)

COMFY = ROOT / 'ComfyUI-HyperFlowH3'

'''
Store large downloaded GGUF weights in scratch storage.
設定 GGUF 模型權重的暫存磁碟位置，減少工作空間占用。
'''
STORE = SCRATCH_ROOT / 'hyperflow_h3_models'
PORT = 8194  # 本機可用 Port
COMFY_REF = 'v0.45.0'

'''
Set the Hugging Face cache directory before any model downloads.
在下載模型之前指定 Hugging Face 快取目錄，避免大型檔案占滿工作空間。
'''
HF_CACHE_DIR = SCRATCH_ROOT / 'hyperflow_h3_hf'
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'

'''
Select the HyperFlow GGUF quantization. Q8 is memory-intensive on T4.
選擇 HyperFlow 的 GGUF 精度；Q8 對 T4 記憶體要求較高。
'''
QUANT = 'Q8_0'  # 'Q3_K_M', 'Q4_K_S', 'Q4_K_M', 'Q5_K_M', 'Q6_K', 'Q8_0'
DEFAULT_QUANT = QUANT

'''
Set GPU 0 memory budget in GiB; 14.5 is aggressive on a 15 GiB T4.
設定 GPU 0 的記憶體預算（GiB）；15 GiB T4 設為 14.5 剩餘緩衝很小，OOM 可調低。
'''
GPU_COUNT = 2  # Kaggle T4 ×2；其他 NVIDIA 主機可改成 1～4
MODEL_GPU_ID = 0  # 生成模型的主要運算 GPU（單張，並非分散式權重切割）
TEXT_GPU_ID = 1  # 文字編碼器 GPU，例如四張卡可指定為 3
GPU0_MODEL_GIB = 14.5  # 14.5、14.0、13.5
GPU1_MODEL_GIB = 14.5  # 14.5、14.0、13.5

'''
Select the text-encoder weight placement strategy.
設定文字編碼器權重管理：dynamic 分批調度較節省 GPU 記憶體；streaming 使用時整體載入；resident 盡量常駐 GPU 記憶體。
'''
TEXT_MODE = 'dynamic'  # 'dynamic', 'streaming', 'resident'
TEXT_GPU = f'cuda:{TEXT_GPU_ID}'  # 隨第一格 GPU ID 自動調整
NODE_CACHE_MODE = 'classic'  # 'classic', 'none'

'''
Choose the HyperFlow text encoder; select a format actually supported by the installed loader.
選擇 HyperFlow 文字編碼器；需搭配實際支援的權重與 Loader。
'''
ENCODER_VARIANT = '8B_FP8'  # 可選 4B_FP8、4B_BF16、8B_NVFP4、8B_W4A8、8B_INT8、8B_FP8、8B_BF16、32B_NVFP4、32B_INT8、32B_BF16
DEFAULT_ENCODER_VARIANT = ENCODER_VARIANT
WEIGHTS_IN = 'disk'  # 'disk', 'auto', 'ram'

'''
Select the local directory used by R2V image uploads.
指定 R2V 參考圖片的儲存目錄；圖片檔案由該 Task Cell 管理。
'''
R2V_REF_DIR = ROOT / 'hyperflow_h3_r2v_references'
'''
Keep generation and MP4 output at the H3-supported frame rate.
將影片生成與 MP4 輸出統一設定為 H3 支援的影格率。
'''
FPS = 24

REMOTE_PORT = 8765  # 例如 8765、8766，範圍 1024～65535
REMOTE_SERVICE_ID = '06_HyperFlow_H3_Flash_影片生成_Kaggle'

'''
Place optional API job data inside the configured workspace.
將遠端工作與暫存資料存放在第一格設定的工作空間底下。
'''
REMOTE_ROOT = ROOT / 'remote_api_jobs' / REMOTE_SERVICE_ID
REMOTE_QUEUE_CAPACITY = 32  # 建議 12～32，依課堂人數調整
REMOTE_UPLOAD_CONCURRENCY = 2  # 建議 1～2

'''
Choose output retention and storage safety limits.
設定遠端成果保留期限、空間限制及主機磁碟最低剩餘量。
'''
REMOTE_RESULT_TTL_HOURS = 24
REMOTE_RESULT_LIMIT_GIB = 8
REMOTE_MIN_FREE_GIB = 2
REMOTE_STALL_HOURS = float('inf')  # 不限制單次生成時長，也不將執行中的工作誤判為停滯
GGUF_REPO = 'VideoRebirthAI/hyperflow-h3-flash'

'''
Validate user-editable local ports and remote queue limits.
確認教師設定的本機 Port 與遠端佇列限制有效，避免服務啟動後才失敗。
'''
assert 1024 <= REMOTE_PORT <= 65535, 'REMOTE_PORT 必須介於 1024～65535。'
assert 1 <= REMOTE_QUEUE_CAPACITY <= 1024, 'FIFO 佇列容量設定無效。'
assert 1 <= REMOTE_UPLOAD_CONCURRENCY <= 8, '上傳並發限制設定無效。'
assert REMOTE_RESULT_TTL_HOURS > 0 and REMOTE_RESULT_LIMIT_GIB > 0 and REMOTE_MIN_FREE_GIB >= 0
assert REMOTE_STALL_HOURS > 0
assert 1024 <= PORT <= 65535 and PORT != REMOTE_PORT
print('✓ 第 1 節完成：共用參數已設定，請繼續執行環境安裝。')

assert 1 <= GPU_COUNT <= 4, 'GPU_COUNT 必須介於 1～4。'
assert all(0 <= gpu < GPU_COUNT for gpu in (MODEL_GPU_ID, TEXT_GPU_ID)), 'GPU ID 超出 GPU_COUNT 的範圍。'
assert GPU_COUNT != 1 or MODEL_GPU_ID == TEXT_GPU_ID, '單張 GPU 時，兩種元件應使用相同的 GPU ID。'


2. 安裝 HyperFlow H3 Flash 環境


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import os
import subprocess
import sys
from pathlib import Path
import torch
assert torch.cuda.is_available() and torch.cuda.device_count() >= GPU_COUNT, '可用 CUDA GPU 數量不足；請檢查 NVIDIA 驅動、PyTorch 與 GPU_COUNT。'
for i in sorted({MODEL_GPU_ID, TEXT_GPU_ID}):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} / {torch.cuda.get_device_properties(i).total_memory / 1024**3:.2f} GiB')

'''
Install the official HyperFlow loader and the 8B/4B projection runtime.
安裝官方 HyperFlow Loader 與 8B／4B 投影節點。
'''
repositories = {
    COMFY: 'https://github.com/Comfy-Org/ComfyUI.git',
    COMFY / 'custom_nodes/ComfyUI-GGUF': 'https://github.com/city96/ComfyUI-GGUF.git',
    COMFY / 'custom_nodes/ComfyUI-HyperFlow-Flash': 'https://github.com/Video-Rebirth/ComfyUI-HyperFlow-Flash.git',
    COMFY / 'custom_nodes/ComfyUI-ClipProj': 'https://github.com/nicolab28/ComfyUI-ClipProj.git',
}
H3_RESTART_REQUIRED = False
for directory, url in repositories.items():
    assert not directory.exists() or (directory / '.git').is_dir(), f'不是 Git 倉庫：{directory}'
    if (directory / '.git').is_dir():
        dirty = subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True).strip()
        if not dirty and directory != COMFY:
            before = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()
            subprocess.run(['git', '-C', str(directory), 'pull', '--ff-only', '-q'], check=True)
            H3_RESTART_REQUIRED |= before != subprocess.check_output(['git', '-C', str(directory), 'rev-parse', 'HEAD'], text=True).strip()

        elif dirty:
            '''
            Preserve local changes for manual inspection.
            保留本機修改供使用者自行檢查。
            '''
            print(f'本機修改未覆寫：{directory.name}')

    else:
        directory.parent.mkdir(parents=True, exist_ok=True)

        '''
        Pin only the core ComfyUI checkout to a release tag.
        只有 ComfyUI 核心固定於正式版標籤。
        '''
        options = ['--branch', COMFY_REF] if directory == COMFY else []
        subprocess.run(['git', 'clone', '--depth', '1', *options, url, str(directory)], check=True)
        H3_RESTART_REQUIRED = True

'''
Fetch the selected release on every run.
每次執行都確認指定的 ComfyUI 正式版。
'''
subprocess.run(['git', '-C', str(COMFY), 'fetch', '--depth', '1', 'origin', 'tag', COMFY_REF], check=True)
previous_core = subprocess.check_output(['git', '-C', str(COMFY), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run(['git', '-C', str(COMFY), 'checkout', '-q', '--detach', COMFY_REF], check=True)

'''
Restart if an existing process is using an older core checkout.
如果執行中的 ComfyUI 核心版本過舊，標記為需要重新啟動。
'''
H3_RESTART_REQUIRED |= previous_core != subprocess.check_output(['git', '-C', str(COMFY), 'rev-parse', 'HEAD'], text=True).strip()
constraint = ROOT / 'hyperflow_torch_constraints.txt'
constraint.write_text(f'torch=={torch.__version__}\n', encoding='utf-8')

'''
Install only the dependencies declared by ComfyUI and HyperFlow.
僅安裝 ComfyUI 與 HyperFlow 官方宣告的相依套件。
'''
requirements = [
    COMFY / 'requirements.txt',
    COMFY / 'custom_nodes/ComfyUI-GGUF/requirements.txt',
    COMFY / 'custom_nodes/ComfyUI-HyperFlow-Flash/requirements.txt',
]
for requirement in requirements:
    '''
    Skip an optional file that is not present in this release.
    如果某版沒有相依套件檔案，就安全略過。
    '''
    if requirement.is_file():
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint), '-r', str(requirement)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-c', str(constraint),
                'huggingface_hub', 'hf_xet', 'requests', 'websocket-client', 'tqdm', 'ipywidgets>=8,<9'], check=True)

'''
Display installed source revisions for reproducibility.
顯示各程式庫的實際版本，方便問題追蹤。
'''
for directory in repositories:
    revision = subprocess.check_output(['git', '-C', str(directory), 'rev-parse', '--short', 'HEAD'], text=True).strip()
    print(f'{directory.name}: {revision}')

print('✓ 第 2 節完成：HyperFlow H3 Flash / ComfyUI v0.45.0')


3. 下載 HyperFlow 模型與輔助權重


In [ ]:
'''
Check disk capacity before downloading gigabyte-scale files.
下載大型權重前先檢查可用磁碟空間。
'''
import shutil
os.environ['HF_HOME'] = str(HF_CACHE_DIR)
os.environ['HF_HUB_CACHE'] = str(HF_CACHE_DIR / 'hub')

'''
Disable the optional extra Xet chunk cache.
停用不必要的 Xet 分塊快取以節省空間。
'''
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'
from huggingface_hub import hf_hub_download

'''
Choose the generation task before model download; select the same task for generation.
在下載模型前先選擇任務；生成 Cell 會沿用此設定。 Options / 可選值: 't2v' 文字生成、'i2v' 起始圖片、'flf2v' 首尾圖片、'r2v' 多張參考圖片。
'''
TASK = globals().get('TASK', 't2v')  # 't2v', 'i2v', 'l2v', 'flf2v', 'r2v'
assert TASK in ('t2v', 'i2v', 'l2v', 'flf2v', 'r2v'), f'不支援的 TASK：{TASK}'

'''
Select the compressed FL2VA or the separate Ref2VA diffusion architecture.
一般影片使用 FL2VA，多圖參考 R2V 則使用獨立的 Ref2VA。
'''
MODEL_FAMILY = 'ref2va' if TASK == 'r2v' else 'fl2va'
task_models = globals().get('TASK_CONFIGS', {}).get(TASK, {})
ENCODER_SIZE = ENCODER_VARIANT.split('_', 1)[0]

'''
Map published GGUF sizes for the two independent diffusion architectures (decimal GB).
記錄 FL2VA 與 Ref2VA 兩套獨立模型的公開權重容量（十進位 GB）。
'''
MODEL_SIZES_GB = {
    'fl2va': {'Q3_K_M': 7.22, 'Q4_K_S': 8.01, 'Q4_K_M': 9.05,
              'Q5_K_M': 10.3, 'Q6_K': 11.6, 'Q8_0': 14.9},
    'ref2va': {'Q3_K_M': 7.4, 'Q4_K_S': 8.14, 'Q4_K_M': 9.22,
               'Q5_K_M': 10.45, 'Q6_K': 11.76, 'Q8_0': 15.16},
}
QUANT_SIZES_GB = MODEL_SIZES_GB[MODEL_FAMILY]
assert QUANT in QUANT_SIZES_GB, f'GPU 0 不支援 {QUANT}。'

'''
Obtain the specific compressed HyperFlow checkpoint, not the old 31B GGUF.
下載真正的 HyperFlow 壓縮模型，而不是先前 31B 的 GGUF。
'''
DIT_NAME = f'minimax_h3_{MODEL_FAMILY}_hyperflow_flash_14b-{QUANT}.gguf'
ENCODERS = {
    '32B_NVFP4': ('Comfy-Org/MiniMax-H3', 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', 15.7),
    '32B_INT8': ('Comfy-Org/MiniMax-H3', 'qwen3vl_32b_minimax_h3_int8_convrot.safetensors', 27.1),
    '32B_BF16': ('Comfy-Org/MiniMax-H3', 'qwen3vl_32b_minimax_h3_bf16.safetensors', 51.5),
    '8B_NVFP4': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_nvfp4.safetensors', 6.31),
    '8B_W4A8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_w4a8.safetensors', 6.31),
    '8B_INT8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_int8_convrot.safetensors', 9.35),
    '8B_FP8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_fp8_scaled.safetensors', 10.6),
    '8B_BF16': ('Comfy-Org/Qwen3-VL', 'qwen3vl_8b_bf16.safetensors', 17.5),
    '4B_FP8': ('Comfy-Org/Qwen3-VL', 'qwen3vl_4b_fp8_scaled.safetensors', 5.24),
    '4B_BF16': ('Comfy-Org/Qwen3-VL', 'qwen3vl_4b_bf16.safetensors', 8.88),
}

'''
Stop before downloading an unsupported GPU 1 model variant.
若 GPU 1 模型選項不存在，下載前就明確停止。
'''
assert ENCODER_VARIANT in ENCODERS, f'GPU 1 不支援 {ENCODER_VARIANT}；可選：{list(ENCODERS)}'
ENCODER_REPO, encoder_name, ENCODER_GB = ENCODERS[ENCODER_VARIANT]
ENCODER_FILE = f'text_encoders/{encoder_name}'

'''
Select the matching v3.1 projection only for smaller encoders.
只有小型 Encoder 需要對應的 v3.1 ClipProj 投影矩陣。
'''
PROJECTION_FILE = f'mmh3-{ENCODER_SIZE.lower()}-ClipProj-v3.1-mlp.safetensors' if ENCODER_SIZE != '32B' else None
TEXT_LOADER_NODE = 'ClipProjDeviceLoader' if ENCODER_SIZE == '32B' else 'ClipProjLoader'
print(f'GPU 0：HyperFlow {QUANT}（{QUANT_SIZES_GB[QUANT]} GB）；GPU 1：{ENCODER_VARIANT}（{ENCODER_GB} GB）；/tmp 剩餘 {shutil.disk_usage("/tmp").free / 1024**3:.1f} GiB')

'''
Download exactly one task-specific GGUF plus its encoder and two VAEs.
只下載所選任務的 FL2VA 或 Ref2VA GGUF，以及 Encoder 與兩個 VAE。
'''
models = [
    (GGUF_REPO, f'{MODEL_FAMILY}_gguf/{DIT_NAME}', 'diffusion_models'),
    (ENCODER_REPO, ENCODER_FILE, 'text_encoders'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_video_vae_fp16.safetensors', 'vae'),
    ('Comfy-Org/MiniMax-H3', 'vae/minimax_h3_audio_vae_fp32.safetensors', 'vae'),
]
if PROJECTION_FILE:
    models.append(('NicoLab28/ClipProj-MiniMax-H3', PROJECTION_FILE, 'clip_projections'))

STORE.mkdir(parents=True, exist_ok=True)

'''
Estimate the additional space needed for all selected model weights (decimal GB).
估算本次所選全部模型權重還需要多少暫存空間（十進位 GB）。
'''
model_sizes = [QUANT_SIZES_GB[QUANT], ENCODER_GB, 5.21, 0.605]
if PROJECTION_FILE:
    model_sizes.append(0.577 if ENCODER_SIZE == '8B' else 0.481)

missing_gb = sum(size for (_, filename, folder), size in zip(models, model_sizes)
                 if not (STORE / folder / filename).is_file())

'''
Read free space on the actual filesystem containing the model directory.
取得真正儲存模型權重的檔案系統剩餘空間。
'''
free_gb = shutil.disk_usage(STORE).free / 1e9
assert free_gb >= missing_gb * 1.05, (
    f'/tmp 空間不足：尚需約 {missing_gb:.1f} GB，剩餘 {free_gb:.1f} GB；'
    '請清除暫存模型，或改用較小的 GPU 0／GPU 1 權重。'
)
print(f'下載容量檢查：尚需約 {missing_gb:.1f} GB；剩餘 {free_gb:.1f} GB。')

'''
Track newly linked files so a running service can rescan dropdowns.
追蹤新增的模型連結，確保已啟動的服務更新選單。
'''
for repo, filename, folder in models:
    print('模型：', Path(filename).name, flush=True)
    source = Path(hf_hub_download(repo_id=repo, filename=filename, local_dir=STORE / folder))

    '''
    Make the model visible to ComfyUI without copying it.
    利用符號連結讓 ComfyUI 辨識模型，避免複製權重。
    '''
    target = COMFY / 'models' / folder / source.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_symlink() and (not target.exists() or not target.samefile(source)):
        target.unlink()
    if not target.exists():
        target.symlink_to(source.resolve())
        H3_RESTART_REQUIRED = True

    '''
    Reject unexpected same-name files to avoid silent model substitution.
    遇到同名但內容不同的權重檔案時，避免默默載入錯誤檔案。
    '''
    assert target.samefile(source), f'模型檔案衝突：{target}'

H3_MODEL_CHOICE = (MODEL_FAMILY, QUANT, ENCODER_VARIANT)
print(f'✓ 第 3 節完成：{TASK.upper()} / {MODEL_FAMILY} / GPU 0 {QUANT}；GPU 1 {ENCODER_VARIANT}（{TEXT_LOADER_NODE}）；Video/Audio VAE 已就緒。')


4. 生成與文字編碼 GPU／CPU Offload 設定

In [ ]:
'''
Require the teacher settings cell before applying ComfyUI GPU configuration.
套用 ComfyUI GPU 設定前，確認已執行教師共用設定 Cell。
'''
assert all(name in globals() for name in ('GPU0_MODEL_GIB', 'GPU1_MODEL_GIB', 'TEXT_MODE', 'TEXT_GPU', 'NODE_CACHE_MODE')), '請先執行「教師共用設定」Cell。'
gpu0_capacity = torch.cuda.get_device_properties(MODEL_GPU_ID).total_memory / 1024**3
gpu1_capacity = torch.cuda.get_device_properties(TEXT_GPU_ID).total_memory / 1024**3

'''
Reject physically impossible budgets and unsupported load modes.
避免設定不可能的 GPU 預算或不支援的載入模式。
'''
assert 0 < GPU0_MODEL_GIB < gpu0_capacity and 0 < GPU1_MODEL_GIB < gpu1_capacity
assert WEIGHTS_IN in ('auto', 'disk', 'ram')
assert TEXT_MODE in ('dynamic', 'streaming', 'resident')

'''
Reserve GPU 0 VRAM for activation tensors and decoder workloads.
保留 GPU 0 的部分 VRAM 給中間 Tensor 與解碼運算。
'''
GPU0_RESERVE_GIB = round(gpu0_capacity - GPU0_MODEL_GIB, 3)

assert NODE_CACHE_MODE in ('classic', 'none')
H3_DESIRED_CONFIG = (GPU0_RESERVE_GIB, WEIGHTS_IN, TEXT_GPU, TEXT_MODE, NODE_CACHE_MODE, MODEL_GPU_ID)
print(f'✓ 第 4 節完成：GPU 0 目標 {GPU0_MODEL_GIB:.1f} GiB、保留 {GPU0_RESERVE_GIB:.1f} GiB；GPU 1 使用 {TEXT_MODE}，參考值 {GPU1_MODEL_GIB:.1f} GiB（非硬上限）；offload={WEIGHTS_IN}')


5. 啟動 ComfyUI 與節點相容性檢查


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import requests
import time
LOG = ROOT / 'hyperflow_h3_comfyui.log'
BASE = f'http://127.0.0.1:{PORT}'

'''
Test the ComfyUI API without exposing any public tunnel.
不開放公開網址，直接測試本機 ComfyUI API。
'''
def ready():
    '''
    Handle normal startup connection errors without crashing.
    安全處理服務啟動期間尚未開放連線的情況。
    '''
    try:
        return requests.get(f'{BASE}/system_stats', timeout=3).ok

    except requests.RequestException:
        return False

'''
Read a node's enumerated dropdown options from the live API.
從服務目前註冊的節點規格讀取下拉式選單。
'''
def choices(node_name, input_name):
    '''
    Inspect the requested node's input declaration.
    讀取指定節點的輸入規格。
    '''
    groups = NODE_INFO[node_name].get('input', {})
    for kind in ('required', 'optional'):
        spec = groups.get(kind, {}).get(input_name)

        '''
        Return the dropdown list only when its schema is enumerated.
        只有明確的選單才回傳可選值。
        '''
        if isinstance(spec, (list, tuple)) and spec and isinstance(spec[0], list):
            return spec[0]
        if spec is not None:
            return None
    raise RuntimeError(f'{node_name}.{input_name} 欄位不存在。')

need_restart = globals().get('H3_RESTART_REQUIRED', False) or globals().get('H3_ACTIVE_CONFIG') != H3_DESIRED_CONFIG
if ready() and not need_restart:
    need_restart = globals().get('H3_ACTIVE_MODEL') != H3_MODEL_CHOICE

'''
Shut down only a child process owned by this notebook kernel.
只允許關閉目前 Notebook Kernel 自己管理的服務行程。
'''
if ready() and need_restart:
    '''
    Avoid killing a model service that belongs to a different session.
    避免終止其他 Notebook 或 Session 的模型服務。
    '''
    assert 'H3_PROC' in globals() and H3_PROC.poll() is None, f'埠 {PORT} 已被其他行程使用；請重新啟動 Kaggle Session。'
    H3_PROC.terminate()
    try:
        H3_PROC.wait(timeout=20)

    except subprocess.TimeoutExpired:
        H3_PROC.kill()

        '''
        Wait for the kernel to release process resources.
        等待系統完全回收該行程資源。
        '''
        H3_PROC.wait(timeout=10)

if not ready():
    command = [sys.executable, 'main.py', '--listen', '127.0.0.1', '--port', str(PORT),
        '--default-device', str(MODEL_GPU_ID),
               '--enable-dynamic-vram', '--fast-disk', '--reserve-vram', str(GPU0_RESERVE_GIB),
               '--cache-classic' if NODE_CACHE_MODE == 'classic' else '--cache-none',
               '--disable-pinned-memory']

    '''
    Store complete startup warnings in a diagnostic file.
    將完整的啟動警告保存於診斷檔案。
    '''
    with LOG.open('w', encoding='utf-8') as stream:
        H3_PROC = subprocess.Popen(command, cwd=COMFY, stdout=stream, stderr=subprocess.STDOUT)
    for attempt in range(180):
        if H3_PROC.poll() is not None:
            raise RuntimeError(LOG.read_text(errors='replace')[-6500:])
        if ready():
            break
        time.sleep(2)
    assert ready(), LOG.read_text(errors='replace')[-6500:]

H3_ACTIVE_CONFIG = H3_DESIRED_CONFIG

'''
Retrieve registered node types and parameter schemas.
取得已註冊的節點類型與參數規格。
'''
NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()
REQUIRED_NODES = (TEXT_LOADER_NODE, 'HyperFlowLoader', 'HyperFlowSigmas',
                  'MiniMaxH3ImageToVideo', 'MiniMaxH3ReferenceToVideo', 'VAELoader', 'BasicGuider', 'KSamplerSelect',
                  'RandomNoise', 'SamplerCustomAdvanced', 'VAEDecode', 'VAEDecodeAudio',
                  'CreateVideo', 'SaveVideo', 'LoadImage')

missing = [name for name in REQUIRED_NODES if name not in NODE_INFO]

'''
Show actionable errors for incompatible or incomplete plugin installs.
外掛版本不相容時顯示可追蹤的具體缺失。
'''
assert not missing, f'ComfyUI 缺少節點 {missing}。請檢查 {LOG}。'
assert DIT_NAME in (choices('HyperFlowLoader', 'unet_name') or []), f'HyperFlow Loader 未辨識模型：{DIT_NAME}'
assert Path(ENCODER_FILE).name in (choices(TEXT_LOADER_NODE, 'clip_name') or []), '找不到 Qwen3-VL Encoder。'

'''
Validate the projection only when selecting a smaller encoder.
只有選擇小型 Encoder 時才確認 ClipProj 投影矩陣。
'''
if PROJECTION_FILE:
    assert PROJECTION_FILE in (choices(TEXT_LOADER_NODE, 'projection') or []), '找不到 ClipProj v3.1 投影矩陣。'

assert TEXT_GPU in (choices(TEXT_LOADER_NODE, 'device') or []), f'{TEXT_LOADER_NODE} 不支援 {TEXT_GPU}。'
assert TEXT_MODE in (choices(TEXT_LOADER_NODE, 'mode') or []), 'Encoder Loader 不支援指定的 Offload 模式。'

'''
Require the native MiniMax text-encoder architecture when using 32B.
選擇原生 32B 時確認節點有 MiniMax 的文字編碼器型別。
'''
if ENCODER_SIZE == '32B':
    '''
    Avoid loading the 32B weights under an unrelated encoder type.
    避免將 32B 權重當成其他架構載入。
    '''
    assert 'minimax' in (choices(TEXT_LOADER_NODE, 'type') or []), '缺少 MiniMax 原生 Encoder 型別。'

for name in ('minimax_h3_video_vae_fp16.safetensors', 'minimax_h3_audio_vae_fp32.safetensors'):
    assert name in (choices('VAELoader', 'vae_name') or []), f'找不到 VAE：{name}'

assert 'steps' in NODE_INFO['HyperFlowSigmas']['input'].get('optional', {}), 'HyperFlow Sigmas 缺少 steps 欄位。'

'''
Verify the canonical Euler sampler remains available.
確認官方 HyperFlow 所需的 Euler Sampler 已存在。
'''
assert 'euler' in (choices('KSamplerSelect', 'sampler_name') or []), 'Sampler 不支援 euler。'
H3_ACTIVE_MODEL = H3_MODEL_CHOICE
H3_RESTART_REQUIRED = False

'''
Confirm the server is ready for generation.
確認 HyperFlow 服務已可接受生成工作。
'''
print(f'✓ 第 5 節完成：GPU 0 {QUANT} / GPU 1 {ENCODER_VARIANT} / Disk Offload。')


6. T2V：Prompt 與生成參數


In [ ]:
'''
Describe the text-to-video scene and synchronized sound.
設定文生影片的場景與同步音效描述。
'''
T2V_PROMPT = '''
A cinematic close-up of a red panda in a misty forest. The red panda slowly turns its head.
Natural fur details, gentle camera movement, soft morning light. Sound: birds chirping and light wind.
'''.strip()
T2V_WIDTH, T2V_HEIGHT = 384, 224
T2V_SECONDS = 10

'''
Select the official HyperFlow 4-, 6-, or 8-step schedule.
選擇官方 HyperFlow 的 4、6 或 8 steps 排程。
'''
T2V_STEPS = 6
T2V_SEED = 42
T2V_OUTPUT = 'hyperflow_h3_flash_t2v.mp4'

'''
Preserve configurations for every task independently.
分開保存各項任務的設定，避免不同 Prompt 互相覆寫。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['t2v'] = dict(prompt=T2V_PROMPT, width=T2V_WIDTH, height=T2V_HEIGHT,
                           fps=FPS, seconds=T2V_SECONDS, steps=T2V_STEPS, seed=T2V_SEED,
                           encoder_variant=ENCODER_VARIANT, quant=QUANT,
                           output_name=T2V_OUTPUT, first_image='', last_image='')

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 't2v'

print(f'✓ 第 6 節完成：T2V | {T2V_WIDTH}×{T2V_HEIGHT} | {T2V_SECONDS} 秒 | {T2V_STEPS} steps | {QUANT} | GPU 1 {ENCODER_VARIANT} | {T2V_OUTPUT}')


7. I2V：Prompt、圖片與生成參數


In [ ]:
'''
Describe how the uploaded first frame should animate.
描述上傳的第一張圖片要如何轉化為動態影片。
'''
I2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()
I2V_FIRST_IMAGE = str(ROOT / 'first_frame.png')
I2V_WIDTH, I2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定圖生影片的目標秒數。
'''
I2V_SECONDS = 10
I2V_STEPS = 6
I2V_SEED = 42

'''
Set an independent image-to-video output filename.
指定圖生影片自己的輸出檔名。
'''
I2V_OUTPUT = 'hyperflow_h3_flash_i2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['i2v'] = dict(prompt=I2V_PROMPT, width=I2V_WIDTH, height=I2V_HEIGHT,
                           fps=FPS, seconds=I2V_SECONDS, steps=I2V_STEPS, seed=I2V_SEED,
                           encoder_variant=ENCODER_VARIANT, quant=QUANT,
                           output_name=I2V_OUTPUT, first_image=I2V_FIRST_IMAGE, last_image='')

'''
Confirm the configuration cell has run without attempting to open the image yet.
顯示設定完成；實際圖片檔案會在生成時再驗證。
'''
TASK = 'i2v'

print(f'✓ 第 7 節完成：I2V | {I2V_WIDTH}×{I2V_HEIGHT} | {I2V_SECONDS} 秒 | {QUANT} | GPU 1 {ENCODER_VARIANT} | 圖片：{I2V_FIRST_IMAGE}')


8. L2VA：最後影格生成影片


In [ ]:
'''
Describe how the uploaded last frame should animate.
描述上傳的最後一張圖片要如何轉化為動態影片。
'''
L2V_PROMPT = '''
Animate the subject in the reference image with subtle natural motion and smooth cinematic camera movement.
Keep the person's or object's identity, clothing and environment consistent. Include realistic ambient sound.
'''.strip()
L2V_LAST_IMAGE = str(ROOT / 'last_frame.png')
L2V_WIDTH, L2V_HEIGHT = 384, 224

'''
Configure output duration for image-to-video.
設定最後影格生成影片的目標秒數。
'''
L2V_SECONDS = 10
L2V_STEPS = 6
L2V_SEED = 42

'''
Set an independent image-to-video output filename.
指定圖生影片自己的輸出檔名。
'''
L2V_OUTPUT = 'hyperflow_h3_flash_l2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['l2v'] = dict(prompt=L2V_PROMPT, width=L2V_WIDTH, height=L2V_HEIGHT,
                           fps=FPS, seconds=L2V_SECONDS, steps=L2V_STEPS, seed=L2V_SEED,
                           encoder_variant=ENCODER_VARIANT, quant=QUANT,
                           output_name=L2V_OUTPUT, first_image='', last_image=L2V_LAST_IMAGE)

'''
Confirm the configuration cell has run without attempting to open the image yet.
顯示設定完成；實際圖片檔案會在生成時再驗證。
'''
TASK = 'l2v'

print(f'✓ 第 8 節完成：L2V | {L2V_WIDTH}×{L2V_HEIGHT} | {L2V_SECONDS} 秒 | {QUANT} | GPU 1 {ENCODER_VARIANT} | 圖片：{L2V_LAST_IMAGE}')


9. FLF2V：Prompt、首尾圖片與生成參數


In [ ]:
'''
Describe a video that connects its first and last reference frames.
描述從第一張圖片過渡到最後一張圖片的影片內容。
'''
FLF2V_PROMPT = '''
Create a smooth cinematic transition from the first reference frame to the last reference frame.
Maintain consistent subject details and lighting, with natural intermediate motion and synchronized ambient sound.
'''.strip()
FLF2V_FIRST_IMAGE = str(ROOT / 'first_frame.png')
FLF2V_LAST_IMAGE = str(ROOT / 'last_frame.png')

'''
Set output dimensions for first-and-last-frame generation.
設定首尾影格模式的輸出解析度。
'''
FLF2V_WIDTH, FLF2V_HEIGHT = 384, 224
FLF2V_SECONDS = 10
FLF2V_STEPS = 6

'''
Set this task’s seed; use -1 to randomize each generation.
設定此任務的種子；-1 代表每次生成都重新抽取。
'''
FLF2V_SEED = 42
FLF2V_OUTPUT = 'hyperflow_h3_flash_flf2v.mp4'
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})

'''
Register both conditioning images and all parameters.
登記兩張參考圖片與全部生成參數。
'''
TASK_CONFIGS['flf2v'] = dict(prompt=FLF2V_PROMPT, width=FLF2V_WIDTH, height=FLF2V_HEIGHT,
                             fps=FPS, seconds=FLF2V_SECONDS, steps=FLF2V_STEPS, seed=FLF2V_SEED,
                             encoder_variant=ENCODER_VARIANT, quant=QUANT,
                             output_name=FLF2V_OUTPUT, first_image=FLF2V_FIRST_IMAGE,
                             last_image=FLF2V_LAST_IMAGE)

'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'flf2v'

print(f'✓ 第 9 節完成：FLF2V | {FLF2V_WIDTH}×{FLF2V_HEIGHT} | {FLF2V_SECONDS} 秒 | {QUANT} | GPU 1 {ENCODER_VARIANT} | 參考圖 2 張')


10. R2V：Prompt、參考圖片與生成參數


In [ ]:
'''
Describe the primary picture; the prompt stays valid when selecting only one image.
描述第一張參考圖；即使只選一張圖片，Prompt 仍能正常使用。
'''
R2V_PROMPT = '''
<Picture 1> defines the main subject's appearance.
Create a cinematic scene with natural motion and ambient sound.
Keep the main subject consistent, and use additional pictures as visual references when supplied.
'''.strip()
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import io
import tempfile
import shutil
from html import escape
R2V_FORMATS = {'.png': 'PNG', '.jpg': 'JPEG', '.jpeg': 'JPEG', '.webp': 'WEBP'}

'''
Preserve the most recently confirmed batch across cell reruns in this Kaggle session.
在同一個 Kaggle Session 中重跑 Cell 時，沿用最後一次成功選取的圖片組合。
'''
R2V_REF_DIR.mkdir(parents=True, exist_ok=True)
R2V_ACTIVE_FILES = sorted(p for p in R2V_REF_DIR.iterdir()
                          if p.is_file() and p.suffix.lower() in R2V_FORMATS
                          and len(p.stem) == 3 and p.stem.isdigit())

assert len(R2V_ACTIVE_FILES) <= 9 and all(
    path.stem == f'{index:03d}' for index, path in enumerate(R2V_ACTIVE_FILES, 1)
), 'R2V 圖片資料夾編號不連續；請重新選取整組圖片。'

'''
Keep the public reference list synchronized with the files actually on disk.
讓公開的參考圖片清單與磁碟上的實際檔案保持一致。
'''
R2V_REF_IMAGES = [str(path) for path in R2V_ACTIVE_FILES]
R2V_REF_VIDEOS = []  # Example: [str(ROOT / 'clip.mp4')]
R2V_REF_AUDIOS = []  # Example: [str(ROOT / 'voice.wav')]

'''
Use soundtrack of a reference video if the uploaded video has audio.
若參考影片含音軌，選擇是否同時使用該音軌作為聲音條件。
'''
R2V_VIDEO_AUDIO = False  # True, False
R2V_REF_IMAGE_SIZE = 'match'
R2V_WIDTH, R2V_HEIGHT = 384, 224

'''
Set the approximate video duration in seconds.
設定目標影片秒數，實際影格數會依 H3 規則對齊。
'''
R2V_SECONDS = 10
R2V_STEPS = 6

'''
Encoder sizes and precisions; B means one billion parameters. Prefer native 32B for picture references; it needs GPU 1 offload on T4.
Encoder 大小與精度；B 代表十億個參數，32B=320 億、8B=80 億、4B=40 億。 Setting / 設定值  | Model / 模型                 | Precision / 權重精度 32B_NVFP4       | 原生 Qwen3-VL-32B            | NVFP4（4-bit 浮點） 32B_INT8        | 原生 Qwen3-VL-32B            | INT8 ConvRot（8-bit 整數） 32B_BF16        | 原生 Qwen3-VL-32B            | BF16（16-bit 浮點） 8B_NVFP4        | Qwen3-VL-8B + ClipProj      | NVFP4（4-bit 浮點） 8B_W4A8         | Qwen3-VL-8B + ClipProj      | W4A8（4-bit 權重／8-bit activation） 8B_INT8         | Qwen3-VL-8B + ClipProj      | INT8 ConvRot（8-bit 整數） 8B_FP8          | Qwen3-VL-8B + ClipProj      | FP8（8-bit 浮點） 8B_BF16         | Qwen3-VL-8B + ClipProj      | BF16（16-bit 浮點） 4B_FP8          | Qwen3-VL-4B + ClipProj      | FP8（8-bit 浮點） 4B_BF16         | Qwen3-VL-4B + ClipProj      | BF16（16-bit 浮點） 多圖參考預設使用原生 32B；在 T4 上須透過 GPU 1 卸載管理。 8B／4B + ClipProj 可節省資源，但多圖參考的影像條件品質尚需實測。
'''
R2V_SEED = 42
R2V_OUTPUT = 'hyperflow_h3_flash_r2v.mp4'

'''
Retain the other three task configurations in the same notebook session.
保留同一個 Notebook Session 內其他三種生成模式的參數。
'''
TASK_CONFIGS = globals().get('TASK_CONFIGS', {})
TASK_CONFIGS['r2v'] = dict(prompt=R2V_PROMPT, width=R2V_WIDTH, height=R2V_HEIGHT,
                           fps=FPS, seconds=R2V_SECONDS, steps=R2V_STEPS, seed=R2V_SEED,
                           encoder_variant=ENCODER_VARIANT, quant=QUANT,
                           output_name=R2V_OUTPUT, first_image='', last_image='',
                           ref_images=R2V_REF_IMAGES, ref_image_size=R2V_REF_IMAGE_SIZE,
                           ref_videos=R2V_REF_VIDEOS, ref_audios=R2V_REF_AUDIOS,
                           video_audio=R2V_VIDEO_AUDIO, quant=QUANT,
                           encoder_variant=ENCODER_VARIANT)

R2V_UPLOAD_STATUS = widgets.HTML()

'''
Keep the active file picker replaceable so the next selection cannot append old uploads.
讓選檔元件可重新建立，確保下次多選不會累積上一批檔案。
'''
R2V_UPLOAD_PANEL = widgets.VBox()
R2V_UPLOAD_HELP = widgets.HTML(value=(
    '<b>R2V 參考圖片（1～9 張）</b>：一次選取所有圖片；每次重新選取會完整取代上一批，'
    '從 001 重新編號，並刪除不再使用的舊圖片。'
))

'''
Apply one complete browser selection as a new image set, never merging it with an earlier batch.
將瀏覽器本次選取視為完整的新圖片組合，絕不與上一批合併。
'''
def on_r2v_upload(change):
    '''
    Update the global image list when a valid new batch is committed.
    新圖片批次確認有效後，更新全域參考圖片清單。
    '''
    global R2V_REF_IMAGES
    uploaded = change['new']
    if not uploaded:
        return
    staging = None
    message = ''

    '''
    Try to stage and atomically replace the entire selected batch.
    嘗試建立暫存圖片組合，再以整批替換方式更新正式資料夾。
    '''
    try:
        assert 1 <= len(uploaded) <= 9, '請一次選取 1～9 張圖片。'
        staging = Path(tempfile.mkdtemp(prefix='.hyperflow_r2v_new_', dir=ROOT))
        mapping = []

        '''
        Keep the browser's selected file order for <Picture 1> onward.
        按照瀏覽器回傳的選取順序，對應 <Picture 1> 等圖片標籤。
        '''
        for index, item in enumerate(uploaded, 1):
            original = item['name'].replace('\\', '/').split('/')[-1]
            extension = Path(original).suffix.lower()
            assert extension in R2V_FORMATS, f'不支援的圖片格式：{original}'
            content = bytes(item['content'])

            '''
            Reject an empty upload before opening the image decoder.
            在開啟圖片解碼器前先拒絕空白檔案。
            '''
            assert content, f'圖片內容為空：{original}'
            with Image.open(io.BytesIO(content)) as picture:
                assert picture.format == R2V_FORMATS[extension], f'圖片內容格式不符：{original}'
                picture.verify()
            filename = f'{index:03d}{extension}'
            (staging / filename).write_bytes(content)
            mapping.append(f'&lt;Picture {index}&gt; = {escape(filename)} ← {escape(original)}')
        backup = ROOT / f'.hyperflow_r2v_old_{__import__("uuid").uuid4().hex}'
        R2V_REF_DIR.rename(backup)
        try:
            '''
            Publish the verified batch as the sole active image directory.
            將已驗證的新批次整批設為唯一有效的圖片目錄。
            '''
            staging.rename(R2V_REF_DIR)

        except Exception:
            backup.rename(R2V_REF_DIR)
            raise
        shutil.rmtree(backup, ignore_errors=True)

        '''
        Read only the currently numbered files in exact Picture order.
        只讀取現在有效的圖片，並依 Picture 順序排列。
        '''
        R2V_REF_IMAGES = [str(path) for path in sorted(R2V_REF_DIR.iterdir()) if path.is_file()]
        TASK_CONFIGS['r2v']['ref_images'] = list(R2V_REF_IMAGES)
        message = '<b>已套用最新 ' + str(len(uploaded)) + ' 張圖片：</b><br>' + '<br>'.join(mapping)

    except Exception as error:
        message = '<b>未更新參考圖片：</b>' + escape(str(error))

    finally:
        '''
        Always clean up any unused staging directory and reset the browser picker.
        無論成功或失敗，均清除暫存目錄並重新建立瀏覽器選檔元件。
        '''
        if staging is not None and staging.exists():
            shutil.rmtree(staging, ignore_errors=True)
        R2V_UPLOAD_STATUS.value = message
        R2V_UPLOAD_PANEL.children = (new_r2v_uploader(), R2V_UPLOAD_STATUS)

'''
Create a new browser file picker and subscribe to its batch-upload event.
建立新的瀏覽器選檔元件，並監聽每批圖片上傳完成事件。
'''
def new_r2v_uploader():
    '''
    Offer one browser picker for multiple image files.
    提供可一次選取多張圖片的瀏覽器元件。
    '''
    uploader = widgets.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=True,
                                   description='多選參考圖片')
    uploader.observe(on_r2v_upload, names='value')
    return uploader

R2V_UPLOAD_STATUS.value = ('目前圖片：' + (', '.join(Path(path).name for path in R2V_REF_IMAGES)
                                          if R2V_REF_IMAGES else '尚未選擇'))

R2V_UPLOAD_PANEL.children = (new_r2v_uploader(), R2V_UPLOAD_STATUS)

'''
Render the upload widget directly in this R2V settings cell.
直接在 R2V 參數 Cell 顯示圖片上傳元件。
'''
display(widgets.VBox([R2V_UPLOAD_HELP, R2V_UPLOAD_PANEL]))
'''
Select the current generation task and its model settings.
選擇目前生成模式及此模式的模型設定；尚未執行生成。
'''
TASK = 'r2v'

print(f'✓ 第 10 節完成：R2V | {R2V_WIDTH}×{R2V_HEIGHT} | {len(R2V_REF_IMAGES)} 張參考圖 | {QUANT} | GPU 1 {ENCODER_VARIANT}。素材設定完成後請執行第 11 節 Generate。')


11. 生成影片（T2V／I2V／L2VA／FLF2V／R2V）


In [ ]:
'''
Serialize notebook generation with optional remote FIFO GPU jobs.
讓 Notebook 手動生成與選用的遠端 FIFO 共用 GPU 鎖，避免同時推論。
不必先啟動 ngrok 或任何 HTTP API。
'''
import threading
GENERATION_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
with GENERATION_LOCK:
    import json
    import shutil
    import uuid
    import math
    import websocket
    from tqdm.auto import tqdm
    TASK = globals().get('TASK', 't2v')
    assert TASK in globals().get('TASK_CONFIGS', {}), f'請先執行 {TASK.upper()} 的參數設定 Cell。'
    config = TASK_CONFIGS[TASK]

    '''
    Share sampling, decoding and video output logic across four task modes.
    四種生成模式共用採樣、解碼及影片輸出流程。
    '''
    MODE = TASK
    PROMPT = config['prompt']
    WIDTH, HEIGHT = config['width'], config['height']
    SECONDS, STEPS = config['seconds'], config['steps']
    SEED, OUTPUT_NAME = config['seed'], config['output_name']

    '''
    Resolve task-specific reference images without mixing modes.
    依據所選任務使用對應參考圖片，避免不同模式混用。
    '''
    FIRST_IMAGE, LAST_IMAGE = config['first_image'], config['last_image']
    REF_IMAGES = list(config.get('ref_images', []))
    REF_VIDEOS = list(config.get('ref_videos', []))
    REF_AUDIOS = list(config.get('ref_audios', []))
    VIDEO_AUDIO = bool(config.get('video_audio', False))
    REF_IMAGE_SIZE = config.get('ref_image_size', 'match')
    '''
    Keep frame calculation and video encoding synchronized at 24 FPS.
    讓影格數計算與影片編碼使用相同的 24 FPS。
    '''
    FPS = int(config.get('fps', 24))
    assert FPS == 24, 'MiniMax H3／HyperFlow H3 僅支援 24 FPS。'
    
    ENCODER_VARIANT = config['encoder_variant']

    '''
    Derive the encoder architecture needed by the selected loader.
    取得 Loader 所需的文字編碼器架構大小。
    '''
    ENCODER_SIZE = ENCODER_VARIANT.split('_', 1)[0]
    QUANT = config['quant']
    assert QUANT in QUANT_SIZES_GB and ENCODER_VARIANT in ENCODERS, f'不支援的 GPU 模型組合：{QUANT}/{ENCODER_VARIANT}'
    assert STEPS in (4, 6, 8), 'HyperFlow STEPS 僅支援 4、6、8。'

    '''
    Ensure edited GPU 0 memory budgets have been applied to the running service.
    確認調整過的 GPU 0 記憶體預算已套用到目前服務。
    '''
    assert globals().get('H3_ACTIVE_CONFIG') == H3_DESIRED_CONFIG, 'GPU 0 預算已變更，請重新執行第 4 節啟動 ComfyUI。'
    assert globals().get('H3_MODEL_CHOICE') == (MODEL_FAMILY, QUANT, ENCODER_VARIANT) and globals().get('H3_ACTIVE_MODEL') == (MODEL_FAMILY, QUANT, ENCODER_VARIANT), (
        f'目前選擇 GPU 0 {QUANT}／GPU 1 {ENCODER_VARIANT}，但尚未完成下載或服務預檢。請重跑第 2 節與第 4 節。'
    )
    assert MODEL_FAMILY == ('ref2va' if TASK == 'r2v' else 'fl2va'), '目前 GPU 0 模型架構與任務不一致，請重跑第 2、4 節。'
    assert DIT_NAME == f'minimax_h3_{MODEL_FAMILY}_hyperflow_flash_14b-{QUANT}.gguf', 'GPU 0 模型檔名不一致，請重跑第 2 節。'

    '''
    Reject a stale GPU 1 encoder that belongs to an earlier selection.
    避免切換 GPU 1 模型後仍使用前一次載入的 Encoder。
    '''
    assert Path(ENCODER_FILE).name == ENCODERS[ENCODER_VARIANT][1], 'GPU 1 模型檔名不一致，請重跑第 2 節。'
    LAST_OUTPUT = None
    assert ready(), 'ComfyUI 已中斷，請先重新執行第 3 節。'
    assert MODE in ('t2v', 'i2v', 'l2v', 'flf2v', 'r2v') and PROMPT.strip() and STEPS > 0 and math.isfinite(SECONDS) and SECONDS > 0, (
        '請確認生成模式、Prompt、STEPS 與 SECONDS 設定正確；SECONDS 必須為正數。'
    )

    '''
    Require positive dimensions aligned to 32, without an artificial resolution limit.
    寬高需為正數且符合 32 的倍數，不另外限制解析度或長寬比。
    '''
    assert WIDTH > 0 and HEIGHT > 0 and WIDTH % 32 == 0 and HEIGHT % 32 == 0, (
        'WIDTH 與 HEIGHT 必須為正數且是 32 的倍數。'
    )
    FRAMES = max(22, 17 * math.ceil((SECONDS * FPS - 5) / 17) + 5)
    assert FRAMES <= 3600, f'影格數 {FRAMES} 超出官方 H3 節點上限 3600，請縮短影片秒數。'
    import secrets
    RUN_SEED = secrets.randbelow(2**32) if SEED == -1 else int(SEED)

    '''
    Require a supported seed before building the inference workflow.
    在建立推論工作流程前，先檢查種子範圍。
    '''
    assert 0 <= RUN_SEED < 2**32, 'SEED 必須介於 0 至 4294967295，或設為 -1。'
    print(f'本次生成 seed：{RUN_SEED}')
    output_stem = Path(OUTPUT_NAME).stem
    run_name = f'{output_stem}_{QUANT}_{uuid.uuid4().hex[:8]}'
    text_inputs = {'clip_name': Path(ENCODER_FILE).name,
                   'type': 'minimax' if ENCODER_SIZE == '32B' else 'auto',
                   'device': TEXT_GPU, 'mode': TEXT_MODE}

    '''
    Add the learned projection only when a 4B/8B encoder is selected.
    只有使用 4B／8B Encoder 時才接入學習式投影矩陣。
    '''
    if PROJECTION_FILE:
        text_inputs['projection'] = PROJECTION_FILE
    graph = {
        '1': {'class_type': TEXT_LOADER_NODE, 'inputs': text_inputs},
        '2': {'class_type': 'HyperFlowLoader', 'inputs': {'unet_name': DIT_NAME, 'weights_in': WEIGHTS_IN}},
        '4': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_video_vae_fp16.safetensors'}},
        '5': {'class_type': 'VAELoader', 'inputs': {'vae_name': 'minimax_h3_audio_vae_fp32.safetensors'}},
        '6': {'class_type': ('MiniMaxH3ReferenceToVideo' if MODE == 'r2v' else 'MiniMaxH3ImageToVideo'), 'inputs': {
            'clip': ['1', 0], 'vae': ['4', 0], 'prompt': PROMPT,
            'width': WIDTH, 'height': HEIGHT, 'length': FRAMES}},
        '7': {'class_type': 'BasicGuider', 'inputs': {
            'model': ['2', 0], 'conditioning': ['6', 0]}},
        '8': {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'euler'}},
        '9': {'class_type': 'HyperFlowSigmas', 'inputs': {'model': ['2', 0], 'steps': STEPS}},
        '10': {'class_type': 'RandomNoise', 'inputs': {'noise_seed': RUN_SEED}},
        '11': {'class_type': 'SamplerCustomAdvanced', 'inputs': {
            'noise': ['10', 0], 'guider': ['7', 0],
            'sampler': ['8', 0], 'sigmas': ['9', 0], 'latent_image': ['6', 1]}},
        '12': {'class_type': 'VAEDecode', 'inputs': {'samples': ['11', 0], 'vae': ['4', 0]}},
        '13': {'class_type': 'VAEDecodeAudio', 'inputs': {'samples': ['11', 0], 'vae': ['5', 0]}},
        '14': {'class_type': 'CreateVideo', 'inputs': {
            'images': ['12', 0], 'audio': ['13', 0], 'fps': float(FPS)}},
        '15': {'class_type': 'SaveVideo', 'inputs': {
            'video': ['14', 0], 'filename_prefix': run_name, 'format': 'mp4'}},
    }
    if MODE == 'r2v':
        assert isinstance(REF_IMAGES, (list, tuple)) and 0 <= len(REF_IMAGES) <= 9, 'R2V_REF_IMAGES 最多 9 張；也可以改用影片或音訊。'
        assert all(Path(path).is_file() for path in REF_IMAGES), 'R2V 有不存在的圖片；請重新選擇。'

        '''
        Restrict the reference resize mode to the two official choices.
        參考圖尺寸選項只能填寫官方支援的 match 或 max。
        '''
        assert REF_IMAGE_SIZE in ('match', 'max'), 'R2V_REF_IMAGE_SIZE 只能是 match 或 max。'
        import re
        mentioned = {int(number) for number in re.findall(r'<Picture\s+(\d+)>', PROMPT, re.IGNORECASE)}
        assert mentioned <= set(range(1, len(REF_IMAGES) + 1)), 'Prompt 的 <Picture N> 必須對應已提供圖片。'
        graph['6']['inputs']['audio_vae'] = ['5', 0]

        '''
        Apply the documented image-sizing method for every reference.
        對所有參考圖套用官方定義的縮放模式。
        '''
        graph['6']['inputs']['ref_image_size'] = REF_IMAGE_SIZE
        for index, image_path in enumerate(REF_IMAGES):
            source = Path(image_path)
            assert source.is_file() and source.stat().st_size > 0 and source.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp'), f'無效的 R2V 參考圖片：{source}'
            node_id = str(20 + index)
            destination = COMFY / 'input' / f'r2v_{index + 1}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'

            '''
            Create the input directory if it does not exist.
            視需要建立 ComfyUI input 資料夾。
            '''
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}
            graph['6']['inputs'][f'ref_images.ref_image_{index}'] = [node_id, 0]
    if MODE == 'r2v':
        assert len(REF_IMAGES) <= 9 and len(REF_VIDEOS) <= 3 and len(REF_AUDIOS) <= 3, 'R2V 最多 9 張圖片、3 段影片、3 段獨立音訊。'
        assert REF_IMAGES or REF_VIDEOS, 'R2V 至少需要一張參考圖片或一段參考影片；音訊可作為附加條件。'
        assert len(REF_IMAGES) + len(REF_VIDEOS) + len(REF_AUDIOS) <= 12, 'R2V 參考素材合計最多 12 個檔案。'
        tags = {'Picture': len(REF_IMAGES), 'Video': len(REF_VIDEOS),
                'Audio': len(REF_AUDIOS) + (len(REF_VIDEOS) if VIDEO_AUDIO else 0)}
        for label, count in tags.items():
            numbers = [int(n) for n in re.findall(r'<%s\s+(\d+)>' % label, PROMPT, re.I)]
            assert all(1 <= n <= count for n in numbers), f'<{label} N> 編號超出實際上傳數量：{numbers}。'
        for index, filename in enumerate(REF_VIDEOS):
            original = Path(filename)
            assert original.is_file() and original.suffix.lower() in ('.mp4', '.mov', '.webm'), f'參考影片不存在或格式不正確：{original}'
            destination = COMFY / 'input' / f'h3_ref_video_{uuid.uuid4().hex[:12]}{original.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, destination)
            node_id = str(120 + index * 2)
            parts_id = str(121 + index * 2)
            graph[node_id] = {'class_type': 'LoadVideo', 'inputs': {'file': destination.name}}
            graph[parts_id] = {'class_type': 'GetVideoComponents', 'inputs': {'video': [node_id, 0]}}
            graph['6']['inputs'][f'ref_videos.ref_video_{index}'] = [parts_id, 0]
            if VIDEO_AUDIO:
                graph['6']['inputs'][f'ref_video_audios.ref_video_audio_{index}'] = [parts_id, 1]
        for index, filename in enumerate(REF_AUDIOS):
            original = Path(filename)
            assert original.is_file() and original.suffix.lower() in ('.wav', '.mp3', '.flac', '.ogg', '.m4a'), f'參考音訊不存在或格式不正確：{original}'
            destination = COMFY / 'input' / f'h3_ref_audio_{uuid.uuid4().hex[:12]}{original.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(original, destination)
            node_id = str(140 + index)
            graph[node_id] = {'class_type': 'LoadAudio', 'inputs': {'audio': destination.name}}
            graph['6']['inputs'][f'ref_audios.ref_audio_{index}'] = [node_id, 0]

    '''
    Attach a verified local image to an H3 image-conditioning socket.
    將確認存在的本機圖片接到 H3 圖片條件輸入。
    '''
    for mode, image_path, node_id, input_name in [
        ('i2v', FIRST_IMAGE, '16', 'first_frame'),
        ('flf2v', FIRST_IMAGE, '16', 'first_frame'),
        ('flf2v', LAST_IMAGE, '17', 'last_frame'),
        ('l2v', LAST_IMAGE, '18', 'last_frame'),
    ]:

        '''
        Only attach images required by the active mode.
        只接入目前模式需要的圖片。
        '''
        if MODE == mode:
            source = Path(image_path)

            '''
            Stop before generation if a reference file is missing.
            若圖片不存在，在生成前立即停止。
            '''
            assert source.is_file(), f'找不到參考圖片：{source}'
            destination = COMFY / 'input' / f'{node_id}_{uuid.uuid4().hex[:8]}{source.suffix.lower()}'
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
            graph[node_id] = {'class_type': 'LoadImage', 'inputs': {'image': destination.name}}

            '''
            Connect the loaded image to H3 conditioning.
            將已載入圖片連到 H3 條件輸入。
            '''
            graph['6']['inputs'][input_name] = [node_id, 0]
    NODE_INFO = requests.get(f'{BASE}/object_info', timeout=90).json()
    assert TEXT_GPU in (choices(TEXT_LOADER_NODE, 'device') or []), 'Encoder Loader 不支援 GPU 1。'
    assert TEXT_MODE in (choices(TEXT_LOADER_NODE, 'mode') or []), 'Encoder Loader 不支援指定 Offload 模式。'

    '''
    Verify that the official HyperFlow Euler sampler is available.
    確認官方 HyperFlow Euler Sampler 可用。
    '''
    assert 'euler' in (choices('KSamplerSelect', 'sampler_name') or []), 'HyperFlow 必須使用 Euler Sampler。'
    assert 'HyperFlowSigmas' in NODE_INFO, '缺少 HyperFlow Sigma 節點。'
    if MODE in ('i2v', 'l2v', 'flf2v'):
        frame_inputs = NODE_INFO['MiniMaxH3ImageToVideo'].get('input', {}).get('optional', {})
        expected = ('first_frame', 'last_frame') if MODE == 'flf2v' else (('last_frame',) if MODE == 'l2v' else ('first_frame',))
        assert all(field in frame_inputs for field in expected), f'MiniMaxH3ImageToVideo 缺少 {expected} 輸入。'
    if MODE == 'r2v':
        required_nodes = {'LoadVideo', 'GetVideoComponents'} if REF_VIDEOS else set()
        if REF_AUDIOS:
            required_nodes.add('LoadAudio')
        missing_nodes = required_nodes - set(NODE_INFO)
        assert not missing_nodes, f'ComfyUI 缺少參考媒體節點：{sorted(missing_nodes)}，請更新 ComfyUI。'

    '''
    Validate that all graph dependency links target existing nodes.
    驗證所有 Workflow 連線的來源節點確實存在。
    '''
    for node_id, node in graph.items():
        '''
        Examine each declared input for possible node-reference links.
        檢查節點輸入中是否包含連到其它節點的參照。
        '''
        for value in node['inputs'].values():
            if isinstance(value, list) and len(value) == 2:
                assert str(value[0]) in graph, f'節點 {node_id} 連結到不存在的節點 {value[0]}。'
    for node_id, node in graph.items():
        schema = NODE_INFO[node['class_type']].get('input', {})
        accepted = set().union(*(set(schema.get(group, {})) for group in ('required', 'optional', 'hidden')))
        if node['class_type'] == 'MiniMaxH3ReferenceToVideo':
            assert 'ref_images' in accepted, 'MiniMaxH3ReferenceToVideo 不支援 ref_images；請更新 ComfyUI。'
            for family, paths in (('ref_videos', REF_VIDEOS), ('ref_audios', REF_AUDIOS), ('ref_video_audios', REF_VIDEOS if VIDEO_AUDIO else [])):
                assert not paths or family in accepted, f'目前 ComfyUI Ref2VA 不支援 {family}。'
            accepted.update(f'ref_images.ref_image_{index}' for index in range(len(REF_IMAGES)))
            accepted.update(f'ref_videos.ref_video_{index}' for index in range(len(REF_VIDEOS)))
            accepted.update(f'ref_video_audios.ref_video_audio_{index}' for index in range(len(REF_VIDEOS)) if VIDEO_AUDIO)
            accepted.update(f'ref_audios.ref_audio_{index}' for index in range(len(REF_AUDIOS)))
        unknown = set(node['inputs']) - accepted

        '''
        Fail early if a workflow input is no longer supported.
        當 Workflow 參數不再被節點支援時，立即停止並指出名稱。
        '''
        assert not unknown, f"節點 {node_id} ({node['class_type']}) 不支援的輸入：{sorted(unknown)}"
    print(f'{MODE.upper()} | {WIDTH}×{HEIGHT} | {FRAMES} frames | {STEPS} steps | GPU 0 {QUANT} | GPU 1 {ENCODER_VARIANT} | HyperFlow', flush=True)
    client_id = uuid.uuid4().hex
    ws = None

    '''
    Connect before queueing to avoid missing early progress messages.
    在送出任務前先連線，避免漏掉初期進度事件。
    '''
    try:
        ws = websocket.create_connection(f'ws://127.0.0.1:{PORT}/ws?clientId={client_id}', timeout=5)
        ws.settimeout(2)

    except (OSError, websocket.WebSocketException):
        ws = None

        '''
        Explain why no step percentage will be shown.
        說明為什麼目前不會出現步數百分比。
        '''
        print('即時步數事件不可用；將顯示階段與經過時間。', flush=True)
    response = requests.post(f'{BASE}/prompt', json={'prompt': graph, 'client_id': client_id}, timeout=120)
    if not response.ok:
        if ws is not None:
            ws.close()
        raise RuntimeError(response.text[:5000])
    prompt_id = response.json()['prompt_id']
    start = time.monotonic()

    '''
    Create a single notebook progress bar with real step events only.
    建立單一 Notebook 進度條，僅使用真實採樣步數更新。
    '''
    pbar = tqdm(total=STEPS, desc='排隊／載入模型', unit='step', leave=True)
    item = None
    last_history_check = 0.0
    phases = {'1': '文字編碼器', '2': '載入 HyperFlow GGUF', '6': '建立影片條件',
              '11': '採樣中', '12': '影片解碼', '13': '音訊解碼',
              '14': '合成影片', '15': '儲存影片'}

    '''
    Wait until ComfyUI reports a final history record for this task.
    持續等待 ComfyUI 回傳此任務的最終執行紀錄。
    '''
    try:
        while True:
            '''
            Measure real wall-clock time instead of estimating model progress.
            計算真實經過時間，不將時間誤當模型百分比。
            '''
            elapsed = time.monotonic() - start
            message = None
            if ws is not None:
                try:
                    message = ws.recv()

                except websocket.WebSocketTimeoutException:
                    pass

                except (OSError, websocket.WebSocketException):
                    '''
                    Release the unusable client connection.
                    關閉失效的 WebSocket 連線。
                    '''
                    ws.close()
                    ws = None
            if isinstance(message, str):
                event = json.loads(message)
                data = event.get('data') or {}

                '''
                Reject progress from another client or generation.
                忽略非本次任務的進度，避免步數混亂。
                '''
                if data.get('prompt_id') == prompt_id:
                    if event.get('type') == 'executing' and data.get('node'):
                        pbar.set_description(phases.get(str(data['node']), '執行中'), refresh=False)
                    if event.get('type') in ('progress', 'progress_state'):
                        progress = data
                        if event.get('type') == 'progress_state':
                            progress = data.get('nodes', {}).get('11', {})

                        '''
                        Ignore progress emitted by loading or decoding nodes.
                        忽略非採樣節點的進度，以免誤顯示百分比。
                        '''
                        if str(progress.get('node', progress.get('node_id'))) == '11':
                            value, maximum = progress.get('value'), progress.get('max')
                            if isinstance(value, (int, float)) and isinstance(maximum, (int, float)) and maximum > 0:
                                pbar.total = int(maximum)
                                pbar.n = max(0, min(int(value), pbar.total))
                                pbar.set_description('採樣中', refresh=False)

            '''
            Check the persistent history regardless of WebSocket availability.
            不論 WebSocket 是否正常，都定期檢查 ComfyUI 任務紀錄。
            '''
            if elapsed - last_history_check >= 4:
                last_history_check = elapsed
                if not ready():
                    '''
                    Preserve recent server logs for actionable diagnostics.
                    保留最近服務紀錄供後續除錯。
                    '''
                    raise RuntimeError('ComfyUI 服務中斷：\n' + LOG.read_text(errors='replace')[-4500:])
                item = requests.get(f'{BASE}/history/{prompt_id}', timeout=60).json().get(prompt_id)

                '''
                Stop polling once the job has a final record.
                任務產生最終紀錄後停止輪詢。
                '''
                if item:
                    errors = [m for m in item.get('status', {}).get('messages', []) if m[0] == 'execution_error']
                    if errors:
                        raise RuntimeError(str(errors[-1][1])[:5000])
                    break
            pbar.set_postfix_str(f'{elapsed / 60:.1f} 分鐘', refresh=False)
            pbar.refresh()
            if ws is None:
                time.sleep(1)
        pbar.n = pbar.total

        '''
        Label the status as completed.
        將顯示狀態設為完成。
        '''
        pbar.set_description('完成', refresh=False)
        pbar.set_postfix_str(f'{(time.monotonic() - start) / 60:.1f} 分鐘', refresh=False)
        pbar.refresh()

    finally:
        pbar.close()

        '''
        Close the WebSocket when it exists.
        若 WebSocket 仍存在，關閉它以釋放資源。
        '''
        if ws is not None:
            ws.close()
    files = [p for p in (COMFY / 'output').rglob('*') if p.is_file() and p.name.startswith(run_name)]
    files = [p for p in files if p.suffix.lower() in ('.mp4', '.webm', '.mov')]
    assert files, f'工作流程完成但未找到影片：{item.get("outputs", {})}'
    source_video = max(files, key=lambda path: path.stat().st_mtime)

    '''
    Save the user-visible output in Kaggle working storage.
    將成果儲存至 Kaggle working 目錄。
    '''
    OUTPUT_DIR = ROOT / 'hyperflow_h3_outputs'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_base = OUTPUT_DIR / f'{Path(OUTPUT_NAME).stem}_{QUANT}{source_video.suffix}'
    output = output_base

    '''
    Number duplicate filenames deterministically.
    依序為重複檔名編號。
    '''
    for index in range(2, 1000):
        if not output.exists():
            '''
            Preserve the current unique output path.
            保留目前可用的輸出檔名。
            '''
            break
        output = output_base.with_name(f'{output_base.stem}_{index:03d}{output_base.suffix}')
    shutil.copy2(source_video, output)
    print(f'完成：{output}｜耗時 {(time.monotonic() - start) / 60:.1f} 分鐘')
    LAST_OUTPUT = output
    print('✓ 第 11 節完成：影片已生成；請執行第 12 節預覽與下載。')


12. 影片預覽、下載與 GPU 使用量


In [ ]:
'''
Display the most recently generated video and its download link.
顯示最近一次成功生成影片的預覽與下載連結。
'''
from IPython.display import Video, FileLink, display
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片。'
display(Video(str(LAST_OUTPUT), embed=True, width=480))

'''
Provide a clickable Kaggle file download link.
顯示 Kaggle 影片檔案下載連結。
'''
display(FileLink(str(LAST_OUTPUT)))
# 顯示指定兩種模型元件的 GPU 記憶體用量，不依賴 nvidia-smi。
import torch
for gpu_id in sorted({MODEL_GPU_ID, TEXT_GPU_ID}):
    free_bytes, total_bytes = torch.cuda.mem_get_info(gpu_id)
    used_gib = (total_bytes - free_bytes) / 1024**3
    budget = GPU0_MODEL_GIB if gpu_id == MODEL_GPU_ID else GPU1_MODEL_GIB
    print(f'GPU {gpu_id}：使用 {used_gib:.2f} / {total_bytes / 1024**3:.2f} GiB（參考預算 {budget:.2f} GiB）')

print(f'✓ 第 12 節完成：影片預覽與下載連結已顯示；{LAST_OUTPUT.name}')


13. 擷取最後影格（選用）


In [ ]:
'''
Import the libraries needed for this section.
匯入本段程式所需的函式庫。
'''
import json
from IPython.display import FileLink, display
from PIL import Image
import shutil
assert shutil.which('ffmpeg') and shutil.which('ffprobe'), '找不到 ffmpeg 或 ffprobe；請先安裝 FFmpeg。'
assert globals().get('LAST_OUTPUT') is not None and LAST_OUTPUT.is_file(), '請先執行第 9 節生成影片，並確認第 10 節可正常預覽。'

'''
Build a PNG path next to the generated video.
在生成影片旁建立最後影格 PNG 檔案路徑。
'''
LAST_FRAME_PATH = LAST_OUTPUT.with_name(LAST_OUTPUT.stem + '_last_frame.png')
probe = subprocess.run(
    ['ffprobe', '-v', 'error', '-count_frames', '-select_streams', 'v:0',
     '-show_entries', 'stream=nb_read_frames', '-of', 'json', str(LAST_OUTPUT)],
    capture_output=True, text=True, check=True,
)
last_index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1

'''
Save the exact last frame as a PNG for the next I2V clip.
將最後一個影格儲存成 PNG，供下一段 I2V 使用。
'''
subprocess.run(
    ['ffmpeg', '-y', '-loglevel', 'error', '-i', str(LAST_OUTPUT),
     '-vf', rf'select=eq(n\,{last_index})', '-frames:v', '1', str(LAST_FRAME_PATH)],
    check=True,
)
display(Image.open(LAST_FRAME_PATH))
display(FileLink(str(LAST_FRAME_PATH)))
print(f"下一段：請在第 7 節 I2V 設定 I2V_FIRST_IMAGE = '{LAST_FRAME_PATH}'")

'''
Confirm completion of the extracted-frame export.
確認最後影格已成功輸出。
'''
print(f'✓ 第 13 節完成：已擷取最後影格；{LAST_FRAME_PATH.name}')


14. （選用）啟動 ngrok 與 FIFO 遠端 API


In [ ]:
'''
Require the first configuration cell before enabling the optional remote API.
選用遠端 API 啟動前，必須先執行第一格共用參數設定。
'''
assert all(name in globals() for name in ('REMOTE_PORT', 'REMOTE_SERVICE_ID', 'REMOTE_ROOT')), '請先執行第 1 節共用參數設定。'
import zlib
import math
import re

import queue
import copy
import secrets
import io
import json
import os
import shutil
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.request
import uuid
import zipfile
from email import policy
from email.parser import BytesParser
from getpass import getpass
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from urllib.parse import urlsplit, parse_qs

REMOTE_KIND = 'video'
REMOTE_FIELDS = ['task', 'prompt', 'seed', 'width', 'height', 'seconds', 'fps', 'steps', 'ref_image_size', 'quant', 'video_audio']
REMOTE_IS_LTX = False
REMOTE_SCRIPT = zlib.decompress(b'x\xda\xd5}ks\x14W\x96\xe0w~EFGLTU[\xc8H\xe0\x17\xd3\xf4\x06\x06ak\xda\x06\x16\x81{g\x19B\x91\xaaJI\xd9TUVWV!\x18\xad"$@H<\x84d\x9b\x87\x01\xd9\x80yX\xcdC\x80m\x8c\x10 >\xcc?\xd9Q\xd6\xe3\x93\xff\xc2\x9e\xd7\xbdy3+K\x88v\xcfn\xecl\xac\x1bUe\xe5=\xf7\xdc\xf3~\xddT*\xb5\xa1\xb7P\xf2\xca\x15\xab2\xecXyw\xa0l\x97]\xc7\xb7\x8a\x8e\x93sr\xd6\xa0W\x86/\\\xdf\xf2\x9dl\xc5\xf5\x8a\x9d\x1b\x82s\x8f\x82\xc9;\xb5\xf9\x07\xb5\xc5\xa7\xf5\x85\xb3\xc1\xcb\xd9\xda\xe9\xf1\xe6\xfcx\xfd\xea\xc9`\xea\x15\xfc\x19,\xdf\xff\xcf\xf1\xe3\x1bR\xf0f\x97\xdf\xfc\x17\xdf+\xaa\x7f\xfb\xc3\xd5\x8a\x9bW\x7fU\xabnN\xfd\xbb`W\x86\xd5\xbfG\x9c\x01\xdf\xcb\x1ev*\x1b\x06\xcb^\xc1\xaa\xfc5W\xe8\xb4\xab\x15\xcf\x92\xef\xf1\x83\r\xb4\xc4\x01\xdf!\xc8+\xb6\x7f\xd8r\x8e\x96\xf2n\xd6\xad\xe4\x8f\x01\xbcy\x80\x18v\xe0\x16\xe9\xfb\x82\x97s\xf2\x1bs\xdeH1\xef\xd9\xb9p;\xb5\x1f_\xd7/,\xd4\x16n\x06\xdf\x9d]]:\xdbxy\xba~\xffF\xfd\xd1x\xed\xdcT\xb0x\x156e\xed\xdf\xde\xf7\xa7__\x9e\xab\x7f\xbf\xbc\xfa\xfa[\xeb\x93\xbd\x07\xacM\x16?Y{x\x0b>\xad}\xf7K\xed\x87\xb3\xb0ekO\t\xdf\xe9[\xefZ\xc1\xec\xa3\xe6\xc4R0\xfer\xab\x95\xaat\x1fIuX)\x97\xffg0?\xc8\xff(\xc3\xff(<\xe1\x12\xd66k(\xef\r\xd8y?\x9d\xe9\x1cr*\xe9\x14~\x8aO\xe2\x0b2\xbc\xdb}\x0e\xc0\x8e\xdb\xd1\xdb\xc3}\xa7|\xabTv\x8e\xb8^\xd5\x87\x9d;G\x9dl\x15\xbf\xcaz\xc5Aw\xa8Z\xb6\x11*+\xeb\xe4\xf3\x9d\x1b\x1a\x8b\xe3\xc1\xec%<\xb1\x89\xa5\xd5\x17/\x82\xb3W\x82g?\x06\xd7\x9f5n\x9ekN\x9c\x87\xed6\x16\x1e\xc2\xbe\xad\x1d\xf0\xb4\x82\xce\xf6}\x07pN@\x026\x13\xa0\xec\xdf\xb1g\xf7\xae\xdeO\xfa\x00\xda\xd1\xb1L\x875\x98j\xdc?\x1bLN\xf3\x8b\xadQ|\xa6\xb3Z*9\xe5tf\xccBJ\x99=Q\xbb\xb8\x14]+\xa5\xb6\xe8{\xf9#N\xe2.\xddb\xce)9\xf0\x9f"\x10\x92S\xa9\xb8\xc5!\x1f(r\xf6R\xb0r\xd9\xdc\x13\xacP\x9fY\xa8\x03\x0c\xb4\x8e\xda\x08#\x04\x10m\xc2|\x10\xff8\xc4k\xf7\r\xdbeX\xd5.\x00\x19\x15\x87:\xac\x9c\x93\xf5r\xf0/\xcb.\xe6\xac#n\xce\xf1,\xafZ)U+V\xde\x1br\xb3\x96\x9d-{\xbe\x0fLR-3\x01"\x95!D\xd7\xae\xd5\x17\x16\xeb\x17\xae\xd7\xa6\xe7\x90\xb4\x80)&\x9f \x99\x9d\xbfY[\xb8\xf5\x9f\xe3\x13\x8d\x1f\x80r^\x06\xb3g\x82WO\xea\xa7\xa7\x1a/\x97\x82\xa9\xe5\xda\xd3\t`(\x05\xec\xe7{v\xf6\x08\xa8QJ/\x01G\x94`\xff\x15\xaf\x1c\x92w\x04Sr\xd6\xab\xaf\x90\xb4\xeb\xd7\x16\x83\xd33\x8c\x17\x03\xe1V\xe3\xd6M\xa4\xee\xbd\xf46\xb5\xe8\xde}{>\xdf\xbb\x1f\x96eT\x1dL\xf1b\xa9CQ\x10\xf4jY\xbbx\xc4\xf6\xcd\xb3\xb8s\x19\xf7\x19=\x0b\xde^\xf0x9x\xa4\xcf\xe2\xcf\xbd;\xf7\x7f\xdaa}\xda\xd3\xfb\xc9\xa7\xe6\x82#n\xae2\x9c:\xd4\xa1?\x18v\xdc\xa1\xe1\x16\x08p\x9b\x1b\xfd\x92\x93u\x07\xe1\x1cr\x8a\xc8\xf1\x9c\xd4\xf1\xc1\x1b\xaa\xc5J\x1b\x90\xea?|\x85\x048=\xc5GR{x\xc7\xa0\x93\xbe\x1e\xa0\x8d\x9d}\x1dV\xdf\xfe\x9e\xbd}\x06p 5\xbcb\xce7\xc1\xf3+N\xc9W\xd0m/\x95\x80\x01[\xe1\xf3A\x98\x12lB=\x83n\xde)\xda\x05\xa7\x1dp\x0b\x8b\xc1\xc39\x00N\xe8\xe2\xde\x85`n&\x04\xaegg\x87\xb5\xe7\xc0\xfe\xbd\x07\xf6\xf7\xef\xde\xfeyO\x04>\'g\x02\xc7\xcb\xf5\xe3R\nD\xcd^\x11\x00\xcb\xce\xa0Sv\x8aY\x07\x04\xac=\x04\xe2\x7f\xc4\xad\x0c\xc3\xaf\xad\x82{\x14Q)t\xbd\xba\xf2m\xed\xeb\xab&\xb4Lc\xc1\xe3\x99\xda\xd4i`\xb6\xc6\xf8\x89`\xfe\x12P4\x08\xcc\xe6\xc4\xeb`rfui&\x98;\xc7LP{\xf6\x0c\x1eV\x1b\xd9\xd5\xbb\xafo\x7f\x7f\xef\xe7\xdb?\xe9\xe9\xb0>\xdb\xae\xfemlg\xd0-\xfb\x95~\x82\xc8\xdcU\xde\x0e?5\xc4\xa2W\x14\xdc\x17<\xbf\x02;\xca\x82\x94\x80O\xe8W\xe5\x02\x1c@\xc9\xcdV\xaaeE\xbdH.\xa8\xdd\xf6u\x7f\xd1\xb9\x01\xfe\x032\xfb\x9eH\xc7\xf9\xf1`\xe5\xdc\xea\xd2x\xed\xc1M`\xe0\xe0\xcc\xf5\xe0\xf6U\x14\xe7+\xe7Pt\xd1\xfe\xeaOO\x06s\xd3j+\xfbzv1\xf0H,y\xd7\xaf\xa4\x19V\x96\x90\x80\\\x06\xd7\x07\xf9x\xf0PFdy\x1f\x81ay\xa44\xec\xbcq\x06$i|&\xe6\n\xfc\xd7\xce{E\xc7\xb2\xab9\xd7S\xd2\x0e\xc0A\xde&I\x8aHg12=\xc5b\xafy\xfd\xa7\xc6\xc2\x19\x13\xba/zw\xf6\xeci\x0b\x1d/\xa8\xa0\xc3\xe7\xb7\x1f\xd8\xd9\xdb\xfey\x02E?O\xef\xe6_\xc0\x0f\x06</\x1f\xf9\x01\xbd\x9c\x7f\x02\xbf\xd8\x05\xaa\xc3Q(\xd81\xecy\xc0\xd0\xe1\xce\xcb\x8e\xef\xfe\xbb"8+\r&Av\xd8\x82C*\xd8G3\xf1\xd3\n\x89\x8d\x89\xd0\xe2\xa7k\xd3\x97\xf0q\x8b\xc5M\xe3\xca\xc5\xfa\xdc)&\xbf\x96\xd3\xea\xef\xeb\xfd\x9f!\xbd\xc5\x0e\xab\x1f qP\xfd\xd2k#\n\x18m\x80.PG\xceQ\x1b\x0e\xb0\xe2\x1c\xadX\x00=@\\\xb6\x8e\x80\xf5d\x83vb\xb3\x05\r\'Q]\x11e\x0c\xa7\xb8\xf2\x1d\x9aP\x0fn\x9a\x82\x99\xc9\x8f_\x8fZ\xb2vi*xx\xb9\xfel\x01u\xc5\x95\x85\xfa\xe9i\xf8\x8d\xda\x05\xfe\xff?9N\t\x96\x02\xf6\x06I\x9c\xcfV\xf3\xa1\x14dmE`!>\xfdc\xc5\xecp\xd9+\xc2\x9e@\x12U\xac\xee-\xd6\xae\xbd}h\x0f|\r\xb4S\xbb\xf1\x92\xd4\xf1t}\xf12\x10\x11S\x13\xaf\xab\x94\xc8\x12p1\xea\x0b\xfe\xa1\xe6a\x12\x8fn1J"\x83%\xa4\x8d\xee-p\xccb;\xd0s\xdb\xe0#\xc0\xe7\xe7n\xd1\xfd\x1c\x0e\xe8\xd3\xcd\xbf\xbe\x9c\xfd\xf4\x18\x18\x05\xbb\xf2\xde\x08\xfci\x05\'&k\x17\x1e\xd5f\x7f\x0eW\x01\xa3\xa0g\xf7\x0eP\x85\xfb\xfa\xbf\xd8\xbe\xafw\xfbnSW\x08\xd2\xfb\x05\xe9J\x1a\xect\xca\xae\x18\x10\xeaX\xecrv\xd8\xad8\xcc\xffb\xda\x0e\x1c\x8b\xea2\xb4\x0b\x9d\xb2b0\xeb3\xfa\xd3\xd2\xc6m\xec0\xd8\xea\x0bn\xff\x10<\xd6\x84\xa5 \x15\xb2\x8a\x01\xde\xe9\x83Z\x02\xec\xf4\x03n\xba2\x077\xb5\xe8\x0c\rI\x88\x94O>9\xb0\xcb\xfak\x156\xe7\xfe\xbb\xadE\x16\xd9\xb2\x88\x05\x1f\x98\x13\xb6\x08Bn\xd8\xc9\x1e\xf6\x95\xca\xaf=\xbc\xcdd\x15\x1a\xb1\xb1W6\xa7f\x83s\x97\xe0\xd8\x81-@T\xaf\xbe\xb8\xc3\xe6-\xdb\xc1\xb5{\xdf\xd7\xae\xdf\x01\x01\xd8\x98\xfaY\xed\xed\xbf\x1f\x88\xe2\x9e\x80\n\xe5\xef_P\x94U\x8b>\x18y`\x9f\xc3\x1e"@\xa3\x02\xf2\xad\x01\x07\x80\x079V\xa98`S U\x0e9EGq\x04[K`\xaa\x80\xddX\xbbp\xa16}\x0fT\x07\x93\x03l\xc0\x04\x18\x94a}\xe1I\xcc:e\xf8\xc0\x1a\xa2\x7f\xd0\x11\xf4\xf5\x7f\xf21\xf1B\x9c\x82\xe0)\xf9\xa8\x0fmUs\x1db>\xc6\x02\x8b\xf8__^\x1d\xa5W\x8e\xbd;\x1a{\xcfX*j\x93x\x83\xa0J]\x90\xe4!\xb2}w\xa8`[>\x1cO\xae\x9a\x07\x14\xa0\x8e\x02\n;1)zs\xf1\x9b\xda\xa5\xe7\xc6\xf3\x08\xc1\x16\xb0\x0f\xdf\x87\xff\xff\xa1\x05V\x89\xd5Go\xa8\x9d\xff\xca0\re\xcbl\x9d\xc0f\xd2\xc0V\xefwX\x1f\x82\xe5\x9d\n\xdf\xc5_\x87<\xa5\xdf\x1b\xda\xda=p`p"N\xce\xc5#c\xd7\xa6\xe0\x14\xbc\xf21k\xa0\x9a\x03^\xf6\xada\x1bXi\xc0q@\xaa\x00\xa1\xbahaz\xb4\xdbr\xb5X$\xc9\xe2\x94\x8f\xb8Y0e\x80\x82\x1a\xf7f\x1a\xf7^\xd7.\xfe\xcc\xbe\x84\xbc\xb1\xb1\xf0M\xed\xd4/\xcd\xfb\x17\x9a7\xe6@\xc0\xa0\xc3AVO0\xfd\x98\x8d\xd3\xda\xfc\x0c\x10klw17\xe3\xd3\xcd\xfd\xdbw\xec\xef\xfd\xa2G\xec\xf6T\x06\x05\n|\xba\xb3\xa7\xafw_\xcfN\xf9\x180\xc0\x8b\xea\xb5\x1a\x8bgj\xd7~\x06"\x07\x97\xa495S\xbb\xf4\x98\xbd\x92\xfa\x83\x07\xd6\x16\x0b<\xbd\xe0\xe2\xf5\xe0\xecEk\x87W\x18<v\xa07D\xce^p\xa9\xd0\xd1\x08\x89\x14\x08\x1c|W\xda}v\x18\x14W1J\xe5#`\t+\x07S\xec\xbd\xb23\x84,\n\x86:\x9c:\x18\x9b\xb3\xd7"\x84\xbcr\x0e\xe0\n^O6oL\xa3\xb7\xb4x\x0e8\x80]K\x90\xc2\n"!\xc7\xc6\xc2\x8b\xe0\xd4\x997"\t\xdd\x86\xcf\xfaw|\xba\xa7wG\x0f\xe3(\xcd\x1f\xed\xda\xfey\xefg\xff\xda\xc1\xfc\xd1\x11\xe7\x89\x0c\x81\xdb\x16\xe7\xf4\x8a\xb7y]\x87\x95\xde`\xc1\xff\r\xa6\xf8\x88\xc1^\xa9}=%\x04!\x0c\x05\xe2\x9fU]\x0bc\xa1Hzu*x|\xb56\x7f\xcf\xc4\n\xe8v\xa6\x158]\x10R\x80\x0c>\xd4\xc6\xb3/\xf18\xbb\xf18\xd1\x08RGK\x87\x99\x89\xd0:\x1e\xde\x88W><\x88,\x82v8m\\\xe9\te\x98\xfbbLD\xc4\xf2\x08y"\xbe\xa2t\xeb\xcf\xea5\x9a\xd4aG]\x16\x9b\xeb\xcd\x13\xaf\xd8*6\xf7\x8fj\xe4\xc1\x19\x008v\x8c&J\t\xc7h\x86t\x1f\xb1S\x96;\xc8~7|H\x11\x03\xcb\x01\xf3\t\xa3\x08\xf8-\xf2;\xbf^\x10+\xe2\x9bT\x13\xe0AL\xf4\xa5\x19\x16\xe5\x9a\x07\x14\xba@\x18\x84h"x\xbe\xb0\xf3n\xce\xae\xc4\xbc\xbc\x9c;8X\xf5\xc9\xb20\xf5(".\xc2\x00\xf6\x90\xed\x16}\x8e$\x19\x8c\x80H\x05\xf2\x07/\xe0\xcaqV-18\xeb?\xae\x04\xcbw\xb5\xcb\x00\x07\x1dL\x82\xfa9\x03`\x07\xe3g\xb5[D\xaa+\x86\xb9\x9d\xbd\xca\t\xda\x06\xb4V\x00\xd3\x02l\xbf\xfe\xe1\xcd\xfd\xa3&N\xc7\xfa\x87Q,\xe2a\xf5\x0f\x82\xfb0\xdc\xdf\xb5e`\xa3\x90a\xe7\xd0Pu0\xa5\x05\x87R\x81x\x88\xedQ\x17E\x9b\xe8?\x1b\xcdu\xa0(\xa6jES\x95a\xb0\xb7\x06\x1c\xb0\xe1\x87|\x14\x9ev\xd1r\xec2\x88\xd2r\xe8\x89tn\xe0\xad\xb3\x8c\x90\x170$\xe8\x8a\xbc\x98\x11}qz\x86\xdd\x12F\x11\xe2\xa4\x87W\x89\xa1e\xaf]\x19N+\xae\xda\xd5\xfbYO\xa6\x13\xc9\x1a\xb1\xa44\xdf\xc1\x18\xd3\x1d:\xd8uH\x90\xd0\xf5w!a\xfb\x11\xcf\x05\x87e\xd8\x1b\xa1\x88\x89\xe0\x02,\xfaj^\xecas\xe7\xd5l\xd6\xf1\xfd\xc1j\x1e\x15\x89e\x0fV\xd0J\xb3\x06m7\x0f\xa4\xa5\xd0\xc1&\xb2X\x06\xb7\x9f\xd4.^f\x91\x19zh7^\x02r\x1a\xf7n7o>\xaa\xdf^nLK@E\xa1\x83\xdcJ\xf6\x95\xc1v\xd9\rn\x948\x1e\xec\x14\xb2\x8dx\x14D4\x82\x9c\xf7\xc0\x88V\xfa\xcc\xc2\x98\'H\xfb\xbcRt\x8a\xedQP\x83\x91BJ\x03\xf9\x19@\xfc\xdb\xf5\xda\xf9\x85\xc6}\x91M\x00P\xed\xe1\xad`i\xa99qv\xf5\xd5|\xecd\xca\xe0@\x1cK#\xe7*\xf1\x0e:ju\xe9a\xed\xd23\xc6/\xbc?\xae\xa66\xaf\xc1\xa4\x86vB\xb3\xb0C\xc2C\x1da\x18\x84"\x14\x1d\xc4\xad%\xcfw+h\x1b\xe7\xb4#\xc2\x0ci\x86\xaa@*\xa8\xa0\xd0\x84\x19\x1d\tf\xcfp\xd4\xa4\xf6\xcd\xa3`\xeenm\xfet\xed\xe2t\x82 #\x8b$\x16\xf5\xcc\'\x04?Y\xe5p\xc4\xa9\xd3\xaf\x94\xddR\x9a?b\xc3\xe5\x8f\xc0\x8b\xf8\x17\x86\x85;]\x7f\x10X\xbb\xe2\xa4%4#\x0f\xf2\x1f\xf8\xa8\xd27\x18x\xe4sj\xb7%~;\xeaW\xf5sv\xbe8\x98\xfb\xeb\xcbk\xeac\xd6\xcb\xf5\xe3\xcb\xf0\r\x07\x87\xb4.\xd9\xe7\xfc\xb5\xea\x82\x04\x0c\xf1\xe9\x16\x9c\xa2O\xc1_8\x9c\xa1"\x9bI\x9b\xbb;t \xc5F\xe1Yq\xc5B,c\x14\xa6J\xa7\x96w\x0b.F\xa9\x1e=h\xde\xff\x06]\r\xb5\xe0\xea\xd2\x85\xfa\x83\xbb`\x82\xc2{(f:>\x03\x9f\xa2j\\\x9a\tf\xef\x06\xb7/5\xaf\xcc\x05\xd3\xbf4~\xb8U\xfbn\x0e\x04(h\xc7\xe6\xc5g\xf0\xa2\xda\xa3\x0b\xb1s\xa1h\x9bF\xa9\x04\xdd\xd4\x9f\xfc\xe5?\xe12 \x1f"\x8f\xe8\x0f5~\xf9aD\x9f<\x12C\x13@\r\xf4\x11\x01\xd9\xc4\xdc\x0eqSA{\x80\xaf\x0ctl\x15\xab\x85\x01\xe0}o\x10=@\xf2e}+\xdd\xf5A\xd1z\xc7z/\x03\x0e*9\xa5\x82\x08x\xa2qw68\x8d\x8aT\xbb\xad\xbf\xbe\x9c\x96\xc7\x7f}yZ;\xa6\xfb@)\xa0o\n\xea \xdd\r\xe7\xd0\xf5\x81\xf5{\xa6\xa5\xac\xe3\xe6\xd3\x8a\x90\xe0C\xf4M7\xc2Z\xd6\xbb\xf0P\x86\x96\xd5\xa1\xb2\x12\x8aud\xb3\xa2M\xe7\x1c\xfa\xaeV\x11\xb8\r\xfd\xfaj\xb9\x8cV\xe2\xe6\xf77m\xda\xc8\xae8\x9a\xfb`\xb3\xcb\xc16\'\x9e\x06\x8b\xd3b\xe9\x1b\xbf\x07\x96n\xbe\xf8N\x19\xc0\xe3\xc1\xed\xab\xf4\x0e\x85\x02\x0c\x9b\xde\x9d\x85\x1d\xc6NR6\xf6\x87m\xf44\xba/\x1a\x11\xd6(\x7f9f5~\x99\xc4\x80\xabx\x17j-\xd0\xa8@1\xf4;\x165\xf5\xa5\xc5\xfa\xf5\x87\xe2\xed\x13k\x87B\xe6\x13\x96+xN\x83@\xad\xc3p\xa2\x1b\x07\xdc\n\x872)\xd462\x0c\x06\xf0\xc6.\x14\x94e\xe0\x07\x07\xe4\x0cZ\xb9\xb3\xf7@2H\xa0\x19\xbeE\x9d\x0f\xe2\xf2\xc5r\xfd\xfeY\x10j\xa8\xb2\xe4M\xcd\xab\x0b 99\xd6\x19K\x18\xf9N\xb6\x0c\xceG4b\x8910Z\x1d49)\r\xc5Z\xde\x11\xa7<RvI\x86\xd3Yy\x15g\xc0\xf3\x0e\xab\xb04\x08\xb9G\xb3Z\x8f\xd4\x9e\x00b\xae\xb2i\xc1\x8b\xb3\xb5\t\\\xd5\xb8{*xt\x1f\xd4\x84\xfc\x1e\xa9\xec\xfc\xf5\x00l\x14\xda\x8e\x8e\x1f\x1d\xd8\xdd\x8fQX\xa0.\x01\xb4\xb3\x0c<\x83*~$\xdd\xfd\xfb\xdfo\xee\xce\xa0\xdd\xc6\x8flC$\x90\xd1\x86Q\x12\xfc,&C@Qj\xa7\x99\xb6\'n\xf2@\xd5\xcd\xe7\xd4\x96\xdc\xa2\x8a\x90)\xdb\x150=\xbf h%\xe5\x13<\xbb\x03\xda\x86s\x08@R\xe8X\x80#\xcd\xd2\x9d\xb6Y\x7ft2\x98\x8f[\x9e\x9b\x90\x92\xf4\x86\xfe`\x11\xf8 \xa0\xe9Of\xed\xd5\x17pp/\xd1\x8f\x9b\xfa\xc9\xda\xd2\xfd\xd1\x96\x8f\xde\xff\xa0\xfb\xa3\xf7P\rO_\x02\xcc\x00\xeb\xc3\x1eC\xca\xe9\x03\x0b\x80\x80\xb6\xc1L$\x85\x8a;\x03\xbb\xc7B,\x81\x19\x80g\x97\x05y8\x80\xd6\x01\xa8\xab\\5\x8b\xe1\x17 \xb62\xb0\x0b)\xf2\xe0\xd1\xdd\xe6\xd5k\x12{R\x01q\\\xf1\xd2\xf3\xd5\x95\xd7\xab\xcf\xd1*\x02-Y\x9f]\x11*\xa2\xa3\xad?\x9d\xab}\xa7\x15n\xa9\x8c\x18\x1fL\x99F\x04\x01\x83.\xbd\xda\xf2\x98\n\xec}\x8c\xe8\x06\x18\xabE\x17hY\xfb^\x12\xb0/\x81=\xee\x1eUQ=\xc7\xaa\xfa\x86\xb3\x00\xf4u\xe39\xc6\xc5)\xda\xc2F\x13\x9f\x0c\xd0\x14\x9aK\xb7Q^\xe9WJ6\xe4\xf4L\xfd\x17\x1d]\x91@=\xb0P\x01\xa8\x8a\xac7#\xc2\x9f\xe9\xc4/\x18\xce\xcf\xed\xc3`\xb2\xd8\xe0\x9b \x1a\xcd\x84\x18z\xab\xcc\x94*\x1dca6\xabZ\xcc\x0e\xdb\xc5!\xe2\xcd\x99\x9f\x00\xa5L\xcc\x1c\xd1C\xd2\x7f~\x1d\x85\xb7\xc1!\xab\xaf\xbf\xad_\xbc\xc2\x91d\xd8M\xed\xa6\xd6\xf2\xb0d?\xdb\x91 wF\r\xa0\xc7\xfa\xc5\x98\xee\x1f\xc5\x94n\'\xfeg\x0b\xb8\x91\xc3\xce\xd1\x83[?<\xa4"%;$\x08\xea\x982us\xf7\xc7\x12{\xc3H/\xb0C\xc1\xce\xe7\xe1\x8f\x1dy\xb7\x04Z\xfb/\xca\x90F]\x80\x90\x03K\x02\x9c\xf431~\x99\x14Aw6\xeeN\xd4\xce\\D\x81v\xfdo\xcd+\xb7\x90\x7f\x1f\xcf\x82!\x1b\xb7\x921d\xdb\xef\x16\x01z\x1f\xb62\x9a\xca\xc2R\x9c%\xd9\xda\xcet\xee \xf5g\xfc_\xaa\x02.\x05<\xaf\x9c\x0e\xf2\xd5\xa2\x11@\xf0\xd9\x00J\xe5\xb3a\xfe:\xd5\xfa\x9a\x9c\x83\xe6&\xbch\x7f\xcf\xff\xd8\xdf\x0f\xe67\x86\x9cQ\xb7\xc8\'hR\x8d\x89\x85\x9d\xe3\xe4o\x1e\xach41\x80w\xfe"\x99\x8cP"\xdb\xd6\x96\x8f\xdf\xfd\xf0c\xed~P\xee\x9e\x1d9%\x9d%^\xbe\xe5c\xf0W?\xd4X\x14Q];\x7f\x07\\\x8b\xe0\xe1R\xfd\xf5W\x98\xb31\xf0\xa9\xa5\xf4 \x9am\xff\xd2\xb3c\x7f\xef\x9e\xdd\x84\xa3\xad\x1b\xd8:\x80/\xf1\x7f).M\xa9\x18\xf4\xa2\xa3\x90J\xc2R\x81\xc7\'O\xb9]<\x8eN\xfa}\xf0x\n\xe8\x13\x94>\x1a\xd8\x06\x00\xc1\x99\x1b\x00\x9b\x06XB\xb4:\x9f\x8c\xf0\x99`\x18\xc7L\x99L\x01!u\x08y,\xba\x01S\x08\xb4\x89\xe5Q\xe0t\xfb\xde^k\xa8l\x97\x86I\xfb\xc4\xb2hlzc\x18\xf6\xc5\x9d\x96\xd0\x1e\xfd\x9c\x15\xb1\x08\x07N\xbf?\x9e\xc6\xb3P\xb1\x04\x85b^\x03h\x937\xd4\x05\xd4\x80d\nR\xbb_\xe8\x8e\x88\xe3\xb3=\xdb\x91\xdcv\x03\x8d\xa0\xb1M[\x85\xef\x8c\x8d\x8f1\xc5\xa5\xba[\xdf\x10\xc6\n\x19\x91)\xf3\x15\xa3\xa9j\xd1\xa9(\xa6Pn6<!Q\x10x=|\xfeg\xb2\x01\xfb\xfa{w\x8f\xa9\x85\xb6$,\xf4\xc5\xf6\x9e\xc4%\x8e\xd8\x8eZ\xc1\xf4\xdd9c\x84_\x0e\x96\xba\xde\xef\xf4\xedA\xa7\x02\xe6\xb5W\xf6Sz\x99\xf7~\xfb2\x94\x92\x92e6w\'/\xf3~\xeb2i\x95\xc1\xf8t\xf3>\x95\xb8\xda\xef}\x810\x93\x0c\xe0\xc4}4^\xa3\x7f\xd1\x8b\xa9%\xf5t&\x02\xa6\x96\x0c$\x8e\xe0\x93\x83p\xec\x1d\xd6&\xf4\xca\x01H\xfa`\x8b\xfa@\xf2\xf2[\xc5\x7f\n\xc5\x8a$\xd0\xb7Z\x92`W\xf9\xf3\xadb\xaf\xa33\xe6\x14\x87\xe8\x111\x17\xd5^?H@\xe9\xc7\xb6\xeff?\x01i\x1eGj\xb8 e#\x08\xban\x05\x1df\xc9]d5\xd0B\xf4\xcd\xfb\xf4\x8d^\xe9\xc3\x84\x95\xfe\xd4\x87\x0e\xabS\xe6\xb4h\xec\x04}\xfeN\x9f\xa2S\x85\xbf\xc2S\xfah-\xe2\xa6\x08\xba\x1f{a\x02\xd0\x9c\xc9\xdf\xca\xee\xa7~u\xd7\xa6\x84w\xef#Cf\xb7\xe7\xfaN\xec\xbdE\xfc\xac\x9f\xf2\xf0[\xb5]\x15\xbe,\x81\x8fS\xb2\xef\x1dU\xbf\xe2\x15\xb6\xe7\x8e\xd8@O\xb9v\xc8\xa6\xf73mlR\x80\x0f\xf1\xf1\xe0\x87\x1f\xf0g\xe1\xf3\x828\xfa\xf2C\xbdS\xc6\x08~\xf6\x91\xfa\x0c\xcd\xaf\xa2J\xab\xab#\xeb\n\x8f\xac\xab;\x99\xe1vb\x89\x8c\x93x\\\xbcBW\x1b"\x0e\xdf\xbcy\xad7o\x97\xbc\xf1\xfa_\xff^\xec\xf5I\x02iG\xd9\x81\xfd2\x17\xb6\xc1\xb4d\xeci\rM#\x9c\xc5\xa6\xcf6\xab\xcf\x06\x89j@x\xdb\x954\xf8\x92\x99p\xe5\xf7\x12O\xfb\xc8\xda\xeb\x92\xf0\xe3%4\xb7+K\xb3\x9f\xedP\xf8VYb\xf8\xa5W\x065K\xb2\xad\xb4\x85xB\xcc\x85\xbe\xaaN-\xe6\x9c\x9c\x9b\xb5\xd1\xc5\x00\x91\xd5\xfd\xc5vKs(\xe8O\xd6d\xf0\x14\xe1\xda\x02\xd4K\xeeB\xf9\x1cFN\x9e\x93\xefb@\xc8\xbbX\x85\xd5\xbe\xffv\xf5\xc5/\xac\xe0(Hq\x97-\t\xe3\xad\x98ZVi{N\xcfG\x13\xf3q\xe1\x19\xb3*\xc8\xf4\xd5A|2\xc1\x8d\x82?\x90\x88\x88:\xf4\x9f*#\x98\xaf\xa2\x9a\t\xd8\x07\x08!)k\xb1J`\xdb\x89\x89\xc1\xf9N\xdeHc|\xb2\xf6\xcdy,\xe5\x9b\x9d[]\xb9fu\xfd\xfa\xea\xbb\x8f\xac\xe0\xe5S\xb3\xc8\xa0\xf1\xecQ\xb0\xf2e\xdc\xc4\x107\xca\xf51\xf0\x8d\x9c\x9b\x0ekA:\xac4VOtX\x95*\x90k\x86cV\xe4p\xc1i\x1a\xcfe\xf0\xa3\x8f\xe0$\x01\xb5\xfdF)\x89D\x05\x08\x92__^\x03C=\x98}\x04\xa6E\xed\xc2s\x8c\x05\x93\xdb\x8ev\xaf*\xf6HE\xd1%A\xe9\x82\xeb\xfbxz\x1e\x06]A$\xfa\x94\xfc\xd6\x87\xab\xd3\xb2h\x88q\xa9\xdb1\xc1\x16\xb8O\x9e\xf2Z\x18i\x1cvf|\x04\xd3\x8f\xb5K\xc3\xfef\xed\xecW\xf5\xa7\x171:\xf5\xf0\x1bpN\x014L\xbf}\x7f\x1f\xdc\xfe\xda\xe9\xe7\xe0^\x04\xb3\x97V_\xdc\xd254mp\t\xd6\x7f\x9a\x8cp<\xadL\xa7\xeb\xf7#\xf9\xa7\xb9\xfe\x03?C\x13\xd2\xc0\x1e#\xceB\xbbV-\xad\x97\x00\xb4\xe9\xb4\x1f\xa7`\x92\xf0\x84\xc1G\x89\xf3\xc4JQ\xb8nT\xa5<+#^h\x18f\x87=\xb0\xdb\x95\xc1\x1a%kX\xaayc\x12\x0b\x8aN\xbc\nn\xde\x0f\x1e\xddgs0L3GjU\xda "V\xa7BaU.F\xa1\xaa\x94\xa3\xa9L\x9cf\xf8A^\x17\xa3p\xf1URq\x86\xc2\xaf\xf9\xb09\x0c\x87\x04 \x06\xba\x94\x19\xca\xd6\xbb6\x0e\xd8\xbeI7@O9z\x9e\x11P\xbb\xb1\x14<\x9e\x91jB+,\x93z\xb6\xd0\xb8r\x9d\x03\xc6&\xf7\xc3W\xcd\x1b\xa7\x82\xe5Y\x95Y\x98\x0eV\xbe\xb3\xba\xac\xc6\xd3g\xf5\xc5\xcb\x12\xc73A\x95\xc0P\xd9\x89\x9f\x9d\x9dC\x82\xd1\x82\xc0\xda+E^y{\x00L/\xb5\x19\x14\x16\x1b1\xceB\x19l\xde\x9a:9\xaa\xf6\x10\xc0W\x97\x1e\x06\x93\x0b\xcd\x13\x0b\xd6\x1f\xd4\x8bv\xff\xd1\xaa-\\\xa9?\xb9\x1d\x07\xa9\x00\xfa\x12\x05h\x0e\xcdt\x0c50\n\x99N%\xaa\t\xcb\x97\x9d\xceA\x17k\xb8\xf2\xe9rJ\xbd\xf5\xdf\xfcw\xd2\xff\x96{\'\xf3G8J\xb1\xe0\xf0\xc1\xdeOv\xef\xd9\xd7\xb3c{_Of,\xbaQI\xae\xa0\x1e@a\xa7\x8e\x81\x19V\x8e\x90\x12M\x98\x84.z\x15-\x0ey\x97\x9cNQ\xbb\xc4@\xc5\xdc\x97\x11n1\x0e\xa6\r5\x86\xdb\xfd\x03\xc6\xbd*\xe92F\x15\xd2]\x1d-\xd2\xec\x1d\xab+\x83\xa4\x19\x12C\x04\x9b\x92\xef\xe6\x92\xc1g?\xb2\xcc\xd5K\xc7Ht\x87W,\xaa\x18\xac\xc4\x93l\xadO\xaa~X\xc6\x13\x12\xa6\xa1\xda\xb47\x19\x8b\'\x19:\xa99~\x0b\x95\x14I\x10C\x83\xc5\x91@\xfe\x18\x1aE\x87\x0e*\xad\r\xff\xd2\x1e\x04\xb9\x94\xca\xf4\x88\x9d\\\xa8\x84\xbdl\x15\xb1\x88U\xbbx`\x1b\x95^u*\xc3\x1e\x97\xda;G\x9crx\xba\x8e\x82\x7f\xa6vz\x1c\xe4[(e\xb8~\x82d\n\xc6rV\xfe\x86A\xb2\xa5\xc5\xda\x85\x95P\xa3\xbe\x19\xfcX\xad\x1b\xee!*J\xa2;\xd9\x0b"\xd1)\xab\xca*\xaa}#\xbe\xca\x81MR\\C6p<)T\xb5\xaa(\xa9U\x1a\xc4\xc1F\x94`\x88\xebh\x07c\xac_\x89\x7f\x07\xf8\x8b\xc2\xd3&\xe1\x89\xb9`\xbe\x80H\x08k\xa3X\n\x94$#\xcd\xfc"\x8a/\xeb\x95\x8e\x91\xdc\xab$h<\xd2z\xb7\xa7\x1a\xb7^\xc5\xf4]09-\xf9@\xd6\x86\xc4I\x98\xfc\xd0i\xb1$X|\xafZ\xce:*\xc4\x17\xee)\x93\x0czT\x85wXX7u\x0c5\xb9Yi\xa5\x94\xb5*,\xf6C\xc8c\x1a\xf9?\xc7\'\xea\x7f[\xae_y\x85u\x14T\x81b\x96>\xf1F\xc2\x80_\x12@"\x0bx\x1b\x86j\xa6RU\xfe\x10\xcc\xa0J\x1a#\x96DT:\xc9\xa4\xbe\xad\x82\xfe<\xda\tn\x19\xf6\x02\xb0N\xeb,\x81\x97\x08\xf2\xa2\xf3/%\xf5\xbf\x0e\xffc\xc4\x19(\xa5\xa8\x9f\xa0~\xf2&\xa0\x15Y7^\xfc\x89\xd1\\~\xf9X\xaa\x1d\x16\x99lup\x17S7V\xef\xce\xb0\xb9%V\xfel \xf0\xd1\xaci\x06\x8a\x00\xa1\xc8(r\x1c\xa7pH\xc7\xb5\xc3\x18\xae\xd5\xef\xa2\x8e\x00+#\xdd\xbd\t\xa4#Qt\x9b\x13\xef\xc3\xaa~\x89\xefj\xa0\xaaE\xaaa\x8c\xc7\xa6\xc3#\xe7(\xf4:\x80\xe6\xe04\x88\t\x94y\xd1\xf7\xd4\xaf-6\xcf\x9dl\xb7\x8d\x9c\x839r\xce3o\xb3v\xec\xf9|\xd7\xbfZ\xef\x8a\x03\x93\x82\x7f\r\xa2\xc1\xde?J[C\x05\xd0.\x10<\x9aH\x07\xedN\x8e\xdd4\xc9\x80 \x909\xb7\x0c\x0c\x81\xf5i\xe0+\x00\xc3\xe6<\x87\xd5\x1d%\xf1\r\x96\xbd+\xe1`\x0e\xb5\xc5\xb6\xda\xf8i\xaav\xe9Jp{e\x1d\xbb\xed,\xd9\x98\xdf\xeb,\x1c\x86\xb5\xd3\xfc\x87\xbfm\x7f\xb9\n\xae\x17-\xda\xef\x1d\xa6?\xdb\x1c\xe8\x0e\x90.1\x05\xc5\x0c\xab+\xec\xbd\x9c;(\x02\x88\xca\x84\xd1\x0f\x0fw\xc2\xc23\x94\xfc\x93?\x04\x8b\xcf\xc1\xb9\xd2B\t\xd3\t\x93w\xde\xb8%\xee\xbb\xeaDa\xd7\x9d\xe6C\xe80\xf7\xd9V\x02q\x01\x1b\xf3I\x08y4H\x8aQ\xb7^6#\xc9GT\xbfn\x8e\xcf6\'\xceK(4|\x88\xf9\xc6t&\xdaA\xcdzKx\xe8\x90\xc4\xeaMwZ\xbf4\x16"Pq\x0c\xf3$\x91K\xc6\xc6\x92\xf7\x895\x9c\xff\xee\x94=1s1V?T\xc6\x02N\xeaB\xf3\xff\x99\xab\xbcc\xc6%h?\xf42\xe57D\xfbZ\xe9\x91\xea\xdf\xdb+\xae2Z\xb7\x9b\xc4\xbaE!\xa6\xdf_{pr\xf5\x158O\xd7\x94iF\xa6f\x00\n\xc5\xb0\x87\xd7\xc6NL\xab\x0f\x1a\xfd\x06\x9d\xa1\x86g\xd6\x1ccCE\x10\xca\xc6J\xa4\xa4$\xa4\xd2\x82\x93sm\xee\xa1\xe1V\x046i\xb5\xf9l\x95\xf2v\xd6\x19\xf6\xf2 \x9b@\xe54\xffv\xb9\xf1\xf0\xb4:\xcd\tU\x863\xc1\xde)\xc8#n\xb71=\x04\xa2\xe9Pz&G\x00\x0c\xad\x93\xe45\x13d\xea\x0bnr\xa0/6G\xbe\xe0n\x06\xfeB\xbb\x8b\xa1s\xad\xa1\xdel\xd5\x16\x9fj\xd0\xe9\xafXCE\xb2\x83\xd6\x87:9\x84@\x96hL\xfd\x14<\xfe\x92\xe5\x10\xe6\x9b\xa3\xf1\x04\xd0\xc1X\xc0\xb4\xf8\xd4\xec\xe0\x00B\x10\x84\x81\xa7\xffj\xbe~|\xb9y\xe5dp\xe6\x06\x9b\xa6\xf1\xf5[\r\xef8&\xdeIBAW\xb7\x00(\'\xf0\xf3\x8d\xda\xb7s\xc1\xdctcaZ\xb0\xd2\xd5m\x05\xe3a\xeeO\xb2)\xf6\x10g\xcb\x84\x0b\x80\xbbb\x00\xc0k\xbf\x90\xe8U\x0c\x90h\xda+\xb5]\xc2hq\xe0\xde\xb1\xd2\xf1\x1d\x00E\x98\xfd%\x14N\xdf\x04\xde\x912\x0f\x89\x15;\x98N\xc9\xdb\x03(;\xdd\x8aS\xf0\xd3\x99\xad\xa1\x16&\xdaE\xf0\x0f\x92\xa7&NZ\xab\x7f\xf6O~\xe8\x9aY\xff\xa4\xden\xbah\x99Cqk\x08\x7f\xda\x85\x88-\xe2\x7f\x18\x12\xfdzY\x99\x8c\x98?\x8c\xd2\xfb\xc6\xd0\x0bb\xba\x97B\r\xf6N\x96\xce\x04\xc7\x7f\x026iN\xcd\xa2Q#\xbf\x1cK\x8aX\xd8\xb9\xd6v\xa1*\x85y$\xa9\xaa\x14\x1e\xb7\x81 \xbf\xfba-\x12\x88\xb2\xb2m\x95\xdc\x92U\xb2\xb3\x87IP\xb0\xa1\xcea=m\xe5R\xa2U\xcaCL\x91\xad\x08\x16lH`\xe9\xb0\x15\x89\x9eD\xfa}qn\rK^\x19,\xadv\xbc\x9cyxl^\xd9\x1dr\xb17J\x0cf\xf5\xd3L\xfc\x08\xd4\x831\x93T\x7f\x9chvb\xb8\x14\xad\xcc\x82wDY\x9b\x05\xb66M\x964\xe3Y\xa1\xd1,\xc5bWG\xd5\x12coc+\ro\xee\xd7mWI\x86RW7XJm\x80\x1f\xfb\xadfJ\xa2E\xa0\x16\x8b\xd9\x04\xc9VlWh\xc6Z\xbf\xb7\xba\xc3\xc7`]\xcc\x0c\x86\xcfu%?\xb7.\xb5\x9e\x10%\xa7@x\xb2V\x8f\xbeZ\x01\x92\xf4\xeeO\x9c\n\xbd\x1a\xc8\xbc\x04\xda\x1b\xf0\x14O\x17\xaa\x08\xbc\xa1&[VH\xd4\xba\xcc\x89\x9d\xc6\xd9\x9aZW\x01EjW\xbd-*\xdf\xb6F\xe4\xe4\x9bV\x92>\xbc\xce\xd8\x07\xedV\xed\x8aE&>\xa3F\xf8X\x83\xa1\x85\xc9G\x1f\xcc\xbb\xb2W\x1db3/K];\xf4eh\xde%\xca\n\xd6]\xa6\xac\xd0\xbeRc\xe2G\xf8\x96\x19+\xeac\xbe\x95\x88\x10M\xf1\x7fKD\x8c\xd8,\x1a\n\xa5\xcd\xf4\xbf\x83`\xf3\xd0?\xbc!\xf6P\x0b[lSd0\x02\xfe\x0bE\x86\x1c\xf0\xff\x7f"c\x8b\xe1\xf9\xbe\x95\x18HJ\x00\xaa4\xdc\x1b\xe5@"\xef\x18\\\x93\xc0/-\xb6\xf1\xf6J\x05\xddq\xdb:\xe2\x94\xddA\x97:\x05\xb1P\\\x82\xaeT\xd0\xff\xe9f\t\xe9\x99\x99oq\x1e:7`9\x0b\xc7\x8aT\xbc\x95\xab\xc6\xc5\x10<\x7f\x07cK\xd8\xff\xc8\x7f\x93\xad\xc7n\x9d\xb2\x8bU\xd7\x9f\x19\x07\xeb\xb04\xa8\xb4\xc1~\xc51\x07\t\ti\xa9\xbd\x8e\xb4Y\xa7\xba\xde\xa7:lj\xac\xa6J\xd3\x94\xd8fi]\x9c\xfd\xd6?\x08{\xb7\xf1\xf9\x0f\xa8\xde\xdbn}<\x9f\xf0\xf0\x87\xad\x0f\x1f\x8ae\xfd\xf6`e\x93\xcdg \xcd\xe9e.\x9d\xd4\xb1_;K6OA\x0b\',s\xa2\xe4\xa3\x94\xd7RL\x94\r\xf1\xb6\xd9\'\xc3\xf5\xc0\x17mm\xe7\x0fK\xd5f1\xec\x1b`B\xa0^"\x8e(\x19\xad3\x85\x88?\xccT\xc0\xfd\xf3\x0cT,\xee\x06\x0f\x801\xa8\xa2u\xff\x90\x18b_\xc5+\xa9P\xa7\xd1,\x00\xdb\xb5\r\x1b\x92\xa0\x07\x0f_B\x8dF$\xe5\xac\xb8\xe8Z\xa8ae)\x10\xb1j\xd7\xc4)\x1b3?\x05\x13\xf3\xb5\x87\xdf\xbfe\xd8\x10\xe5f\xed\xf4\n\xbey\xfa\xf1\xdb\xc5\xf4tX\x85#:\x92\xa5\xd2a\x1e:\x11\x1d,\nw\x83\xa9\x00\xce\xe6\xb6\xc4t\x99\xe1~k\x14lTx\xf2\x1f\x1e\xff\xaa\xd8\xe5!\xa7\xa2b~\xe4us\xbd\x1fw3\xb7\r}\xc9v\xd7\x17$\xfa\x07\xc6\xbd"\xc9\x82v\xb1/\xaaL\xe5\xd0WK\xc4\x8b\n\xaf%\xb8NA/\xee|h\x8e\xff\xd8\xb8yn\xf5\xf5b\xed\xc2\xf3\xff\xbaX\x97\x02T\xeaR%\x8b\xa8\x8b\xff\x92\x08\x8a\xcd\x1d\xae\xef\xe32E|\xdc\xac\xeb\xfb\x7f\x1a\xe02\x13i\x91,\x08\xb0\x0e\xa8\x1eSoEw\xf7\xecG3P\x87\xe92VVq5\xb5\xee\x00U\xa8\xac\x92\xf5\xed>gc\x99\x1d\xdc\xbcs\x04g)\xe4\xca^\t\xdb\t\xb1T!kc\xd0\x8d\x1b\xe0\xb1\xb8\xc2\xa7l\x08\x96\x82\x00\x89\xc2++\xe8\xe5\x17\xeccLZ\x8e4\x96\x81\x91[\xaeTKT3\x8f\xf9\xc6\xe5\xdb\xd8\x92IU\x01,\x92W\x97\xce\xd6\xce\x9e\xc6\x19\x1f\x97\x16u\x1f"\xf7x\x05+\xe7\xb8\x85Xz\xe2\xa8\xe6=X\xbc\xda\x1c\xbfA}\xa93XI\xbd\xf8\\ij,\xc5\xec\xef\xdd\xbd\x0bGaH[\x84O\xdd\xb4 \x1b>\xde\xde\xd73\xf6\xae7\x80I\x9d~\xb78\x88fM\xc5-8\xc0\n\xdb>\xda\x94\xe9\xc49Y\xe9LB\x93\x17n0\xe5K[ \x17\x1eD\x9a-%3\xaajj\xd9J_]\xf9\xb6>\xf9C\xfd\xe2/(\xa4\xf5\x0c\tNY\\=\xc9%\n\xdc\x1f /\x8ev9\xa8\xa9OR\x95L\x06\xb1\xd4<\xa4\x13jO\xa5\x8c9\x83\xe7q\xf0\x10\x86\x82bU\xba:\xd5\xc4\xab\xbd\xa9\xe7T\xed\xc5\x1b\x1c\xa4\x81]\xacH9\xca\x186\xd3Fg2\x043K\xd8$\x1c\xa9)2\xb7\xa1\xbb\xd3\xd6\xd8\x07\x95^\xafc\x17\x9c\xc5\xb4\xf6\x08xzQ\xb5)\xb4\x15\x8fqJ\xbeM=q\x0f\xd60ZR\xa0\x87\xba\xd7>b\xbby{ \xaf{\xe9[j\x88\xf9\'R0\x88C\xc6\x8cq:\xb2Q)\x8d\x8cl3\xa1\xb62RPil7\\K\x9a?\xd8\xcd\x8b,\xdc2?\x80\xcaE\xaa\x03\xc0\x8f\xc3\x91\xe9\x15}\x91\xe9\x07\xb4\xc7\x12\xaa\x05\xcco\xb4\xd9\xa19\xee\x00O\x14\xa4OR\xdf{Ku\'\xeeW\xb3\x1e\xb6E\xbf\\\x0e\x1e\x7f\xd9\xf2b\x9d\xdcOE\xfb@\xf5\x10\x1e2t9v\x0cV\xa9\xccV\x91$\x00\xeb\xbdV\x1b\x8f\xe5\xa8\x1a\xfd5\xbf !\x7f.\x80a3\x89k\xb5\xae\x1c\xe79r\xd8\x9d\xc8|H\xe6\x1e\x13T\xf3\xee\xa5\xe0\xf1\x8aD\xd5"\xc6\xbf\xb2L\xc9%Mj\xa8\xcclM\x88\x0f\x92\\P\x14\x15n0\xe2\xa0H\x0b\x84nkQ3\x1d(AJi}\x97\xbbD\xc9\xb0\xa0\xe94\xdc\x95\x8au2\xdcvf\xd4L\xb0T\tn\xde\x17kC\xb7\xafE\xbc|\xc4h\xd8{\xa1\xcf\xec`\x9b\x1a\xe9C<\x8a\x80M,\x1a/\xc7\x1f\xa8\r\xf1g\xf1\xfd\xb3\x91\x8e[\xe2\x03\xa5U;\xacR\xbe\xea\xb3\x06\x0c\x0f\x97:\'\x00\xb5\xbb>\xdb\xd5\xfd\x85\xe9C\xc0\xf9\xa9\xe8\'\xf3\x82d\xca\x1f<\xe0\xc6c\x8c\xd5\x1bg&N\xc6\xf1\x93\xd2\x87b\xb4(\xc7q\xe0\x1c-1\xe5`\xa7\xbf\xe9Z\xc5\x9c\xa1H59\x1f5\x07\xc0\xd3i\xf3\xb9\x8e\xe8\x83\xf9\xf0\xb1\xe8\xcb3\x99\xc4\x02A\xac\x1f\x02!W\x00\x1f\xc2ER\xa1p\x0fQ\x80\x8b\xec\x8e\x15+l\xf1\xb68\x11\x82\xa8\xbb\xc7k\xdf\xcdc\x01\xd8\xeb\x13\xaa\xd1P\x84ds\xfc\xca\xea\xf25\x95\xe3\n\xdd\x06l\xe0\xa2\x12\x8a\xa8\xdb\x90P\x08\x08\xfev\x9e\x86\xdcE\xe8\x06YP\x7f\xa3\xb0I\xeeD"\x11Y"\rF\xd5\xa3c\x96\xe6\xb0TkC8\x95r\xe6\xf3p>:\x8a\x0eV\x05b@\x8a.|*9\xd3F\xedFN\x95\x89\xadXd9 l\xce\xd3\x19t\x94l\xf2Yp\xfcJld\x17\xc7\x8c\xc2!\x9a\x14I\x0b\xab\x90\xda\xa7\xc5\x94\x1b\xdcOK\x92\xed\x18\x89\x9b&\x04:\xc7\x90R\x8cy`D&X\xcc\x95Q>p\x18d\x0bcl\xd1\x85:\xed\\.m\x84f\xf8\xa7\xe28jXb\xc0m\xa4U4\xb7g\xccs\xc6\xfc}\xe4\xe7x\x90\ne|t\x82\xb2{_5\xef_P%\xc6\xe8"b\x01L:\xf2\xdb\xcc\x18\xf7\xb6\xa2\xc1v\xe9q\xebT\x17\xc3\xda\xb0\x89\xc4\xa4\x9dG\xf5\xd1e\x8f\x81]Y<\xec+\x87K{\xf9r\xael\x81p=V8\x01\x04\xcc\xe2\xfa3\xaciZ]9U[\x9e3\xa5{\xe8\xcc\xab@\x8e\xb6x\x99\xc9\x8a\xe2S\xe8\xa4U\x84\x17z\x8e\xda\x05,`\xa6\x92\x90\x9c\x03\xce\x01O}d7\xb0\x8c-\xe8>\xb1-\xbelc\xe8h\xd1&\xa46\x93\x9b@%*\xbbD\xb35\x1erMfpn2\x98\xbb\xcfF}0\xf9K\xb0xB\x8br@9\x98\x8fIA\xda#v\xbeJp\xe3\x92\xa1a\xdfI\x9f\x87;\x88;\x03\x14\xc8\xa9\x8cx\x1b\x1d\x16*b\xbfc\x8f\xa2:l>\x8a,\xbb*\xd8Mo\xb8"\xd4\xb9\xd6\xbcv%\x98\xa4\x8c\xe5\xd2$\x98\xea\xd8Y<\xfb\xa8q\xe2\xd5\xea\x8b[\x8d\x9ba4\x9a7\xc1\x87\xd2\xce7\xc1\x02\x92\xb0j\x9b`\xef \x902:uL\x1f\xd2\xd0\x9bncO\xf1\x17\xe9 \x8b[ APq\xf2\xc78\xbc\x82g\xe0\xa8\xd2\'9\xed\x84\x10\x8b\x84Y\x10\xdc\xa7sp\x10fa\xa6IOh<P\xb0Ezd\'\x9f\xe1L\xa5s\x8f\x1a\xf7n\x9b\x9bL\x82O\x05`*e\xde\xd4\xc1M\x872\x9a\xf2\xa8\xbe\x8aV\xb0t\xf8\xc2J\x84F=\xa5\xde1f\n\xd0\x02\x06\t\x98N\xf5\x80\x1d\xa6\xd3\x04WEH\x9fjz\xb4\x9d\xe1+\xa7E\x1b\x19\xe1\x89\xaa\xc5\xd9\xaa\x9012\x8ff\x83\xf1\xb3!\x17F\xa7\xbd\xbe5\xab\x89\xd9\xe4\xfa\xf1\xde~\x05\x844\xf6#y\xd8\xd9\xacS\xaahN\xa4\x99A\xa6\x99\xc4\x82_\xc2\x01\x93\x13\x8d\x85\'2\x07u\xf2\x0e\xab\xc3p\x88\x99\t\x82\xac`\xdaE\xccdFP\xe0P\xabI\x14\xddGo1\x9b\xaf\xe6\x1c-\x82;B\xeb\x8f*\xf1\x8ea\xc7\xeeF\xdd/1\xec\xe6@\xee\xc5\xfb/\xc1\xe4\x01K\x0e\x0b8\xd8\xa4\x9b\x9e\xaa\xff\xf4\xa2\xfe\xf4I\xf3\xea\x93\xc6\xe5Y\xd9\x03\x95J\xd7\x7f\x9e\xc1]\xd1\x89\xb4hr\x85&.\x19\xcetV\x8b\x00H\xfa\xf7i\xfc\x8b7K\x9b\x19*{\xd5\x12m\x853\xf3\xf47\x1b\xbcj\x17\xa8\xd1\x0c\xa3/\xc5`\xa72q\xab\xa6\xe7h\tw\x19q\xba\xb0\xd8A\x97\xdc\xb0\xb9\xaelz*\xf9\x84\x87\xdd2\xc5kr^\x05\xc15\xce\x13q\xc1^J\xe4-\x12\xb5e<<\xb9\xd8\xbct\x16\xfb\xe0\xc3\x8a\x1f4\x86\x81Z\xaf\\O>h\x90\x0f\xad\x07K\xfa\xbd}s\xe1\xd6\xb6\x91\x1dm\xbaD\x0c\xf8a\xdb\x8fF\xdf6J\xfc<D\x05\x9ey<<\xcd\xa4\xabYPs\x1e\x9btf\x986|\xd1\xdaa \xe5\xb4\x19\x83P\xf1l\x15qt\xac\xb1i#l\x10\xfe\x9a\xbb:\x124\xbcZ\x90\xecC\xd0\x9b\xf9c\x1d\xdc\xf3C\xa4\x14\x9buj\x96\xa7X\xb1\xb9\xa6F\xaaQ}gf\\#\xbfN,Q\xc1\xc1\xa8[\x93D0\xd5\xe6\x13H\x1a\xc6(.\xd4\xc44-u\xa4\xcb*D\xc4(\xff\xcc(\x0c\x89#\\I\x00=\x12W\xd7n)\xb2g1\x0c\xdfV}5D:\x9c\xe3\x11\xd1\xba\x1c\xcc\x94!\n\xf3\xd7k\x0foi\x02\x8f\x943)#\x1f\xe1<\x7f\'\x98\xbd\xbc\xfa\xe2E\xed\xe4lm\xfe^\xfd\xfa\x9dh\x7fv\x0b}\xc8\xe6;\xab%4\xcd\xd2o(a\x0b3\xc5T\xbfCm\x08\xb1J\xa4L\xe6\r/o\x9b\xa9_\xeb\xe5B-\xeb{\xf9\x9b\x92\xf3\xebY(FZoZ\xb7mbs\xad\xb5\x84\xcc\xe3bt\x97\x8b\xf3\xa2\xc2\x98j\xd9)xGp\xf4K\x19\xfe\x89\xe2\x91\xb2o6\x8f\xed\xd0\xb4\x8aS\xe8}\xed\x19\xd6N\xaf`u\x13Yh\xdc\x07V?=\x1d\xaa\xf4\x1f^4\xaf\xdc\xc6\x14\xf9\x85\xe7($\xdbu\xf1W\x8b\x87\x8b\xdeHQTH\xd4\xe6\xcc\x80c\xa1p\x11\x83\xdfv\xf34mLl\xb1\x98IBJ\xde\xc2\x99l4\x8cLj\xe9\xa55\xf0\xe2/q\x9b\x02\x8d\xa0S3\xb0\x07S\x14J<\xc7\xc8\x7fa\xd3\xe3\xb9)\xdcs\xa2\xd07D\x80l\n\xd8\xfdw-\xa6Wz\xb4U5\x8ce,\xb3V\x9f\xe5\xad\xe1\x03\xc9\xfb2c\xbfS\xc3+A\xb1%L\x86G\xd5\x88\xa8\xa0\xa0\xa58\xf4\xe8\x15\x17}\xcc\xa0\xaa\x99a\x14\xc6\x02\x95\x86\xa3o\xc6/\xc3>u\xb6\x8fM\xcf\xc8\xe8\xe2\xe9)\x1e=\xd3\x9c\x9a\x8d\x8f^\xa1!|\xe1\xe0\xff\xffe\x8dRW\xfa\xd8\x7f\\\x1e\xe5nt\xfaLM-\x92\xc9G\xf0\t\xb7`\xf3\xe42\xf8;25R\xfeN\x98\x1a\t\xdf\xe8x\x1fH\xe8\xc1|\xd5\x1fV)*J\xe4\x83\xdd=T\x0c\xeb\xe8\xff\xec\x0c\xf4\x918\xc4\x9a\x17\xb4\xf3\\\x9c\xa3\xe2V\x8eq:\x11H$2\x9f\x16\'\\\xdd\x0e\xc7\xd1QvM\x97\xb7\x18/\x0b\x16\xbf\xafM\xffR\xbf\xff\xa8\x01f\xf3\xf4\x9d\xfa\xf7z\x84\x13/\xc3E\x11\xf1\xec \x83\xb8\xbflg\x0f\x1b\xef\x92\x00\x9e\x9bG\xa8|\x07\xf9\x91\xdc\x0b\x8a\xd8i\x92\x96azpj\xe3ga\xc5\xc6\xebW\x8d\xe7\xb7M\x88(P\\\x1b\xffA\x86\x87\xbdz\xd2\xbc\xfe\x9a\x8b\x0ee&R\xe4\xe6\x81\x11?>+\x8f2GB-\x80\xb9\xaaCs\x89<\x00\x0f\xbb\xd2T\x87)s\x1b\xd8\x95Ce\x80\xc6\x82\xb3\xf4\xb9\xb6\x10\x89i|\x02\xd8BF^P\xe8\x87\xbd3\x9dl\xa9\xbd\x9c\xad\x9d?\x1dL\x7f[\x9b\xbf\xde\x1c\xff1X\xbe\xbb\xba|\x96k^ydK\xf9\xd8\xd66\xada^\xbc\xbbJI\x18\r\n\x0fr\x05\xef\xc7\xb1\x0b\xd2\x107~\x0b\xb4T\x18\x98a\x0b/D\x99\t\xc2\xea\xd2\x8f\x80\xa083\x13\x96\xf4=%\x9dYJ\xd5\xf6\x87\xbe+\x90\xff\x88\xbf\xf5\xddw\xbb\xba?\xe8\xdc\x04\xff\xafk\xeb\xe8\xde=\xfb\xf6\x8f\xbd;\xe2\xff7&\x84\xde\xdc\xb6QM\x12cF.\xe8\xbd\x98,\xfe\x0c\'\x89\xa1\xfcvA\x91Ti|\x9c\x9d\xf3q\xa6\x92\xccK$\xb6\xc6\x10^\xde\xe1A\x9c\xd8\xd6\x84mO\xd4\x955T\xcd\xe3\xc9\xc8\xc6if\x1c\xfa\xce@\xc9\x0fO\x07+\x93\xc0\xde\xcdK_cxn\xf1* \x9f\ryv+\xf5\x0cQ\x1e\xc2Z?;^\x9b<\xdb\x8a\x88N\x1a8D\xb0\xa7\xbb\x01v\xe7(Jd+\xbd\xa7\xaf\xa7\\\xf6\xca\x1d\x06\x9a4\x86{\xe8\x19L\xc6\xb6\x1ek\xc5-V\xcd\x99W\xe1d\xa3<J)}\xac\xc0\xa1F#\x94\x98\xeb4\xe3\x08\xfe\x8b\xdb\xa2CdB\xe3\xf1\x91<\xefH\xf3\x80\xa4\xf4h~\x96\xb9\xadX\x97l\xf9\xb0P\x10k[\xb0\x9ep\xe10\xb8\xce\xdd\xf2\x15\x1b\xed\xb8"\xd8\x7fv\t\x8c\xaa\x8d\x88\x11+\xe7\xfa\xa5\xbc-\xb8\x07%\x01|\xc8D\xc5\xd6+vM_x\x8e\xe2\xe4\xd2\xe2\xea\xd28\x03\x1cL_f1[\xff\xe5\xeb\xe6\xc4y>\x9fD\xf2c&\x8dy?y\x84cd\xf8\x18*8\x94\xa08/\r\xefB\xe0\xc4{>\x8fs\xb7pJ\xa7\n\xeb\xdf\xbb\x87\xcd\xef\xc7\x819\xc7\x9b\xcb/\xe5\xba\x10\xd05\xf3\'\x80c\xeb\xb3+<\xfd\xb1~e%\x98>%\x93\x05\xcd\x15Y\xd6\xa7\x18\xe3\xfc\xacb\x9b\x19\x16<`\xae\x83\x13\xc5[j^=_[|\x8a\xdedtoI\xf2\xba\xcf\x11?N\x0b:auX\xf2\x08\x86\xf0\xb8\x0eK{\xe7j\x105,\xc6T\x0bBgu\xf96\xd7Z\x85\xd3\x94#\xc3\xa6\x81\x8eJ^\xd1w\xcclm\xc9\xf3\x8dt\xadLZ\xe9\xa0[\x8c\xb6\x8d\x86\x93W$v\x92\xd2,\x0c\x9f\x85\xec\x1crsW\xf7&=\xf2\x8d\x9a\x98\xdd"O>T\xdb\xd2-\xf8$Cxx*\xc8\xd4\xbc\xc3\r\xf9\x9a\xdc\x07l\x9cNo\x8cy\x93\x9e\x0b\xa2l\xc0$\xc6\xd3\xaf}\x17\xdcx"\x95\x12<\x12\xf5\xde\xed\xe0\xf1y\x90\xfd\xe8\x90\x12y\xd5\xbe\xff\xd6\x88+\xa31\xa2\x90\xd0\xe9\x1d\x8e3c\x1e\xefg\xb0q\xf259\n\xa1\x80\x14\x90\xcb\xb6\xeb\xab!uT\xefQ\xde\xe8\x83\x1a\xb5\x1cd|\xa1z\x82\x89\x05\x8a\x88\xfe\xcb\xf3\xcdK\xa7\xd9\xa6\n{r\xf5\xbb\x13|\xe5\x11\x9f-\xb6\n\x91|\xdb\xda,@\x99\xccb\x17\x8d\x1e\x8ac\xa3)i\xeal\xed\xc2JD\xd6\xaf\x15\'\x04\xf1\x96E,\xa43m\xbaa\x85\xb0R>\x0f\xb4\xe4\\I\x0e\xc5A\xde7\x1b`C\n\xa4\x8egm\xefJN\x9b\x05\xeeOSrq\x87\xb9\x14"\xd9\xb1\xf6\xe1\xf4\xf1\x82C"5\xad\x8f\x0c\xfb\x92\x0en}o\xd3\xa6M\x874\x95\x15\x1cj>7\xba\xf8{w\xa2\xb6hQ\xcd2\x96X\xdb;r\xa1\x05\x18\x8a\x9d\x1b\xa4k\x97\x18\t~\x1fVE<\x9e\x92H1\xdbAD\x8f!q={fd.yu6x4\xc4\\\xe5\xa0\xae\x16B\xbeQ\xb7/a\x95\x06\xb2\x8d"(\x89\x15\x92\xc5\x91\xb3\xfe\xe2\rtn\x10\xa3\x94\x86\x82J\xf3\xf8\xf2\xed\x18$l\xb2*\x18\xa8\xf6\x03\xd6G\xdcu\x16<\xa0!\xaf\xe8fU\x99\x85TX\xd9\x16Rq\xde\x98\x1dir\x1dw\xbc\xc1\x93y\x16\xa8\xa4\x0c\xf40\x7f.\xb3"\xf9\x1d\x8e\x8e\xd4\xac\x86zU\x8d\xfb\x07\xf7\x19\xd3\xbd\xc6L]\xd6\x02\x1a]\xb8\xd66\xba\xdc,]\xf1*v~\x1b\x19\xc2T\xbb\x94\xdd\x96\xaa\x9d\xff\xaay\xf5\xcc\xaf/g\xb9 \x87\x95\x17\x08&0g+\xdbh|Q\n\x9b\xf1\xed#\x8e)E5\x9d\xda\xd6 \x15J\xc3AS\x07e\xd9\xc9zen;7\xa61\x1bV\x04Y\x10\xe4\r0-\xd4\xe6\xc7\xebO\xc5\x15\xa8\xff<\xde<w\x12\xdd\x1f0,/\xbd\x94*\x1a\xc3V\xd0"\x86G\x1a\x86\xe6$O2(;%D|\x0e\rF/Os\x8a\x15\\\xa4\xe8\xf1\xf8C\x0e-;\x9cv\x1b$\x19-\xb3\x0e1X\x19>\xc2:\xa7y\xe3E\xfd\xf9\x04j\x15\xa2Ul>\x84S\xb8~\xa7\xf1\xb7\xefM\xb8\x15p\x94D\x95u\xfbi\xbb\x00*\x18k\xea\x1e\x11\xeck\xb7*\xe5je\x18QC\x19\x16\x9f\xf4(\xcf\x88\xa7\xaa\\\xc5J\xaa\xa6\x8dU\xaeP\x0f\xd7\xfe\xb2\x83\x07<(C\x0f\xa6\x1f3%\xc8\xbdk\xa4\x12\xd1\x12\xf8\xf1+\xec\x1eg\xc2\xb8r|\xf5\x05%7V\x16\xc0\x8f\x08-/\xa2\x93a\x10s\x9c\xf0\xc3\x81R\xa9X\x19\x0cFGqZS\x8a\xe9$6\x83\x0e\xbf\xc5\xa9f)!\\.M\xa04~d:!\x0f\xabJ1\xb5\xae.=\xc4\xdf\xd1\x10\xa8\x94d.\xe9\n3\xfa\x14\x078\xa58\x8d\xa9>\x8d\xbch\x0b\xfdjnZWB\xd0\xaf\xde\xa3OO\xfe\x88\xd9\x05\xfeT\xe6\x16\xfd\xd9v+r\xd5B\x18W\x90\xbck{\x1a\xd6\x97\xedtn\xa8\x9d\x9b\xa8/\x9fc\xeb64\xf1A\r\x1d\xffI_\x8dBE\xd8H\xcf2f;B\x17\xad\xce\x06M4\x84SE6Abe\xf0\x84o:\xb8\x1cX&\x908l\xf3"Hl\x01H0\x84\x80\xe2\xfb\x88\xea\xd7~\x06"\xe0\xbb\xa5p\xca6\r\x1fb\r\x80\x03}\xc6WD~E\x0c\xbea,\xd1E\xc6n\xa3\xfe>\x07\xe5\x87\xbe\x00\xc9\xa9\x11;\x9f\xdf\x08z\x0b\'\x1f\xb8\x05\xcei\xd3%Y\x83\x16\x16\x0c\x16\xd8\xce\xe0:\xba\x90Xu6\x8a\x87.\x13\x91\x9a\x86\x9a\xf8\x8f\x8f\xa7\xf8O\x00\xb7~\xf1\x17\xb9-%n\x1f\xc6\xc1\x13\xb38A\x12cr\x18\x85t;\xad\x9eu\\\x12_!\xb3\x0b\xd3\xa1e@\xc7\xde6Oo\xd6\x95r\x9d\x82$f\xe8*%\x89\xa7\xea\xf7b\xe6\x98\x8b\\\x0c\xf7!tC\x93\xa0S\x90\x98\xb6x\xeb\x0e\x00\xf1\x86\xce\x90\xa3L4S0\xce\x86\xc92k\x18N\xb12|,\x9e\x005\x90 C\xe6\xc9x\xe1\xd1\x9d\x0c\xfc\xda\x00\xafeL%e\x0ey\x94\'\x13\xb2\xe5(\xa7Mj\x86\x80\tK\xa5\xf8E>\xe6\xafCh\x196&m\xf4.\xa6N\xf3\xd0{\x16\xcbo\xca_\x86\x0c\xb9\xd6SQr\x89y\xff\x1e\x95\x81b\xc1k\xd9.\xfan+\xac\xc4\xa4\xe7\xef\xd4.\xfc\xc22OT\xf8\xf4%SLs)h\x1c\xdcv\xc0\x84\xd41\x82a\xd9\xec\x91t&\xba\x0b\xf1\x94\x13\xfc\xe3\xfd\x8cr\xed&\xafs\xfb\xdb\xc1Zr\x9d\x8a\\w\xaafz\x91\xc2\xe0C/\x17$\r\x88\xe7o\\\xb8\xd0\x82\t\xbet\x86-\x8d\xf9\xeb\xc0\xed\xac\x9e\x98\xba\x82\'\x0f\x98\xfcj\'&k?M\xb0\\\xe0$<\xbb#\xed0\xb4\xaeM\xe8\x10\x80\xb2@\xd0\xf6\xd32_\xdd\x15AE\xc2\x11\x11\x1bI\xa8\x1b\xf4\x86\x17\xf7=\x7f\t\xd2W.\xb6b\x05\xa0%\xfez@*\xd9\xbe\x9fxto\x17\xe4X\x07\xf5\xe24s\xf4-\xb3\x0e\xd8\x1cy\xc3\x1cb2\xc6\xba\xe6\xa2\x83N<Xj`\x04\xe4\\_\xa4\x87\xd9\x04\x10\x0e`\xe0\x08\xc8\xd5\xe6\xc4\xd7\xcco\x88\x8ek\xdf5\xc7\xc7\xd1\n!\x144^\xde\x03\xf3\xc8\x0c\r&\xf3\xe4[\x1d\xa1\xe9\x8b\xa1\xefHq\x925\x9c2\r0\xb9\x86DC\xd3\xd1\xb0j\xab\x97\xb6\xd6\xfaq\xaf\xedM\xcf\xeft\x19\xc4\xc1j\x19@.K\x84M\x17F\x9aX\xd6m]\xf1wpxG_\xe7\xa1}\xa2\xb7\x85|MeBU4\xff\xd2\xb7g7O\xa4\x90\x989\x8e\r\xc1\xb0\x8f\x96y\xe0\x86\xda\xff\xcc\xb7\x02;#\xaao\n\xc3\x82\x03`?\x95[\xd5\n\xbd\x91\xedH\xe1p,\x0bB\xe8\xb9i\x04\x8b\x1do\xcc5\xee\xbe\x92\xb2\xd6\xe3\xcb\xab\xcb\xe7\xe0\x81\xd5W3\xfc\xc0\xfa\xcalD v`,6\xf3\x06\xcd\xb3\xd7.c\xecA+\xf7\x90\x14\x88\x13b\xf53t\tF\xc2\xb3\xad\xda0i-\xe6\xadm\x14\xe3\xe9D\xa9\xe9+P3k\x03\xa9Km\xf9\r%\xfb\x18eV\xf4\xad\x1f\xbe_%wVW`\x1cv\x8e\xf9\xf1i5\xa1\xbcbh\x15\xcaq\x88\xa68\xdd\x13S\x8d\x85\x87\\\x8a\xc1\x99V\x1c(\xb1x\xb5\xb5S,\tH$\x06\xd8\x1c\xa3\x8djG\xf0\x13.\xfb\x1e\x1d{\xd3\x06\xa9HT\x13\x96\\\xe4\xe3\x11\x97\x08Cc\xc1F\x1b\x03 x\xfd\xaa~\xf1N\xf3\xdb\xc8m\x97\x9a3\xc2\xe8?;=\xcf\x9e\xad.\x1f\x7f\xd3n\x80\xa2\x10|\xdeH\x18D\xa0:-\xfd\xe7:\xe5-\xe7[\xc0\x1e\xcb\x0eS\xf3:\xda\xc7\x91\x88#\x05E\xf0>\xed\x96\x8a-:4\xca\xb4\xb4\x16L\x84\xbf\x8fN\x93{\x138\xb05\xe3\x90(\xf3G\xdbJi\x18R\xa4\xbb\xc3\xfdS\xa5R\x92vi\xb7\x04U\xab\xa9\x1b\x1a$\xa4\x81&\n\x1c\xde0_V\x85\xa5@\x08\x04\x07"\xc3\xe0qg\xe2\xbb\xcc\xfb\x1a\xa4\xecN\xf9\xb3x\xa3\xc6\x95kxK&g\x0f\x92<\x84\xf5\x80\x8ba\x11L1\xf4c\x18\xa4\xec\x12\'\xa5\xd9\x07&$`U\x1bb\xe4 c\x83\xfa\x18\xf4i`wv\xd9\xa1+L\xb6\xf1\xdd\xb3\xeb\xa3\x0c\x1c\x08\xa4\xeb\x87\xc8\xa7W\xc1\x7f\xad\x90U\xeb\x86rN\xe1\xb3B\x92I\x89\xc6$\xf62P\x1cT\xbb\xf7t\xebHx\t\xa4\xe9r\xfc&r\xa1\xa2)\xc5\xae)\x1e\x15N\xff\xee\xc7\xc9loO,;0\x0b\xedf\x95\x03C\xd7\x8e3Bd6\x10\xb7Yr\x19\x0b\x85\x1cY\x06&\x93\x0b\xb8\xe0XDfxV\xe0\x0bc\xbd\xc8\xfc\x02K=\xd4\x9d\x14\x1f\r\xaf\xe4~+ZQrj\x1b1\xc9\xdbm\xb5\xe7h\x05k\x80\xb5\xb3\xc6\x07\x8b,\xb0\x91\n\xaa\xf4\xcbQ\xbcc\xb2\x87\x8b\x82-\xee\xe8J\xde0\x17=HF\x84\x9bi\xf8G\xa0i\x83\xc9%\x99~9\xa7j\xac\xfe\xaeM\xb7\x15\x1b\xb1\xa3os\xf2k\xbdZ\xeb8)%\x02y\xa8d[\xca\x0f\xef\x18\x8b\x8e\xfdj9t}\xb3\xf6\xc3\xdb\x86h\xbcz\x92yCU\xeb\xb6#\xfd\xf5\xc0\x18;\xf7P8\xfaf\xd3\x08\r\x0b\xd7\xf5\x92\xeb}\x7f\xefP\x11i>\xf4\'\x0b.\x15\t\x0e\x1c\xd3C\x93A\x03\xe6\x1c\xb9jY\n\xc5\x13\x11\xa1\xb5\xa1\xb1qS\x1b\xe2\xc5\x11\x93\xe1}v\x7f\xb7\xc8\x04\x8a@\xa9\xa8#;\xa1\xae\xe8\xb0Z?$\xf5\x99a\x92\xc1\xa8\xdfo \x13#l/\x91d\xb9\t\x95\xe2\xd9\xb1K\xe9\xd6\xa2\x18\x9a\xcak\xc6\xeen`\x05-\xde\x1dK\xca\x043\x88K\xaf\xccP\xfa\xdfK9R\xf9]\xb0\x8f\xba\x85*\x86\xaa\xa3\xf8\xa1\xafQ\x89D?\xa6\xf1\xcb\x7f\x1f\x9et\x07\x02\r6\x01\xc9Jr\x14k\xac$\xa7\x16N\x04\x14\xd7#\xcc \'\xe2JM"\x95\x80\x05\x99\x87X\xadB\x93H\xf1\xe2\xbe\xf1\x97\x98\x8585\xc3\xd9\x86\xfa\xc5\xfb\xcdo\xbf\xff{\xd1\x95X6\x9fv\x8b\x95\x0e\x1e\x81/c\xceM\x9b\x9f1\x9b\xf4\x94B\xfa\x1f\xadMk\x10\xdd\x9b@\xa2r//\x9f\x17\xa3F]y\xa0\x8a\x1c\x85\xf4d`\xa4;8\x88\xa8\xd6\x9e\x9d\xbe#l\xad\xb7K\x8bST\x93\xa3\xbb<w\x8e\x89\x0f\xf1{\xf2\x1e\x9c\x84\xa8t6\x87\x80@\xd7\xa6\xcb\xf5\xec\x8c\xac\x1f\xde\x02\xdf\xb4.H\xcb\xfc6\x84\x1d(\xe9\x9e[\xcc:\xa9K\x9du\x8e\x8b\x88\xb2\xc3*\xe2DbPt\x18g\x919\x87\x12?v\xd6F\xd9\xea\xca\xb7\xb5\xaf\xafJ\xc6\xcb\xc8u\x85\xa2niF\x8d\x04\xc6t]\xed\xfc\xc2\xea\xcb\xc7\xff\x10d\x15\xe5\xda\xbfM\xc0\xd5n\x11\xa9N:9:\x0cdf~#\xfez\xa9Vl\x90\xcbk9\xbefg\xb3U\xae\xcfZ\x1b5<\t\x08\'=\x82\x84\x07>\xe5h#Y\xcd\xff\x90\xfd\xc7M\xe5H\x0e\xa7\x9d-\x9c<1\x19\xef\x0b\xa0;\xe6+F\xcae\xc8.\xe7\xf2\xa8\x08\xc1YJ.U3\x869\x80\x0fp\xff\x92\x19\xb5\xe6\xcbC)x\x88\xee\xdf\x89Wf\xd9\x91\x96\xf8\xedbt\xb1\xd0\x82J*l\xb4\x12Ry\x7f\xdcfm\xd9\xfa&\xef\x96\xf2GtU\x95\xe7c\xa21k\xee\xf5\xafU\x1c\xc7\xad\x0b\xddrUL\xf1\xf0\x98L\xa9\x15\x89\xbc\x8d)|u\xe9\x0cx\xba\x9ct\xd4i\x13\xc9GRz\x92\xaf\x84\xe3\x8a8\xacc\x7f\xf2F\x9771O){\x7fCc\x12\xbaz\xb2\xab\xbc7\xe4s\xca\xc4\xd1\xb7\xdc\xe6p\x9e\x11\xcf\x17\x96\x8b\xdf\xb1C\xdbo\t\t\x8b\x04\xa4\xda\x99\xe0\xda\r}\xa5{p\xeb\xeb\xe0\xcc5}\x7f\xad*\x0f\x1do\xbc\xfe\x12\xed\x1aJiY\x9fyC\xebp\xea\xb9\x12\x85n\xc7]\xa7\xf7\xaeS\xdb\xb2A.@\xe1}bJ\xd0\xa6 #\xc5\xf5r\xae\rV\x1cH\xael\x82\xd5\xa1\xd3\xdb\x005o\x91\t\x0f-\xf3\x95s\xf5\xe5s\xcd+\xb7a+\xebu\xcb\x12\xca5R\xd1\xea#\x85\xaf\xab\xffVLY\xefX\x9f\xed\xf9\xa4\x13w\xde\x8f!\xbd4\xe5\x0f\xfdm)\xf0>p\xc2j*sp\xe3\x96\xf76m\xdazh}\xa1(j\xb2\xa7\x12%?1Y\x9a\x14v\x8a\xdc\xb3\xac\xca(\x92\xd3\xa3k\x1e"\xe7\xfa\x93\xc7^\x08\x14\xef\x8e\xea\x18\x8dY\xf2\xf8\xbe\x1e\x7fA\xbf\xd2\xcfd\xd6\xd1w\xa7\n\x08<\xac\x15D\xe2\xa6l\x81\x1df\x8b\x936.\x02\xe6\x02\xd5\x05pa\x03\xed\x11\x0e\x9ck\xb99,\xbe\x0e\xba\xc5]\xaf\x93`{p\xf0f\xb6"\x95\x04\xea\xda8\t\xeexEN\x1d\x87\xe5`y\x9bkat}o\x8b]2\xbf\x10,\xde\n\xbe<\xc3\xa9dLl\xf2X\x05\xedO\xaa\xab\x9a%\x87\x95\xd8,\xd8\x0eZ\x81f\x9bu\xb0\xc0\xa5\x0e\xe8\xe1\xe3n\xd9\x04F\x9f\xb2\x1aq\xafT-\x04|v\xf0\x105/\x14\x0en:d\x06\xb0\xbcb?\xbd5uh\xbdi\x04\xf2\xc0\xb9\x92\t\xd0\xe5K\x1b\xa5\xaa{\xa5X\x0c\xce.4GK\xb4\xb1\xde\xae\xcf\xe1\x8d\xaf\xdc\x91\xa7\x8aA\xd9\x92\xa3z\xa6s\xb5\x8b?\'\x15a\xad\'\x08C\x88z\xdb \\\xb5<h\x0b\xc1\xd2\x0bB\x9b\x94\xa7\x13\xc0\x16\xb1\x13\xad\xadg\xade-\xcf\x19\xc0\xdao\xdaV\xe8U\xb6=\xed\xb5\xe0J\x90^\xe8J\xf2\x1e\x0fn\xec\xc2{\xe23a\xc5\xd9\xfa\xf2y\xec\tr\x10\x81\xf3wJ *\xe1\x94\xe5.6\x7f\rb\xe7L\x85n\x1f!vUcD\xd0\'\xac\x01\xf9?\x9d\xab}{\x9d+?\xd6{\x80\x03 z\x0f\xb7\x99\n\xceF2\xdd<\\!=\xa2\x83\x00\xd8\xf5Fea\xca\x04\xa12\x0bN\xac\xa2\x06B?\x87\xe2\x01F\xaa\x05\xb8u\x8eFB|\xff\xad\xae\xf4\x15\x9f,Zd!Aq\xaa\x18\xa9\xfd|\xb9~\xba\xed\x85\x0c\xda\xe0\xc3\xa2\xd5A\xf7h?\x9e\x15\xc8\\\x05\xd5\xbb\xd6\xfb\x9b\xb6vv\r\x8eY\xc1\xf4\xa9\xe6\xdc7\xeb\xb4\x00\xf7\xf13r\xd3\x87t\xc0s\xc7\xe0F\xd2J\xd1\x16\xb1\xe9gx_\xfd\xf9\xebzW\xd2\x07\x06\x1f~\x0f\xcc\xf6Csj\x16G\x1cL<\x02\x19\xb4\xe6N\x04\xb6t\x1b\xb0\xb8v\xac\x82\x17\x1aj\xa9/\x06\x8d\xd1\xcc\x11\x16S\xb8\x91:\xed\x10f#=\xa4\x8a\x94\xb1\xc9ea\x065\x1f\x11Oh\xb2M\x9c\x0f\x1e\x9djU\x06\x895\x15\xeb\xa8\xa7\xf8W\x1a[\xb1c\xef\x01&\x18.\x051\xe6\xf2\x95\xabE}k\x11\xdb3q\x85\xfd\xf5\xea\xcbe4#\xae,p\xe1:g\xe7\x11\xddD/\xab\xaf.`\x90QW"\xc2Ro\xac\xae\xc0\x82\x1c?\xef8\xa5tW&\xa1\xf8\xdd\x0cvK8\x025+\x8f\xfb1x\x9a\xe6\xadT\xf3yR\xc0\xc4\xc7\xe1h:\xe6^\x90\xb1(\xa7\x8c\x1af\xd1\\+\xe7\xb0\x97\xff\xf1\x94\xae\x9b\xd4\xeeQXN\x18\xa9:W\x8e^\xe8\xd1\xc5z&p\xd26CNz\nk\xf6uDJw\xf6J\x98\x8d\xaa\xeb\xf8*\xeb5\x96k\xf1\xab\xf8\xd96,\x15v\x95\x87\xac\xc4\xe6H\xa4A@\x82?R#\xc6\x0e\x84QF\x89d\x89W\xfb\xbc\n\x8bZ\x13\xe1\x8a\xb1\x7f\xba]\x8dUf\xbd\x12A\xad\xa0sE\x0c\xbb\x96\x7f\x14Q\x96\x96\x12\xa5\x88\xd0\\\\\xa3Q$\xce\xe1\xf4\xc6|\xbc\xd2n{~\xc4>F\xa5\x9e\xac\x1b\xa3-\x1f#nn\xc8\x91\x0b\t\x0cFW\xf4\x07<He\xc2\xd4\xc6w\xf2&p\x07#\x93\xa9\x0cky\x94\xe6\x07\xc7\x93\x0b\xc1\xb5\xcc\xc2\xdb\t\xda\x95\x1b\xb44\x85\x90\xe1I\x19\xa1\x96[\xe7c\xb0\xea\xd6\x0f\xb0)\xa5e\xd9\xbc`^J\x04\x08\x85\x89\x08K,>\xe7\xba\xfc\xa8\xdc#[\xc8\x95\x89%j\x18\xc1\xd9;\xa6\xb0{\x11\x8e\xf0\x94B\x8b\xc5\x13\xab/\xee0"0\xff\xb2<\xf7\x8f(\xbe\xe7\xbe\x0ci\xba\xaa\x0eh\x8e\x89\xd7\xe0sg\x86\x99\x0cZ\x7f)>~\xfe\x99\x97U1,\x91\xa0 \x8fx\xe2=H\xd0\xd8\x15\xeej\x1a\xa8K\x1di+\xdf\xc5\xeec\xd7\xb31\xb9i5\xe2\x11\xc9\x85\x102V\x02\xafX\xa2\xc1\xda<\x7f\x85\x12nz\x14(/\x96\xcat\x96\x87\xf2\xde@:\xf5{\x9e\xe2T\x8aM\x9d.\xd1\xac\xc6NbI\x1f\xcd\x88\xb4\xba\xd03#\xc5\xf0TwG\xb2\x15\xc5-\xd9\x1e\xbc3\xbe\xee\xd3\xa7\xbb\xbc\xd5\xe50\x8fj_=\x04|\xea\x19\xfc<tz-\x88\xf93\x82l\xcd\xd9\xf84\x13_\r\xc9\xcf\xe8\t\x8d\x838\x85\xb1\xe2\xb1\x8b\xa2L6\x99G\xea\t\xa0x\x13\x9do\x1fAi\xcb\xa5c\xad\xb3\xd00:t\xefv0};"x\x8d9o\x04&\x8d\xd76T\x06?\xb9\xfa\xeaTm\xfe\x1e\x9e\xd6\xf4cuy\xc6\xd5Q\xed\x9e\xfc\x8eO\xc2\xff\x1d\xf9\'\xea\x02\xfa\x03B\xa1F\xb0\x070L\x01w\x0c\x84\xd0}\x89\xc8\xd0\x82j\x17\x8d\x06\xac\xba\x98\xbdT\x7fp\x17+\x98\x89,\x98b\xb8\xc7\x97\x1de\xb4t"T\xc23_\xb8\xf1[\xc2\x91\xb2\x97\xc3\xce\xb1my\xbb0\x90\xb3i\x0e\xc0V\xfa\xafqkV\x01E\xb7\xea\x80\xb2\x8f\x84\x03 7bk5\xcdrc\x82\x86\x83\xfa\x93=\x84M\x0b\x98R\xe3\xa4\x8aW\xa6D=V\xe8\x82\xee\xf8n\x9e\xcb\xab\x1bS?\xc5\x1f\r\xe7\xdc\xe2:{\x0e\xec\xdf{`\x7f\xff\xce\xde}x\x05\xdc\x9e=\xfb\x91\x94\x87\xb1l\x1c3uxW\xb6`\xb3u\xee_\x8c\xbb\x8ei\xf1\xc3\x17\xac\xc6\x98+:\x0c)\\v]cf#\xdd\x0cF\xa2\x89\x91,%Pt),^\xe3M\xf2\x86\x19\xc4$\xbc\xd8M\x80\xc1\xe9G|\'\x96\x82\x89\xf7\xd3\x8f\xa1n\xc0\x85\x81\x18\x9a\xeeK#\x97\xe5C\xbc\x0f\x1dO\xcc)\x8c\xf5Ko\xf2\xa8y\xec\xc2X\x8a\xf4>\x11\tetS\xc9mv\xba2\x1bK\xffF\xcan\x85\xbb\xbe\\\xe0UA;`S\x02\x15\xf74\xe4\x0c\xb6\xb6S\x1bwO5\xbe>\x8b-`\xa7g\xa2\xa4(\'\xb4\xcd2\xb6\xc6 \xed\xe6K*\xc3\xb0\xa6\x1a\x9e\xefc\xff\x92S\xc6\xbb\xd21d\x86Z\x9aF\xf3\x04\xcb\xb3x%\r\xc5/y\xfd\xe8\xf5=\tc\x06\xba;\xac.p\x15\xe3]\x9e\xa4?)`\x03\xe8\x08;)\xc3\xc1\xff\x18\xc1\xab\x16s\xe1 \x01\x9c\x10\xfd(\xb6{\x1d\xacI\x1c\xafR\x91\x1dw2I\xb6\x1dL\x15\xa1)5\xf3Gz\xc4\x05y\xc4\xa3\xb1\t\xc5R\xf6C\xde\x83\xb4\xe4\x83\xde\xd0\xc4\x94\xb4\x92\xe1j\x86\xe6N>\xd4bE,\xe9\xd3\xb3:\xa27\xbb\xf1\xc0\xd1\xd5\xa5\xb3\xec\xeb\xcb\xcd\xaa\x91\xe5\xd4;\x93N\xbc\x13\xb5\x0ci\x18\xb4\x0f\xcd/\x84\x82\xe9\xdc\xb6n\xda\x9c\x1b\x8b~+D\xac\xfa\xa5\xd4\xdc\xed0\xbb\xcb\xec\'\t$\x99T\x1b\x97\n$\x92\xa4oDM\xde\xd6\x17Bj\xe4E\x85R\xc2\x14gf\xac\x0ey}\xb4\xab\x92\xd3W~5_\xe9\x88\xf8\xe4\x1d\x9c\x8f\xd4\xcd\xc4Y;\x8b9/g\xd8>\xe2b\x8b\xa2X\xafr%\xc5\x04\xdb\xd9`\x0br\x01t\xf0\xfa>\n\xffh\xfb\x8dL7`=D7:\x10<c\xbf\xbe\x9a\xe7\x9f[oe\x80\xc7\x85=\x8e\xd4\xf0+fS\x16\xefKO\x1eU\xa68\x15w\xbe\x8b\xa3\x91\xa9\xe80\xeb\xe4\xf3,\xfbIK\xb1\xcd+mO\xda2\xe0\xde\xc3\xa7w\xcd\xf9\x94\\\xd9\x89\xcd\xdeKX\xf1\xad\xf6I\xa3\xfaY\xd8iZ\x8aOK\x94\xf1\xb6bt\xf9\x1c\xc0\x0f/\x1a\x02J\xce\xe5\xf4\xe5\x9a\xa6\xa5F\rf\x04\xb0\x84pX:\xe3\xb4i$\x05\xea,C\xc8\xad\x1d\xf0\x90\x05;j,\xdf\xd7M\xad\x0cp\xf4<R\xff{\xfek\xab\xfe\xe0\x81\xd5\xd5\x85Uw\xfap\xf4\x8b\xf9}<;H\xa2\xda\xf8t7\xd5\xe8\xb5`\x00N\xe5\xff\x00\x8b\x9fP\xe7').decode('utf-8')
REMOTE_ALLOWED_TASKS = ['t2v', 'i2v', 'l2v', 'flf2v', 'r2v']
REMOTE_MAX_IMAGES = 9

'''
Prepare remote-only defaults for tasks not selected manually in the Notebook.
遠端 API 為尚未手動執行的 Task 建立預設參數，不影響 Notebook 既有設定。
'''
TASK_CONFIGS = globals().setdefault('TASK_CONFIGS', {})
for task_name in REMOTE_ALLOWED_TASKS:
    if task_name not in TASK_CONFIGS:
        defaults = dict(prompt='A cinematic landscape with ambient sound.',
                        width=384, height=224, seconds=10, fps=FPS, steps=6 if REMOTE_SERVICE_ID.startswith('06_HyperFlow_') else 20,
                        seed=42, quant=QUANT, output_name=f'{task_name}.mp4',
                        first_image='', last_image='', ref_images=[], ref_videos=[], ref_audios=[],
                        video_audio=False, ref_image_size='match')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_'):
            defaults['audio_shift'] = 6.0
        else:
            defaults['encoder_variant'] = ENCODER_VARIANT
        TASK_CONFIGS[task_name] = defaults

REMOTE_ROOT.mkdir(parents=True, exist_ok=True)
REMOTE_JOBS = globals().setdefault('REMOTE_JOBS', {})
REMOTE_JOB_LOCK = globals().setdefault('REMOTE_JOB_LOCK', threading.Lock())
REMOTE_ENGINE_LOCK = globals().setdefault('GENERATION_LOCK', threading.Lock())
REMOTE_QUEUE = globals().setdefault('REMOTE_QUEUE', queue.Queue(maxsize=REMOTE_QUEUE_CAPACITY))
assert REMOTE_QUEUE.maxsize == REMOTE_QUEUE_CAPACITY, 'FIFO 容量已變更，請先重新啟動 Kernel 及遠端服務。'
REMOTE_UPLOAD_SLOTS = globals().setdefault('REMOTE_UPLOAD_SLOTS', threading.BoundedSemaphore(REMOTE_UPLOAD_CONCURRENCY))
REMOTE_RESULT_TTL = REMOTE_RESULT_TTL_HOURS * 3600
REMOTE_RESULT_BYTES = REMOTE_RESULT_LIMIT_GIB * 1024**3
REMOTE_MIN_FREE_BYTES = REMOTE_MIN_FREE_GIB * 1024**3

'''
Do not start a second GPU job when a long-running backend appears stalled.
當底層生成疑似卡住時，停止接收新工作，絕不啟動第二筆 GPU 推論。
'''
REMOTE_STALL_SECONDS = REMOTE_STALL_HOURS * 3600  # 無限大代表不因等待時間長而拒絕工作

'''
Verify the CUDA backend and existing model without loading another model.
確認 CUDA 與已啟動的模型後端，不另外載入第二份模型。
'''
def remote_gpu_status():
    try:
        import torch
        if not torch.cuda.is_available():
            return False, 'CUDA GPU 不可用；請檢查 NVIDIA 驅動與 PyTorch，並重新執行模型初始化 Cell。', []
        qwen = REMOTE_KIND == 'image_multi'
        yue = REMOTE_SERVICE_ID.startswith('06_YuE2_')
        if yue:
            selected_gpus = [globals().get('GPU_ID', 0)]
        elif qwen and globals().get('QWEN21_STATE', {}).get('mode') != 'bf16_dual':
            selected_gpus = [globals().get('GPU_ID', 0)]
        else:
            selected_gpus = sorted({
                globals().get('MODEL_GPU_ID', 0),
                globals().get('TEXT_GPU_ID', 1) if REMOTE_SERVICE_ID.startswith(('06_MiniMax_', '06_HyperFlow_'))
                else globals().get('AUX_GPU_ID', 1) if REMOTE_SERVICE_ID.startswith(('06_HeartMuLa_', '06_LTX-2.5_', '06_Z-Image-Turbo_'))
                else globals().get('TEXT_GPU_ID', 1) if qwen else 0
            }) if not qwen else sorted({globals().get('GPU_ID', 0), globals().get('TEXT_GPU_ID', 1)})
        available = torch.cuda.device_count()
        if any(not isinstance(index, int) or index < 0 or index >= available for index in selected_gpus):
            return False, f'需要 cuda:{selected_gpus}，但實際只有 {available} 張 CUDA GPU。', []
        devices = [f'cuda:{index} {torch.cuda.get_device_name(index)}' for index in selected_gpus]
        if qwen:
            state = globals().get('QWEN21_STATE', {})
            if state.get('pipeline') is None or state.get('manager') is None:
                return False, 'Qwen Pipeline／GPU 記憶體管理器尚未初始化，請先執行模型初始化 Cell。', devices
            if state.get('device') != f"cuda:{globals().get('GPU_ID', 0)}":
                return False, 'Qwen Transformer 不在設定的 GPU 上。', devices
        elif REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
            pipeline = globals().get('pipe')
            if pipeline is None or pipeline.lazy_load or pipeline._mula is None or pipeline._codec is None:
                return False, 'HeartMuLa 雙 GPU Pipeline 尚未完整載入，請先執行模型載入 Cell。', devices
            for name, module, index in (('HeartMuLa', pipeline.mula, globals().get('MODEL_GPU_ID', 0)), ('HeartCodec', pipeline.codec, globals().get('AUX_GPU_ID', 1))):
                parameter = next(module.parameters(), None)
                if parameter is None or parameter.device != torch.device(f'cuda:{index}'):
                    return False, f'{name} 未位於指定的 cuda:{index}，請重新載入模型。', devices
        elif yue:
            gpu_id = globals().get('GPU_ID', 0)
            process = globals().get('SERVER_PROCS', {}).get(gpu_id)
            if process is None or process.poll() is not None:
                return False, f'YuE2 CUDA{gpu_id} 常駐服務尚未啟動。', devices
            if f'CUDA{gpu_id}' not in [str(value) for value in process.args]:
                return False, 'YuE2 常駐服務沒有使用預期的 CUDA 啟動參數。', devices
        else:
            base = globals().get('BASE')
            if not base and REMOTE_IS_LTX:
                base = f"http://127.0.0.1:{globals().get('COMFY_PORT', 8188)}"
            if not isinstance(base, str) or not base:
                return False, 'ComfyUI 尚未啟動，請先執行模型服務 Cell。', devices
            with urllib.request.urlopen(base.rstrip('/') + '/system_stats', timeout=4) as response:
                stats = json.load(response)
            comfy_devices = stats.get('devices', [])
            if not any(str(item.get('type', '')).lower().startswith('cuda') or
                       str(item.get('name', '')).lower().startswith('cuda:')
                       for item in comfy_devices if isinstance(item, dict)):
                return False, 'ComfyUI 沒有回報 CUDA 裝置，已拒絕 CPU-only 推論。', devices
        return True, 'CUDA 後端已就緒（不代表所有權重常駐 GPU 記憶體）。', devices
    except Exception as error:
        return False, f'GPU 檢查失敗：{type(error).__name__}: {error}', []

'''
Verify every global required by the selected Notebook generation engine.
驗證所選模型的生成程式所需變數與設定已完成初始化。
'''
def remote_inference_ready(task):
    namespace = globals()
    common = {
        '06_HeartMuLa_': ('ROOT', 'pipe'),
        '06_YuE2_': ('ROOT', 'GPU_ID', 'PORTS', 'SERVER_PROCS'),
        '06_Z-Image-Turbo_': ('ROOT', 'BASE', 'LOG_PATH'),
        '06_Qwen_Image_': ('MODEL_MODE', 'GPU0_MODEL_GIB', 'GPU_ID', 'QWEN21_STATE'),
        '06_LTX-2.5_': ('ROOT', 'ENGINE_PATH', 'COMFY_PORT', 'COMFY_LOG', 'COMFY_PROC',
                          'ltx_available_ram_gib', 'LTX_MIN_AVAILABLE_RAM_GIB', 'importlib'),
        '06_MiniMax_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                              'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                              'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                              'QUANT_SIZES_GIB', 'ready', 'choices', 'check_choice'),
        '06_HyperFlow_H3_': ('ROOT', 'BASE', 'COMFY', 'LOG', 'PORT', 'TASK_CONFIGS',
                                'H3_DESIRED_CONFIG', 'H3_ACTIVE_CONFIG', 'H3_MODEL_CHOICE',
                                'H3_ACTIVE_MODEL', 'DIT_NAME', 'TEXT_MODE', 'TEXT_GPU',
                                'QUANT_SIZES_GB', 'ready', 'choices', 'ENCODERS',
                                'ENCODER_FILE', 'MODEL_FAMILY', 'WEIGHTS_IN',
                                'TEXT_LOADER_NODE', 'PROJECTION_FILE'),
    }
    required = next((names for prefix, names in common.items()
                     if REMOTE_SERVICE_ID.startswith(prefix)), ())
    missing = [name for name in required if name not in namespace or namespace[name] is None]
    if missing:
        return False, 'Notebook 尚未完成初始化，請先執行對應的模型／服務 Cells；缺少：' + ', '.join(missing)
    if task not in REMOTE_ALLOWED_TASKS:
        return False, '此模型不支援生成模式：' + task
    if REMOTE_SERVICE_ID.startswith(('06_MiniMax_H3_', '06_HyperFlow_H3_')):
        config = namespace['TASK_CONFIGS'].get(task)
        if not isinstance(config, dict):
            return False, f'{task.upper()} 尚未設定；請先執行 Notebook 對應的模式設定 Cell。'
        if namespace['H3_ACTIVE_CONFIG'] != namespace['H3_DESIRED_CONFIG']:
            return False, 'GPU／Offload 設定與已啟動 ComfyUI 不一致，請重新啟動模型服務 Cell。'
        quant = config.get('quant')
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            variant = 'ref2va_pruned' if task == 'r2v' else 'fl2va_pruned'
            expected = (variant, quant)
            expected_file = f'minimax_h3_{variant}-{quant}.gguf'
        else:
            variant = 'ref2va' if task == 'r2v' else 'fl2va'
            encoder = config.get('encoder_variant')
            expected = (variant, quant, encoder)
            expected_file = f'minimax_h3_{variant}_hyperflow_flash_14b-{quant}.gguf'
            if encoder not in namespace['ENCODERS']:
                return False, f'{task.upper()} 的 Encoder 設定無效，請重新執行模式設定 Cell。'
            if Path(namespace['ENCODER_FILE']).name != namespace['ENCODERS'][encoder][1]:
                return False, 'GPU 1 Encoder 與此模式不一致，請重新執行模型下載／啟動 Cells。'
        if namespace['H3_MODEL_CHOICE'] != expected or namespace['H3_ACTIVE_MODEL'] != expected:
            return False, f'{task.upper()} 尚未載入相應模型權重；請設定模式並重新執行模型下載／啟動 Cells。'
        if namespace['DIT_NAME'] != expected_file:
            return False, 'ComfyUI 使用的模型檔名與任務不一致，請重新載入正確模型。'
    elif REMOTE_SERVICE_ID.startswith('06_Qwen_Image_'):
        state = namespace['QWEN21_STATE']
        if state.get('mode') != namespace['MODEL_MODE']:
            return False, 'Qwen MODEL_MODE 與已初始化的模型模式不同，請重新執行模型初始化 Cell。'
        if state.get('gpu0_model_gib') != namespace['GPU0_MODEL_GIB']:
            return False, 'Qwen GPU0_MODEL_GIB 與目前載入設定不同，請重新初始化 GPU 記憶體管理器。'
    elif REMOTE_IS_LTX:
        if not Path(namespace['ENGINE_PATH']).is_file():
            return False, 'LTX 模型引擎檔案不存在；請先執行模型與 Workflow Cell。'
        if namespace['COMFY_PROC'].poll() is not None:
            return False, 'LTX ComfyUI 行程已停止，請重新執行啟動 ComfyUI Cell。'
    return True, '模型初始化與所選模式已就緒。'

'''
Validate client parameters before they occupy a FIFO slot.
在進入 FIFO 前檢查 client 參數，避免排隊後才因設定錯誤而失敗。
'''
def remote_validate_fields(task, fields):
    required = {'task'}
    if REMOTE_SERVICE_ID.startswith('06_HeartMuLa_'):
        required.update(('lyrics', 'tags', 'duration', 'seed'))
    elif REMOTE_SERVICE_ID.startswith('06_YuE2_'):
        required.update(('style', 'lyrics', 'duration', 'steps', 'lm_seed', 'seed', 'cot'))
    elif REMOTE_KIND == 'image_text':
        required.update(('prompt', 'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_KIND == 'image_multi':
        required.update(('prompt', 'negative_prompt', 'sampler', 'reference_size',
                         'width', 'height', 'steps', 'seed', 'cfg'))
    elif REMOTE_IS_LTX:
        required.update(('prompt', 'width', 'height', 'seconds', 'fps', 'seed', 'image_strength'))
    else:
        required.update(('prompt', 'width', 'height', 'seconds', 'fps', 'steps', 'seed', 'ref_image_size'))
        if REMOTE_SERVICE_ID.startswith('06_MiniMax_H3_'):
            required.add('audio_shift')
    missing = required - fields.keys()
    if missing:
        raise ValueError('請求缺少必要參數：' + ', '.join(sorted(missing)))
    integer_fields = ('seed', 'lm_seed', 'width', 'height', 'steps', 'fps', 'duration', 'reference_size')
    float_fields = ('seconds', 'cfg', 'image_strength', 'audio_shift')
    numbers = {}
    for key in integer_fields:
        if key in fields:
            raw = fields[key]
            if isinstance(raw, bool) or str(raw).strip() != str(int(raw)):
                raise ValueError(f'{key} 必須是整數')
            numbers[key] = int(raw)
    for key in float_fields:
        if key in fields:
            numbers[key] = float(fields[key])
            if not math.isfinite(numbers[key]):
                raise ValueError(f'{key} 必須是有限數值')
    for key in ('seed', 'lm_seed'):
        if key in numbers and not -1 <= numbers[key] < 2**32:
            raise ValueError(f'{key} 必須介於 -1 至 4294967295')
    for key in ('width', 'height'):
        if key in numbers:
            size = numbers[key]
            divisor = 64 if REMOTE_KIND == 'image_text' else 32
            if size < 1 or size > 4096 or size % divisor:
                raise ValueError(f'{key} 必須介於 1～4096 且為 {divisor} 的倍數')
    for key in ('seconds', 'duration'):
        if key in numbers and not 0 < numbers[key] <= 3600:
            raise ValueError(f'{key} 必須大於 0 且不超過 3600')
    '''
    Reject unsupported FPS values before placing jobs in the GPU queue.
    在工作排入 GPU 佇列前，先拒絕不支援的 FPS 設定。
    '''
    if 'fps' in numbers:
        if REMOTE_IS_LTX and not 1 <= numbers['fps'] <= 60:
            raise ValueError('LTX-2.5 FPS 必須介於 1～60')
        if not REMOTE_IS_LTX and numbers['fps'] != 24:
            raise ValueError('MiniMax H3／HyperFlow H3 僅支援 24 FPS')
    if 'steps' in numbers and not 1 <= numbers['steps'] <= 200:
        raise ValueError('steps 必須介於 1～200')
    if REMOTE_SERVICE_ID.startswith('06_HyperFlow_H3_') and numbers.get('steps') not in (4, 6, 8):
        raise ValueError('HyperFlow steps 只支援 4、6、8')
    if 'image_strength' in numbers and not 0 <= numbers['image_strength'] <= 1:
        raise ValueError('image_strength 必須介於 0～1')
    if 'reference_size' in numbers and not 64 <= numbers['reference_size'] <= 2048:
        raise ValueError('reference_size 必須介於 64～2048')
    if 'audio_shift' in numbers and not 0.01 <= numbers['audio_shift'] <= 100:
        raise ValueError('audio_shift 必須介於 0.01～100')
    if fields.get('ref_image_size', 'match') not in ('match', 'max'):
        raise ValueError('ref_image_size 只支援 match 或 max')
    if REMOTE_KIND in ('video', 'image_text', 'image_multi') and not str(fields.get('prompt', '')).strip():
        raise ValueError('PROMPT 不可為空白')
    if task == 'r2v':
        image_count = int(fields.get('_image_count', 0))
        video_count = int(fields.get('_video_count', 0))
        audio_count = int(fields.get('_audio_count', 0))
        if not (image_count or video_count):
            raise ValueError('R2V 至少需要參考圖片或影片，不能只提供音訊。')
        if image_count > 9 or video_count > 3 or audio_count > 3 or image_count + video_count + audio_count > 12:
            raise ValueError('R2V 最多 9 張圖片、3 段影片、3 段音訊，合計最多 12 個素材。')
        count_by_type = {'Picture': image_count, 'Video': video_count,
                         'Audio': audio_count + (video_count if str(fields.get('video_audio', 'false')).lower() == 'true' else 0)}
        for media, count in count_by_type.items():
            matches = [int(number) for number in re.findall(r'<%s\s+(\d+)>' % media, str(fields['prompt']), re.I)]
            if any(number < 1 or number > count for number in matches):
                raise ValueError(f'R2V Prompt 的 <{media} N> 編號超出已上傳的參考素材數量。')

'''
Report whether the model is ready for CUDA inference.
回報模型是否已具備 CUDA 推論條件。
'''
def remote_backend_ready():
    return remote_gpu_status()[0]

'''
A Python thread cannot safely terminate an already running CUDA inference.
Python thread 無法安全強制終止正在執行的 CUDA 推論。
'''
def remote_stalled():
    with REMOTE_JOB_LOCK:
        return any(record.get('status') == 'running' and
                   time.monotonic() - record.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
                   for record in REMOTE_JOBS.values())

'''
Remove obsolete input files once the model no longer needs them.
模型不再需要輸入檔案時立即移除，避免參考圖片長期佔用空間。
'''
def remote_remove_inputs(job_id):
    folder = REMOTE_ROOT / job_id
    for file in folder.glob('*'):
        if file.name.startswith('input.') or (file.stem.isdecimal() and file.suffix.lower() in {'.png', '.jpg', '.jpeg', '.webp'}):
            file.unlink(missing_ok=True)

'''
Keep only finished results; never delete active jobs or in-progress downloads.
僅清理已結束工作，絕不刪除排隊、推論或正在下載的檔案。
'''
def remote_cleanup():
    now = time.time()
    with REMOTE_JOB_LOCK:
        finished = [(job_id, record) for job_id, record in REMOTE_JOBS.items()
                    if record.get('status') in ('done', 'error', 'expired') and
                    not record.get('_downloads')]
        finished.sort(key=lambda item: item[1].get('finished_at', now))
        sizes = {}
        for job_id, record in finished:
            folder = REMOTE_ROOT / job_id
            sizes[job_id] = sum(path.stat().st_size for path in folder.glob('**/*') if path.is_file()) if folder.exists() else 0
        total = sum(sizes.values())
        for job_id, record in finished:
            old = now - record.get('finished_at', now) > REMOTE_RESULT_TTL
            if old or total > REMOTE_RESULT_BYTES:
                shutil.rmtree(REMOTE_ROOT / job_id, ignore_errors=True)
                total -= sizes[job_id]
                record.update(status='expired', error='結果已過期或因儲存空間不足而清除，請重新提交。')
                record.pop('_path', None)
                record.pop('_last_frame', None)
            if record['status'] == 'expired' and now - record.get('finished_at', now) > 2 * REMOTE_RESULT_TTL:
                REMOTE_JOBS.pop(job_id, None)

'''
Sweep completed records periodically without interfering with inference.
定期整理已完成資料，不干擾單一 GPU worker 的推論。
'''
def remote_maintenance():
    while True:
        try:
            remote_cleanup()
        except Exception as error:
            print('API cleanup warning:', type(error).__name__, str(error)[:200])
        time.sleep(300)

'''
Serve only the generation API without exposing notebook controls.
只提供生成 API，不公開 Notebook 的管理功能。
'''
class RemoteHandler(BaseHTTPRequestHandler):
    def log_message(self, fmt, *args):
        return

    def send_json(self, code, payload, retry_after=None):
        data = json.dumps(payload, ensure_ascii=False).encode('utf-8')
        self.send_response(code)
        self.send_header('Content-Type', 'application/json; charset=utf-8')
        self.send_header('Content-Length', str(len(data)))
        if retry_after is not None:
            self.send_header('Retry-After', str(retry_after))
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        path = urlsplit(self.path).path
        if path == '/health':
            with REMOTE_JOB_LOCK:
                running = sum(record.get('status') == 'running' for record in REMOTE_JOBS.values())
            gpu_ready, gpu_message, gpu_devices = remote_gpu_status()
            requested_task = parse_qs(urlsplit(self.path).query).get('task', [REMOTE_ALLOWED_TASKS[0]])[-1]
            inference_ready, inference_message = remote_inference_ready(requested_task)
            if gpu_ready and not inference_ready:
                gpu_message = inference_message
            self.send_json(200, {'ok': True, 'api': 'kaggle-generation-bridge-v3',
                                 'service_id': REMOTE_SERVICE_ID, 'task': requested_task,
                                 'backend_ready': gpu_ready and inference_ready, 'gpu_ready': gpu_ready,
                                 'gpu_message': gpu_message, 'gpu_devices': gpu_devices,
                                 'stalled': remote_stalled(), 'running': running,
                                 'waiting': REMOTE_QUEUE.qsize(), 'queue_capacity': REMOTE_QUEUE.maxsize})
            return
        if path == '/schema':
            self.send_json(200, {'kind': REMOTE_KIND, 'tasks': REMOTE_ALLOWED_TASKS,
                                 'fields': REMOTE_FIELDS, 'image_limit': REMOTE_MAX_IMAGES, 'video_limit': 3, 'audio_limit': 3,
                                 'upload_field': 'images' if REMOTE_MAX_IMAGES else None,
                                 'queue_capacity': REMOTE_QUEUE.maxsize,
                                 'model_switching': 'configure in notebook first'})
            return
        pieces = path.strip('/').split('/')
        if pieces[0] != 'jobs' or len(pieces) not in (2, 3):
            self.send_json(404, {'error': 'Not found'})
            return
        job_id = pieces[1]
        with REMOTE_JOB_LOCK:
            record = REMOTE_JOBS.get(job_id)
            item = dict(record) if record else None
            if item and item['status'] == 'running':
                item['stalled'] = time.monotonic() - item.get('_start_clock', time.monotonic()) > REMOTE_STALL_SECONDS
            if item and item['status'] == 'queued':
                waiting = [key for key, value in REMOTE_JOBS.items() if value.get('status') == 'queued']
                if job_id in waiting:
                    item['queue_position'] = waiting.index(job_id) + 1
            if item and len(pieces) == 3 and pieces[2] in ('file', 'last-frame') and item['status'] == 'done':
                record['_downloads'] = record.get('_downloads', 0) + 1
                downloading = True
            else:
                downloading = False
        if item is None:
            self.send_json(404, {'error': 'Unknown job'})
            return
        if len(pieces) == 2:
            for field in tuple(item):
                if field.startswith('_'):
                    item.pop(field)
            self.send_json(200, item)
            return
        if not downloading:
            self.send_json(404, {'error': 'Result not available for this job'})
            return
        try:
            file_path = item.get('_path') if pieces[2] == 'file' else item.get('_last_frame')
            if not file_path or not Path(file_path).is_file():
                self.send_json(404, {'error': 'Output is no longer available'})
                return
            result = Path(file_path)
            self.send_response(200)
            self.send_header('Content-Type', 'application/octet-stream')
            self.send_header('Content-Disposition', f'attachment; filename="{result.name}"')
            self.send_header('Content-Length', str(result.stat().st_size))
            self.end_headers()
            with result.open('rb') as stream:
                shutil.copyfileobj(stream, self.wfile, length=1024 * 1024)
        except (BrokenPipeError, ConnectionResetError):
            pass
        finally:
            with REMOTE_JOB_LOCK:
                record['_downloads'] -= 1

    def do_POST(self):
        if urlsplit(self.path).path != '/generate':
            self.send_json(404, {'error': 'Not found'})
            return
        supplied_id = self.headers.get('X-Request-ID', '').strip()
        try:
            job_id = uuid.UUID(supplied_id).hex if supplied_id else uuid.uuid4().hex
        except ValueError:
            self.send_json(400, {'error': 'X-Request-ID must be a UUID'})
            return
        with REMOTE_JOB_LOCK:
            existing = REMOTE_JOBS.get(job_id)
        if existing:
            self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                 'download_url': f'/jobs/{job_id}/file', 'reused': True})
            return
        if remote_stalled():
            self.send_json(503, {'error': 'GPU worker appears stalled; inspect the Notebook before accepting more jobs'}, 60)
            return
        gpu_ready, gpu_message, _ = remote_gpu_status()
        if not gpu_ready:
            self.send_json(409, {'error': gpu_message})
            return
        if shutil.disk_usage(REMOTE_ROOT).free < REMOTE_MIN_FREE_BYTES:
            self.send_json(507, {'error': 'Kaggle disk space is low; clean old outputs before retrying'})
            return
        if not REMOTE_UPLOAD_SLOTS.acquire(timeout=1):
            self.send_json(503, {'error': 'Upload slots busy; retry later'}, 5)
            return
        staging = None
        try:
            self.connection.settimeout(120)
            length = int(self.headers.get('Content-Length', '0'))
            if not 0 < length <= 120 * 1024 * 1024:
                raise ValueError('Request must be between 1 byte and 120 MiB')
            body = self.rfile.read(length)
            if len(body) != length:
                raise ValueError('Incomplete request body')
            content_type = self.headers.get('Content-Type', '')
            fields, files, videos, audios = {}, [], [], []
            if content_type.startswith('application/json'):
                fields = json.loads(body)
            elif content_type.startswith('application/x-www-form-urlencoded'):
                fields = {key: values[-1] for key, values in parse_qs(body.decode('utf-8')).items()}
            elif content_type.startswith('multipart/form-data'):
                message = BytesParser(policy=policy.default).parsebytes(
                    b'Content-Type: ' + content_type.encode() + b'\r\nMIME-Version: 1.0\r\n\r\n' + body)
                for part in message.iter_parts():
                    name = part.get_param('name', header='content-disposition')
                    data = part.get_payload(decode=True)
                    if name in ('images', 'videos', 'audios'):
                        collection = {'images': files, 'videos': videos, 'audios': audios}[name]
                        collection.append((part.get_filename() or 'upload', data))
                    elif name:
                        fields[name] = data.decode('utf-8')
            else:
                raise ValueError('Use application/json, form data, or multipart/form-data')
            if not isinstance(fields, dict):
                raise ValueError('Parameters must be a JSON object')
            if 'abc' in fields and len(str(fields['abc']).encode('utf-8')) > 1024 * 1024:
                raise ValueError('ABC score must be 1 MiB or smaller')
            task = str(fields.get('task', REMOTE_ALLOWED_TASKS[0])).lower()
            if task not in REMOTE_ALLOWED_TASKS:
                raise ValueError('Unsupported task: ' + task)
            if len(files) > REMOTE_MAX_IMAGES or len(videos) > 3 or len(audios) > 3:
                raise ValueError(f'Only {REMOTE_MAX_IMAGES} input images are supported')
            if REMOTE_KIND in ('music', 'image_text') and (files or videos or audios):
                raise ValueError('This model does not accept reference images')
            if REMOTE_KIND == 'video':
                minimum, maximum = {'t2v': (0, 0), 'i2v': (1, 1), 'l2v': (1, 1), 'flf2v': (2, 2),
                                    'r2v': (0, REMOTE_MAX_IMAGES)}[task] if task == 'r2v' else {
                                    't2v': (0, 0), 'i2v': (1, 1), 'l2v': (1, 1), 'flf2v': (2, 2)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError('Input image count does not match the selected video task')
            if task != 'r2v' and (videos or audios):
                raise ValueError('Reference videos/audio are supported only for R2V')
            if task == 'r2v' and not (files or videos):
                raise ValueError('R2V requires an image or a video; audio alone is not supported')
            if REMOTE_KIND == 'image_multi':
                minimum, maximum = {'text': (0, 0), 'edit': (1, 1), 'multi': (2, REMOTE_MAX_IMAGES)}[task]
                if not minimum <= len(files) <= maximum:
                    raise ValueError(f'Task {task} requires {minimum}–{maximum} images')
            for key in fields:
                if key not in REMOTE_FIELDS:
                    raise ValueError('Unsupported parameter: ' + key)
            remote_validate_fields(task, dict(fields, _image_count=len(files), _video_count=len(videos), _audio_count=len(audios)))
            if 'quant' in fields and fields['quant']:
                active_model = globals().get('H3_ACTIVE_MODEL', ())
                if fields['quant'] not in active_model:
                    self.send_json(409, {'error': '指定的量化尚未載入；請先在 Notebook 下載並預檢該 GGUF。'})
                    return
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                self.send_json(409, {'error': inference_message})
                return
            for filename, data in files:
                if not isinstance(data, bytes) or len(data) > 20 * 1024 * 1024:
                    raise ValueError('Each image must be 20 MiB or smaller')
                if Path(filename).suffix.lower() not in {'.png', '.jpg', '.jpeg', '.webp'}:
                    raise ValueError('Only PNG/JPEG/WebP images are supported')
                from PIL import Image
                with Image.open(io.BytesIO(data)) as photo:
                    photo.verify()
            if str(fields.get('video_audio', 'false')).lower() not in ('true', 'false'):
                raise ValueError('video_audio must be true or false')
            for filename, data in videos:
                if not data or len(data) > 80 * 1024 * 1024 or Path(filename).suffix.lower() not in ('.mp4', '.mov', '.webm'):
                    raise ValueError('Reference video must be MP4/MOV/WebM and at most 80 MiB')
            for filename, data in audios:
                if not data or len(data) > 30 * 1024 * 1024 or Path(filename).suffix.lower() not in ('.mp3', '.wav', '.flac', '.ogg', '.m4a'):
                    raise ValueError('Reference audio must be an audio file at most 30 MiB')
            staging = Path(tempfile.mkdtemp(prefix='pending_', dir=REMOTE_ROOT))
            image_names, video_names, audio_names = [], [], []
            for index, (filename, data) in enumerate(files, 1):
                target = staging / f'{index:03d}{Path(filename).suffix.lower()}'
                target.write_bytes(data)
                image_names.append(target.name)
            for kind, uploaded, names in (('video', videos, video_names), ('audio', audios, audio_names)):
                for index, (filename, data) in enumerate(uploaded, 1):
                    target = staging / f'{kind}_{index:03d}{Path(filename).suffix.lower()}'
                    target.write_bytes(data)
                    names.append(target.name)

            '''
            Commit the validated upload and queue entry in one critical section.
            在同一個鎖定區段確認請求、保存資料並加入工作佇列。
            '''
            with REMOTE_JOB_LOCK:
                existing = REMOTE_JOBS.get(job_id)
                full = REMOTE_QUEUE.full()
                if not existing and not full:
                    folder = REMOTE_ROOT / job_id
                    staging.rename(folder)
                    staging = None
                    image_paths = [str(folder / name) for name in image_names]
                    fields['_ref_videos'] = [str(folder / name) for name in video_names]
                    fields['_ref_audios'] = [str(folder / name) for name in audio_names]
                    REMOTE_JOBS[job_id] = {'id': job_id, 'task': task, 'status': 'queued',
                                           'created_at': time.time()}
                    REMOTE_QUEUE.put_nowait((job_id, fields, image_paths))
                    position = REMOTE_QUEUE.qsize()
            if existing:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'reused': True})
            elif full:
                self.send_json(429, {'error': 'Queue is full; retry later'}, 15)
            else:
                self.send_json(202, {'job_id': job_id, 'status_url': f'/jobs/{job_id}',
                                     'download_url': f'/jobs/{job_id}/file', 'queue_position': position})
        except (ValueError, KeyError, TypeError, json.JSONDecodeError) as error:
            self.send_json(400, {'error': str(error)})
        except (TimeoutError, ConnectionError, OSError) as error:
            self.send_json(503, {'error': f'Upload temporarily failed: {type(error).__name__}'}, 10)
        except Exception as error:
            self.send_json(500, {'error': type(error).__name__ + ': ' + str(error)[:250]})
        finally:
            if staging is not None:
                shutil.rmtree(staging, ignore_errors=True)
            REMOTE_UPLOAD_SLOTS.release()

'''
Convert numeric HTTP form fields into explicitly supported generator settings.
只轉換白名單中的 HTTP 參數，不執行客戶端提供的任何程式碼。
'''
def remote_generate(job_id, fields, image_paths):
    with REMOTE_ENGINE_LOCK:
        with REMOTE_JOB_LOCK:
            REMOTE_JOBS[job_id].update(status='running', _start_clock=time.monotonic())
        namespace = globals()
        keys = ['TASK', 'PROMPT', 'WIDTH', 'HEIGHT', 'SEED', 'LM_SEED', 'SECONDS', 'STEPS', 'CFG',
                'LYRICS', 'TAGS', 'STYLE', 'DURATION', 'GPU_ID', 'COT', 'ABC_PATH',
                'OUTPUT_NAME', 'REFERENCE_PATHS', 'ACTIVE_JOB', 'TASK_CONFIGS',
                'R2V_REF_DIR', 'NEGATIVE_PROMPT', 'SAMPLER', 'REFERENCE_SIZE',
                'OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT', 'RUN_SEED',
                'QUANT', 'ENCODER_VARIANT']
        before = {key: copy.deepcopy(namespace.get(key)) for key in keys if key in namespace}
        missing = set(keys) - set(before)
        try:
            task = fields.get('task', REMOTE_ALLOWED_TASKS[0]).lower()
            numerics = {'seed': int, 'lm_seed': int, 'width': int, 'height': int, 'steps': int,
                        'seconds': float, 'fps': int, 'cfg': float, 'duration': int,
                        'image_strength': float, 'reference_size': int, 'audio_shift': float}
            inputs = {}
            for key, value in fields.items():
                inputs[key] = numerics[key](value) if key in numerics else value
            for seed_key in ('seed', 'lm_seed'):
                if seed_key in inputs:
                    if not -1 <= inputs[seed_key] < 2**32:
                        raise ValueError(f'{seed_key} must be -1 or between 0 and 4294967295')
                    if inputs[seed_key] == -1:
                        inputs[seed_key] = secrets.randbelow(2**32)
            with REMOTE_JOB_LOCK:
                for seed_key in ('seed', 'lm_seed'):
                    if seed_key in inputs:
                        REMOTE_JOBS[job_id][seed_key + '_used'] = inputs[seed_key]
            for dimension in ('width', 'height'):
                if dimension in inputs and not 0 < inputs[dimension] <= 4096:
                    raise ValueError('Width and height must be between 1 and 4096')
            for key in ('seconds', 'duration'):
                if key in inputs and not 0 < inputs[key] <= 3600:
                    raise ValueError('Duration must be between 0 and 3600 seconds')
            if 'reference_size' in inputs and not 64 <= inputs['reference_size'] <= 2048:
                raise ValueError('Reference size must be between 64 and 2048')
            if 'ref_image_size' in inputs and inputs['ref_image_size'] not in ('match', 'max'):
                raise ValueError('Reference image mode must be match or max')
            if 'image_strength' in inputs and not 0 <= inputs['image_strength'] <= 1:
                raise ValueError('Image strength must be between 0 and 1')
            if 'steps' in inputs and not 1 <= inputs['steps'] <= 200:
                raise ValueError('Steps must be between 1 and 200')
            if REMOTE_KIND == 'video':
                if 'TASK_CONFIGS' in namespace:
                    if task not in namespace['TASK_CONFIGS']:
                        raise ValueError('Run this task configuration cell before API generation')
                    updated = copy.deepcopy(namespace['TASK_CONFIGS'][task])
                    for field in ('prompt', 'width', 'height', 'seconds', 'steps', 'seed',
                                  'audio_shift', 'fps', 'ref_image_size', 'quant'):
                        if field in inputs:
                            updated[field] = inputs[field]
                    updated['output_name'] = f'remote_{job_id[:12]}.mp4'
                    updated['quant'] = inputs.get('quant') or namespace.get('QUANT', updated.get('quant'))
                    if task == 'r2v':
                        updated['video_audio'] = str(inputs.get('video_audio', 'false')).lower() == 'true'
                        updated['ref_videos'] = inputs.get('_ref_videos', [])
                        updated['ref_audios'] = inputs.get('_ref_audios', [])
                    if task == 'i2v':
                        updated['first_image'] = image_paths[0]
                    elif task == 'l2v':
                        updated['last_image'] = image_paths[0]
                    elif task == 'flf2v':
                        updated['first_image'], updated['last_image'] = image_paths
                    elif task == 'r2v':
                        updated['ref_images'] = image_paths
                        namespace['R2V_REF_DIR'] = REMOTE_ROOT / job_id
                    namespace['TASK_CONFIGS'][task] = updated
                    namespace['TASK'] = task
                else:
                    namespace['ACTIVE_JOB'] = {'mode': task, 'prompt': inputs.get('prompt', namespace.get('PROMPT', '')),
                                               'run_name': f'remote_{job_id[:12]}',
                                               'first_path': image_paths[0] if image_paths else None,
                                               'last_path': image_paths[1] if len(image_paths) > 1 else None}
                    for key in ('width', 'height', 'seconds', 'seed', 'image_strength'):
                        if key in inputs:
                            namespace[key.upper()] = inputs[key]
            else:
                for key in ('prompt', 'width', 'height', 'seed', 'lm_seed', 'steps', 'cfg',
                            'lyrics', 'tags', 'style', 'duration', 'negative_prompt',
                            'sampler', 'reference_size', 'cot'):
                    if key in inputs:
                        namespace[key.upper()] = inputs[key]
                if REMOTE_KIND == 'image_multi':
                    namespace['TASK'] = task
                    namespace['REFERENCE_PATHS'] = image_paths
                if REMOTE_KIND == 'music':
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.mp3'
                    if 'ABC_PATH' in namespace:
                        namespace['ABC_PATH'] = ''
                    if 'abc' in inputs:
                        score = (REMOTE_ROOT / job_id / 'input.abc')
                        score.write_text(str(inputs['abc']), encoding='utf-8')
                        namespace['ABC_PATH'] = str(score)
                if REMOTE_KIND in ('image_multi', 'image_text'):
                    namespace['OUTPUT_NAME'] = f'remote_{job_id[:12]}.png'
            for output_key in ('OUTPUTS', 'output', 'OUTPUT_PATH', 'OUTPUT_MP4', 'LAST_OUTPUT'):
                namespace.pop(output_key, None)
            '''
            Recheck the CUDA backend after queuing to prevent CPU-only generation.
            工作排入佇列後再次確認 CUDA，避免後端異常時改走純 CPU 生成。
            '''
            gpu_ready, gpu_message, _ = remote_gpu_status()
            if not gpu_ready:
                raise RuntimeError(gpu_message)
            inference_ready, inference_message = remote_inference_ready(task)
            if not inference_ready:
                raise RuntimeError(inference_message)
            exec(compile(REMOTE_SCRIPT, '<notebook-generation-cell>', 'exec'), namespace)
            paths = []
            if REMOTE_KIND == 'video':
                paths = [namespace.get('LAST_OUTPUT') or namespace.get('OUTPUT_MP4')]
            elif REMOTE_KIND == 'image_text':
                paths = [namespace.get('OUTPUT_PATH')]
            elif REMOTE_KIND == 'image_multi':
                paths = [namespace.get('output')]
            elif REMOTE_KIND == 'music':
                paths = namespace.get('OUTPUTS', []) if 'OUTPUTS' in namespace else [namespace.get('output')]
            results = [Path(path) for path in paths if path is not None]
            if REMOTE_KIND == 'music' and REMOTE_SERVICE_ID.startswith('06_YuE2_'):
                results += [sidecar for path in list(results)
                            for ext in ('.json', '.abc')
                            if (sidecar := path.with_suffix(ext)).is_file()]
            if not results or any(not path.is_file() for path in results):
                raise FileNotFoundError('Generation finished without a saved result file')
            folder = REMOTE_ROOT / job_id
            if len(results) == 1:
                target = folder / f'result{results[0].suffix.lower()}'
                shutil.copy2(results[0], target)
            else:
                target = folder / 'results.zip'
                with zipfile.ZipFile(target, 'w', compression=zipfile.ZIP_STORED) as archive:
                    for result in results:
                        archive.write(result, arcname=result.name)
            last_frame = None
            if REMOTE_KIND == 'video' and shutil.which('ffprobe') and shutil.which('ffmpeg'):
                try:
                    probe = subprocess.run(['ffprobe', '-v', 'error', '-count_frames',
                                            '-select_streams', 'v:0', '-show_entries',
                                            'stream=nb_read_frames', '-of', 'json', str(target)],
                                           check=True, text=True, capture_output=True, timeout=120)
                    index = int(json.loads(probe.stdout)['streams'][0]['nb_read_frames']) - 1
                    if index >= 0:
                        last_frame = folder / 'last_frame.png'
                        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', str(target),
                                        '-vf', rf'select=eq(n\,{index})', '-frames:v', '1',
                                        str(last_frame)], check=True)
                except (OSError, ValueError, KeyError, subprocess.CalledProcessError, subprocess.TimeoutExpired):
                    last_frame = None
            for original in results:
                if original.name.startswith(f'remote_{job_id[:12]}') and original != target:
                    try:
                        original.unlink(missing_ok=True)
                    except OSError:
                        pass
            comfy = globals().get('COMFY')
            if comfy is not None and (Path(comfy) / 'output').exists():
                for artifact in (Path(comfy) / 'output').rglob(f'remote_{job_id[:12]}*'):
                    if artifact.is_file():
                        try:
                            artifact.unlink()
                        except OSError:
                            pass
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='done', filename=target.name,
                                           bytes=target.stat().st_size, _path=str(target),
                                           last_frame_available=bool(last_frame),
                                           _last_frame=str(last_frame) if last_frame else None)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            for key in missing:
                namespace.pop(key, None)
            namespace.update(before)

'''
Use one GPU worker, regardless of the number of HTTP request threads.
不論有多少 HTTP 連線，都只有一個 GPU worker 消費佇列。
'''
def remote_worker():
    while True:
        job_id, fields, image_paths = REMOTE_QUEUE.get()
        try:
            remote_generate(job_id, fields, image_paths)
        except Exception as error:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id].update(status='error', error=f'{type(error).__name__}: {error}'[:2000])
        finally:
            with REMOTE_JOB_LOCK:
                if job_id in REMOTE_JOBS:
                    REMOTE_JOBS[job_id]['finished_at'] = time.time()
            try:
                remote_remove_inputs(job_id)
            except OSError as error:
                print('API input cleanup warning:', str(error)[:180])
            finally:
                REMOTE_QUEUE.task_done()

if not (globals().get('REMOTE_WORKER') and REMOTE_WORKER.is_alive()):
    REMOTE_WORKER = threading.Thread(target=remote_worker, daemon=True)
    REMOTE_WORKER.start()

if not (globals().get('REMOTE_MAINTENANCE') and REMOTE_MAINTENANCE.is_alive()):
    REMOTE_MAINTENANCE = threading.Thread(target=remote_maintenance, daemon=True)
    REMOTE_MAINTENANCE.start()

old_tunnel = globals().get('REMOTE_NGROK_PROC')
if old_tunnel is not None and old_tunnel.poll() is None:
    old_tunnel.terminate()
    try:
        old_tunnel.wait(timeout=5)
    except subprocess.TimeoutExpired:
        old_tunnel.kill()
        old_tunnel.wait(timeout=5)

NGROK_URL = ''

'''
Install the official ngrok agent only if no usable binary exists.
系統尚無 ngrok 才下載官方執行檔，不重複安裝。
'''
ngrok_bin = shutil.which('ngrok') or str(ROOT / 'ngrok')
if not Path(ngrok_bin).is_file():
    url = 'https://bin.ngrok.com/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz'
    with urllib.request.urlopen(url, timeout=90) as response:
        blob = response.read()
    with tarfile.open(fileobj=io.BytesIO(blob), mode='r:gz') as archive:
        entry = next((member for member in archive.getmembers() if Path(member.name).name == 'ngrok' and member.isfile()), None)
        if entry is None:
            raise RuntimeError('ngrok 官方下載檔案缺少執行檔')
        Path(ngrok_bin).write_bytes(archive.extractfile(entry).read())
    Path(ngrok_bin).chmod(0o755)

ngrok_token = getpass('輸入 ngrok Authtoken：').strip()
if not ngrok_token:
    raise ValueError('未輸入 ngrok Authtoken')

result = subprocess.run([ngrok_bin, 'config', 'add-authtoken', ngrok_token], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok token 設定失敗（請確認 Token 是否有效）')

del ngrok_token
result = subprocess.run([ngrok_bin, 'config', 'check'], capture_output=True, text=True)
if result.returncode:
    raise RuntimeError('ngrok 設定格式檢查失敗：' + result.stderr[-500:])

server = globals().get('REMOTE_SERVER')
server_thread = globals().get('REMOTE_SERVER_THREAD')
server_is_current = (
    server is not None
    and server_thread is not None
    and server_thread.is_alive()
    and getattr(server, 'notebook_id', None) == REMOTE_SERVICE_ID
)
if server is not None and not server_is_current:
    if server_thread is not None and server_thread.is_alive():
        server.shutdown()
    server.server_close()

if not server_is_current:
    try:
        REMOTE_SERVER = ThreadingHTTPServer(('127.0.0.1', REMOTE_PORT), RemoteHandler)
    except OSError as error:
        raise RuntimeError(f'本機 API Port {REMOTE_PORT} 已被其他服務占用') from error
    REMOTE_SERVER.notebook_id = REMOTE_SERVICE_ID
    REMOTE_SERVER.daemon_threads = True
    REMOTE_SERVER_THREAD = threading.Thread(target=REMOTE_SERVER.serve_forever, daemon=True)
    REMOTE_SERVER_THREAD.start()
    print('已啟動新的本機 API Server。')
else:
    REMOTE_SERVER.RequestHandlerClass = RemoteHandler
    print('已沿用本機 API Server；現有生成工作與結果保持不變。')

with urllib.request.urlopen(f'http://127.0.0.1:{REMOTE_PORT}/health', timeout=4) as response:
    if json.load(response).get('ok') is not True:
        raise RuntimeError('本機 API Server 尚未就緒')

'''
Use the agent inspection API to obtain the actual public URL.
透過 ngrok 本機檢查 API 取得實際對外 HTTPS 網址。
'''
NGROK_LOG = REMOTE_ROOT / 'ngrok.log'
with NGROK_LOG.open('w') as log:
    REMOTE_NGROK_PROC = subprocess.Popen([ngrok_bin, 'http', str(REMOTE_PORT)], stdout=log,
                                         stderr=subprocess.STDOUT, start_new_session=True)

NGROK_URL = ''
for attempt in range(25):
    if REMOTE_NGROK_PROC.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:4040/api/tunnels', timeout=2) as response:
            tunnels = json.load(response).get('tunnels', [])
        NGROK_URL = next((item['public_url'] for item in tunnels
                          if item.get('public_url', '').startswith('https://') and
                          str(REMOTE_PORT) in item.get('config', {}).get('addr', '')), '')
        if NGROK_URL:
            break
    except (OSError, ValueError, KeyError):
        pass
    time.sleep(1)

if not NGROK_URL:
    raise RuntimeError('ngrok URL 建立失敗：\n' + NGROK_LOG.read_text(errors='replace')[-2500:])

print('ngrok HTTPS URL:', NGROK_URL)
print('提醒：公開的 ngrok API URL 不需要額外驗證，請只分享給授課學生。')
print('GET /health | GET /schema | POST /generate | GET /jobs/{id} | GET /jobs/{id}/file')
print('影片輸出另提供：GET /jobs/{id}/last-frame')
print('✓ 第 14 節完成：選用 FIFO／ngrok API 已啟動。')


15. （選用）遠端 API 與 client 使用說明


In [ ]:
'''
Query the active bridge for supported parameters and task names.
從正在執行的橋接服務讀取實際支援的參數與模式。
'''
import json
import urllib.request
request = urllib.request.Request(f'http://127.0.0.1:{REMOTE_PORT}/schema')
with urllib.request.urlopen(request, timeout=5) as response:
    schema = json.load(response)

print(json.dumps(schema, ensure_ascii=False, indent=2))
print('用法：POST NGROK_URL + "/generate"（JSON 或 multipart/images），取得 job_id；')
print('輪詢 GET /jobs/{job_id}；完成後下載 GET /jobs/{job_id}/file。')
print('本機 requests 範例：06_HyperFlow_H3_Flash_影片生成_client.py（與本 Notebook 同名對應）。')
print('使用前只需在對應的 client.py 填入 API_URL。')
print('本服務使用單一 FIFO worker，遠端請求不會同時啟動多筆 GPU 推論。')
print('✓ 第 15 節完成：API Schema 和 client 說明已顯示。')
